THE CODE 

In [ ]:
import importlib.metadata as md
print("qiskit:", md.version("qiskit"))
print("qiskit-aer:", md.version("qiskit-aer"))
print("qiskit-ibm-runtime:", md.version("qiskit-ibm-runtime"))
print("qiskit-machine-learning:", md.version("qiskit-machine-learning"))
import shutil, sys
print("python:", sys.executable)
print("dot on PATH:", shutil.which("dot"))   # should print a full path


import importlib.metadata as im, qiskit, qiskit.utils.optionals as opt
print("qiskit:", im.version("qiskit"))
print("HAS_SYMPY present?:", hasattr(opt, "HAS_SYMPY"))


In [ ]:

import qiskit, qiskit.utils.optionals as opt, sys, inspect
print("Py:", sys.version)
print("Qiskit version:", getattr(qiskit, "__qiskit_version__", None))
print("qiskit module at:", qiskit.__file__)
print("optionals at:", opt.__file__)
print("HAS_SYMPY present?:", hasattr(opt, "HAS_SYMPY"))


In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService
import os

# One-time setup: store your IBM Quantum API token securely as an environment
# variable (e.g. `set IBM_QUANTUM_TOKEN=...` on Windows or
# `export IBM_QUANTUM_TOKEN=...` on Linux/macOS), then run this cell once.
QiskitRuntimeService.save_account(
    token=os.environ["IBM_QUANTUM_TOKEN"],
    instance=os.environ.get("IBM_QUANTUM_INSTANCE", "open"),
    region="us-east",                # Open plan requires us-east
    set_as_default=True,
    overwrite=True,
)


In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService
service = QiskitRuntimeService()  # loads the saved default

print("Instances:", service.instances())           # expect ['open/main']
backends = service.backends(operational=True, simulator=False)
print("Real backends:", [b.name for b in backends])

k = 7
backend = service.least_busy(operational=True, simulator=False, min_num_qubits=k)
print("Selected:", backend.name)


In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit.visualization import plot_error_map
import matplotlib.pyplot as plt

service = QiskitRuntimeService()

for name in ["ibm_brisbane", "ibm_torino"]:
    backend = service.backend(name)
    #print(f"\n=== {backend.name} ===")
    plot_error_map(backend)
    plt.show()


In [ ]:
from qiskit.visualization import plot_error_map
ibm_brisbane = service.backend("ibm_brisbane")
plot_error_map(ibm_brisbane)   # should work now


In [ ]:
from qiskit.visualization import plot_error_map
plot_error_map(backend)   # should work now


In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit.circuit.library import z_feature_map
from qiskit import transpile
from math import inf
from collections import defaultdict

service = QiskitRuntimeService()

def twoq_error_lookup(backend):
    """Build {(q0,q1): error} from backend.properties() for ECR/CX."""
    props = backend.properties()
    e = {}
    if props is None:
        return e
    for g in props.gates:
        if g.gate.lower() in ("ecr", "cx"):
            err = None
            for p in g.parameters:
                if p.name in ("gate_error", "gate_error_rate"):
                    err = float(p.value)
                    break
            if err is not None and len(g.qubits) == 2:
                key = tuple(g.qubits)
                e[key] = min(err, e.get(key, err))  # keep the smaller if duplicates
    return e

def edge_err(errmap, a, b, default=0.02):
    # undirected lookup
    return errmap.get((a, b), errmap.get((b, a), default))

def best_chains(backend, k=7, top=3):
    target = backend.target
    cmap = target.build_coupling_map()                 # list of (u,v)
    errmap = twoq_error_lookup(backend)

    # adjacency
    nbrs = defaultdict(list)
    for a,b in cmap:
        nbrs[a].append(b); nbrs[b].append(a)

    best = []
    def dfs(node, path, cost):
        nonlocal best
        if len(path) == k:
            best.append((cost, path[:]))
            best.sort(key=lambda t: t[0])
            if len(best) > top: best[:] = best[:top]
            return
        for u in sorted(nbrs[node], key=lambda v: edge_err(errmap, node, v)):
            if u not in path:
                dfs(u, path+[u], cost + edge_err(errmap, node, u))

    for s in set(sum(([a,b] for a,b in cmap), [])):
        dfs(s, [s], 0.0)
    return best

def transpile_metrics(backend, layout, k=7, reps=1, ent="linear"):
    fm = z_feature_map(feature_dimension=k, reps=reps, entanglement=ent)
    qc = fm.assign_parameters({p: 0.1 for p in fm.parameters}, inplace=False)
    tc = transpile(qc, backend=backend,
                   initial_layout=layout,
                   layout_method="sabre", routing_method="sabre",
                   optimization_level=2)
    ops   = tc.count_ops()
    depth = tc.depth()
    g2    = int(ops.get("ecr",0)+ops.get("cx",0)+ops.get("cz",0))
    swaps = int(ops.get("swap",0))
    return depth, g2, swaps

def score_device(name, k=7):
    backend = service.backend(name)
    print(f"\n=== {backend.name} (k={k}) ===")
    cands = best_chains(backend, k=k, top=3)
    if not cands:
        print("No chain candidates found.")
        return backend, None
    best_layout, best_tuple = None, None
    for i,(cost,layout) in enumerate(cands, 1):
        depth, g2, sw = transpile_metrics(backend, layout, k=k, reps=1, ent="linear")
        print(f"#{i} layout={layout} | sum edge err≈{cost:.4f} | depth={depth}  2Q={g2}  SWAPs={sw}")
        score = (g2, sw, depth, cost)  # prioritize fewer 2Q gates, SWAPs, then depth, then raw error sum
        if best_tuple is None or score < best_tuple:
            best_tuple, best_layout = score, layout
    d,g2,sw,cost = None,None,None,None
    print(f"✅ Suggested initial_layout: {best_layout}")
    return backend, best_layout

bk1, layout1 = score_device("ibm_brisbane", k=7)
bk2, layout2 = score_device("ibm_torino",   k=7)


In [ ]:
# ====== Minimal-training-size finder for WBCD (k=7) ======
# - Builds QSVC kernels via state fidelity:
#   * Noiseless kernel: exact overlaps from statevectors
#   * Noisy kernel: compute–uncompute circuits on Aer with backend noise model
# - Evaluates a learning curve over N in {32, 48, 64, 96, 128, 160}
# - Metrics: TP/FP/TN/FN, Accuracy, F1, AUC
# - Selects smallest N whose NOISY AUC is within tol (default 0.5pp) of the best
#
# You can tailor:
#   BACKEND_FOR_NOISE, N_SET, SHOTS_NOISY, TOLERANCE

import numpy as np
from math import inf
from time import perf_counter
from collections import defaultdict

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, confusion_matrix
)
from sklearn.svm import SVC

from qiskit.circuit import QuantumCircuit
from qiskit.circuit.library import z_feature_map
from qiskit.quantum_info import Statevector
from qiskit import transpile

# Aer noisy simulation
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel

# Optional: use a live backend for the noise model if you have service available
try:
    from qiskit_ibm_runtime import QiskitRuntimeService
    _service = QiskitRuntimeService()
except Exception:
    _service = None

# --------------------------- user knobs ---------------------------
BACKEND_FOR_NOISE = "ibm_brisbane"   # or "ibm_torino"
RANDOM_STATE      = 42
TEST_SIZE         = 0.20
K                 = 7                # qubits/features
REPS              = 1
ENTANGL           = "linear"
N_SET             = [32, 48, 64, 96, 128, 160]     # candidate train sizes
SHOTS_NOISY       = 1000            # keep modest for speed; bump to 2000–5000 later
TOLERANCE         = 0.005           # 0.5 percentage point noisy-AUC tolerance
CHUNK_CIRCS       = 256             # batch size for Aer runs to control memory
# ------------------------------------------------------------------

# 0) Load WBCD, 80/20 split, scale, RFE -> 7 features (train-only fit)
X, y = load_breast_cancer(return_X_y=True)
Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)

scaler = StandardScaler().fit(Xtr)
Xtr_s = scaler.transform(Xtr)
Xte_s = scaler.transform(Xte)

# RFE on train only (LogReg L2 — fast & stable), then transform both
lr = LogisticRegression(max_iter=500, n_jobs=None, solver="lbfgs", random_state=RANDOM_STATE)
rfe = RFE(lr, n_features_to_select=K).fit(Xtr_s, ytr)
Xtr_k = rfe.transform(Xtr_s)
Xte_k = rfe.transform(Xte_s)

# 1) Feature map (modern function API)
fm = z_feature_map(feature_dimension=K, reps=REPS, entanglement=ENTANGL)

# -- helpers --------------------------------------------------------
def _bind_feature_map(x):
    # bind vector x to fm in the correct parameter order and return an Instruction
    return fm.assign_parameters({p: float(val) for p, val in zip(fm.parameters, x)}).to_instruction()

def kernel_noiseless(A, B):
    """Exact state-fidelity kernel via Statevector inner products."""
    sA = []
    for x in A:
        qc = QuantumCircuit(K)
        qc.append(_bind_feature_map(x), range(K))
        sv = Statevector.from_instruction(qc)  # |psi(x)>
        sA.append(sv.data)
    sB = []
    for x in B:
        qc = QuantumCircuit(K)
        qc.append(_bind_feature_map(x), range(K))
        sv = Statevector.from_instruction(qc)
        sB.append(sv.data)

    sA = np.asarray(sA)  # [|A|, 2^K]
    sB = np.asarray(sB)  # [|B|, 2^K]
    # Gram(i,j) = |<psi(x_i)|psi(x_j)>|^2
    Kmat = np.abs(sA @ np.conjugate(sB.T)) ** 2
    return Kmat

def _fidelity_circuit(xi, xj):
    """Compute–uncompute circuit: U(xi) then U(xj)^\dagger, measure all."""
    qc = QuantumCircuit(K)
    qc.append(_bind_feature_map(xi), range(K))
    qc.append(_bind_feature_map(xj).inverse(), range(K))
    qc.measure_all()
    return qc

def kernel_noisy(trainA, trainB, noise_model, shots=SHOTS_NOISY, chunk=CHUNK_CIRCS, seed=RANDOM_STATE):
    """
    Noisy fidelity kernel on Aer:
      - If A==B (train-train), compute only upper triangle, mirror to lower, set diag=1.
      - Else (test-train), compute all pairs.
    """
    same = trainA is trainB or np.array_equal(trainA, trainB)
    nA, nB = len(trainA), len(trainB)
    Kmat = np.zeros((nA, nB), dtype=float)

    sim = AerSimulator(noise_model=noise_model, seed_simulator=seed)
    basis = noise_model.basis_gates if noise_model is not None else None

    def run_batch(circs, idx_pairs):
        tcircs = transpile(circs, backend=sim, basis_gates=basis, optimization_level=0)
        res = sim.run(tcircs, shots=shots).result()
        for r, (i, j) in enumerate(idx_pairs):
            counts = res.get_counts(r)
            prob0 = counts.get("0"*K, 0) / shots
            Kmat[i, j] = prob0

    circs, pairs = [], []
    if same:
        # only (i,j) with i<=j
        for i in range(nA):
            # diagonal = 1 exactly; skip circuit
            Kmat[i, i] = 1.0
            for j in range(i+1, nB):
                circs.append(_fidelity_circuit(trainA[i], trainB[j]))
                pairs.append((i, j))
                if len(circs) >= chunk:
                    run_batch(circs, pairs); circs, pairs = [], []
        if circs:
            run_batch(circs, pairs)
        # mirror
        for i in range(nA):
            for j in range(i+1, nB):
                Kmat[j, i] = Kmat[i, j]
    else:
        for i in range(nA):
            for j in range(nB):
                circs.append(_fidelity_circuit(trainA[i], trainB[j]))
                pairs.append((i, j))
                if len(circs) >= chunk:
                    run_batch(circs, pairs); circs, pairs = [], []
        if circs:
            run_batch(circs, pairs)
    return Kmat

def fit_predict_from_kernels(Ktt, Kte, y_train, y_test):
    """Train SVC on precomputed kernel, return metrics + confusion components."""
    svc = SVC(kernel="precomputed", probability=False)  # decision_function available
    svc.fit(Ktt, y_train)
    y_hat = svc.predict(Kte)
    acc = accuracy_score(y_test, y_hat)
    f1  = f1_score(y_test, y_hat)
    tn, fp, fn, tp = confusion_matrix(y_test, y_hat, labels=[0,1]).ravel()
    # AUC via decision_function if available
    try:
        scores = svc.decision_function(Kte)
        auc = roc_auc_score(y_test, scores)
    except Exception:
        auc = np.nan
    return dict(acc=acc, f1=f1, auc=auc, tp=int(tp), fp=int(fp), tn=int(tn), fn=int(fn))

# Build noise model from a real backend (if available), else fall back to no noise
noise_model = None
if _service is not None:
    try:
        backend = _service.backend(BACKEND_FOR_NOISE)
        props = backend.properties()
        noise_model = NoiseModel.from_backend(props) if props is not None else None
        print(f"[noise] using noise model from {BACKEND_FOR_NOISE}")
    except Exception as e:
        print(f"[noise] could not load backend '{BACKEND_FOR_NOISE}': {e}\n       proceeding without noise model.")

if noise_model is None:
    # very light T1/T2/RO-free sim (not ideal), but we'll still get a "noisy" pass via finite shots
    noise_model = NoiseModel()
    print("[noise] using empty noise model (finite-shot only)")

# 2) Learning curve over N, both NOISELESS and NOISY
results = []
rng = np.random.RandomState(RANDOM_STATE)

# Stratified subsets of train set for each N
idx_all = np.arange(len(Xtr_k))
for N in N_SET:
    # stratified pick of N indices
    sss = StratifiedShuffleSplit(n_splits=1, train_size=N, random_state=RANDOM_STATE)
    (sub_idx, _), = sss.split(Xtr_k, ytr)
    XN, yN = Xtr_k[sub_idx], ytr[sub_idx]

    print(f"\n=== N={N} (k={K}) ===")
    # --- NOISELESS ---
    t0 = perf_counter()
    Ktt_ideal = kernel_noiseless(XN, XN)
    Kte_ideal = kernel_noiseless(Xte_k, XN)
    noiseless = fit_predict_from_kernels(Ktt_ideal, Kte_ideal, yN, yte)
    t1 = perf_counter()

    # --- NOISY (Aer with noise model + shots) ---
    t2 = perf_counter()
    Ktt_noisy = kernel_noisy(XN, XN, noise_model, shots=SHOTS_NOISY)
    Kte_noisy = kernel_noisy(Xte_k, XN, noise_model, shots=SHOTS_NOISY)
    noisy = fit_predict_from_kernels(Ktt_noisy, Kte_noisy, yN, yte)
    t3 = perf_counter()

    print(f"  Noiseless  : AUC={noiseless['auc']:.4f}  Acc={noiseless['acc']:.4f}  F1={noiseless['f1']:.4f}  "
          f"TP={noiseless['tp']} FP={noiseless['fp']} TN={noiseless['tn']} FN={noiseless['fn']}  "
          f"({t1-t0:.1f}s)")
    print(f"  Noisy (Aer): AUC={noisy['auc']:.4f}  Acc={noisy['acc']:.4f}  F1={noisy['f1']:.4f}  "
          f"TP={noisy['tp']} FP={noisy['fp']} TN={noisy['tn']} FN={noisy['fn']}  "
          f"({t3-t2:.1f}s, shots={SHOTS_NOISY})")

    results.append((N, noiseless, noisy))

# 3) Pick smallest N* whose NOISY AUC is within tolerance of the best NOISY AUC
noisy_aucs = [r[2]['auc'] for r in results]
best_noisy_auc = np.nanmax(noisy_aucs)
N_candidates = [N for (N, _, noisy) in results if noisy['auc'] >= best_noisy_auc - TOLERANCE]
N_star = min(N_candidates) if N_candidates else N_SET[-1]

print("\n=== Selection ===")
print(f"Best NOISY AUC = {best_noisy_auc:.4f}")
print(f"N* (smallest within {TOLERANCE*100:.1f} pp of best NOISY AUC) = {N_star}")

print("\nSummary (N, Noiseless AUC/Acc/F1 TP/FP/TN/FN | Noisy AUC/Acc/F1 TP/FP/TN/FN):")
for (N, ideal, noisy) in results:
    print(f"N={N:<4}  "
          f"ideal: AUC={ideal['auc']:.4f} Acc={ideal['acc']:.4f} F1={ideal['f1']:.4f} "
          f"TP={ideal['tp']} FP={ideal['fp']} TN={ideal['tn']} FN={ideal['fn']}  |  "
          f"noisy: AUC={noisy['auc']:.4f} Acc={noisy['acc']:.4f} F1={noisy['f1']:.4f} "
          f"TP={noisy['tp']} FP={noisy['fp']} TN={noisy['tn']} FN={noisy['fn']}")  # end


In [ ]:
# ====== Minimal-training-size finder (WBCD, k=7) with your backend+layout ======
# Uses:
#   - Noiseless kernel (statevector)
#   - Noisy kernel (AerSimulator.from_backend(ibm_brisbane)), transpiled with your initial_layout
# Prints TP/FP/TN/FN, Acc, F1, AUC per N, then selects the smallest N* within tol of best NOISY AUC.

import numpy as np
from time import perf_counter
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix
from sklearn.svm import SVC

from qiskit.circuit import QuantumCircuit
from qiskit.circuit.library import z_feature_map
from qiskit.quantum_info import Statevector
from qiskit import transpile

from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import QiskitRuntimeService

# --------------------------- your choices ---------------------------
BACKEND_FOR_NOISE = "ibm_brisbane"      # or "ibm_torino"
INITIAL_LAYOUT    = [3, 4, 5, 6, 7, 8, 16]
RANDOM_STATE      = 42
TEST_SIZE         = 0.20
K                 = 7
REPS              = 1
ENTANGL           = "linear"
N_SET             = [32, 48, 64, 96, 128, 160]   # tweak for speed if needed
SHOTS_NOISY       = 1000                         # bump later to 2000–5000
TOLERANCE         = 0.005                        # 0.5 pp
CHUNK_CIRCS       = 256
# -------------------------------------------------------------------

# 0) Data: split, scale, RFE->K (train-only fit)
X, y = load_breast_cancer(return_X_y=True)
Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)
scaler = StandardScaler().fit(Xtr)
Xtr_s = scaler.transform(Xtr)
Xte_s = scaler.transform(Xte)

lr = LogisticRegression(max_iter=500, solver="lbfgs", random_state=RANDOM_STATE)
rfe = RFE(lr, n_features_to_select=K).fit(Xtr_s, ytr)
Xtr_k = rfe.transform(Xtr_s)
Xte_k = rfe.transform(Xte_s)

# 1) Feature map (modern function API)
fm = z_feature_map(feature_dimension=K, reps=REPS, entanglement=ENTANGL)

def _bind_feature_map(x):
    return fm.assign_parameters({p: float(val) for p, val in zip(fm.parameters, x)}).to_instruction()

# 2) Noiseless (statevector) kernel
def kernel_noiseless(A, B):
    sA = []
    for x in A:
        qc = QuantumCircuit(K); qc.append(_bind_feature_map(x), range(K))
        sA.append(Statevector.from_instruction(qc).data)
    sB = []
    for x in B:
        qc = QuantumCircuit(K); qc.append(_bind_feature_map(x), range(K))
        sB.append(Statevector.from_instruction(qc).data)
    sA, sB = np.asarray(sA), np.asarray(sB)
    return np.abs(sA @ np.conjugate(sB.T)) ** 2

# 3) Noisy (Aer) kernel — simulator cloned from the real backend, transpiled with your layout
service  = QiskitRuntimeService()
backend  = service.backend(BACKEND_FOR_NOISE)
sim      = AerSimulator.from_backend(backend)     # <-- V2 path, no deprecation warning
sim.set_options(seed_simulator=RANDOM_STATE)

def _fidelity_circuit(xi, xj):
    qc = QuantumCircuit(K)
    U_i = _bind_feature_map(xi)
    U_j = _bind_feature_map(xj)
    qc.append(U_i, range(K))
    qc.append(U_j.inverse(), range(K))
    qc.measure_all()
    return qc

def kernel_noisy(A, B, shots=SHOTS_NOISY, chunk=CHUNK_CIRCS):
    same = A is B or np.array_equal(A, B)
    nA, nB = len(A), len(B)
    Kmat = np.zeros((nA, nB), dtype=float)

    def run_batch(circs, pairs):
        # Transpile *to the simulator cloned from backend* but enforce your INITIAL_LAYOUT
        tcircs = transpile(
            circs,
            backend=sim,
            initial_layout=INITIAL_LAYOUT,
            layout_method="sabre",
            routing_method="sabre",
            optimization_level=0,
        )
        res = sim.run(tcircs, shots=shots).result()
        for r, (i, j) in enumerate(pairs):
            counts = res.get_counts(r)
            Kmat[i, j] = counts.get("0"*K, 0) / shots

    circs, pairs = [], []
    if same:
        for i in range(nA):
            Kmat[i, i] = 1.0  # exact by construction
            for j in range(i+1, nB):
                circs.append(_fidelity_circuit(A[i], B[j]))
                pairs.append((i, j))
                if len(circs) >= chunk:
                    run_batch(circs, pairs); circs, pairs = [], []
        if circs: run_batch(circs, pairs)
        # mirror upper->lower
        iu, ju = np.triu_indices(nA, 1)
        Kmat[ju, iu] = Kmat[iu, ju]
    else:
        for i in range(nA):
            for j in range(nB):
                circs.append(_fidelity_circuit(A[i], B[j]))
                pairs.append((i, j))
                if len(circs) >= chunk:
                    run_batch(circs, pairs); circs, pairs = [], []
        if circs: run_batch(circs, pairs)
    return Kmat

# 4) Train & metrics on precomputed kernel
def fit_metrics(Ktt, Kte, y_train, y_test):
    svc = SVC(kernel="precomputed")
    svc.fit(Ktt, y_train)
    y_hat = svc.predict(Kte)
    acc = accuracy_score(y_test, y_hat)
    f1  = f1_score(y_test, y_hat)
    tn, fp, fn, tp = confusion_matrix(y_test, y_hat, labels=[0,1]).ravel()
    try:
        scores = svc.decision_function(Kte)
        auc = roc_auc_score(y_test, scores)
    except Exception:
        auc = np.nan
    return dict(acc=acc, f1=f1, auc=auc, tp=int(tp), fp=int(fp), tn=int(tn), fn=int(fn))

# 5) Learning curve over N
results = []
for N in N_SET:
    sss = StratifiedShuffleSplit(n_splits=1, train_size=N, random_state=RANDOM_STATE)
    (sub_idx, _), = sss.split(Xtr_k, ytr)
    XN, yN = Xtr_k[sub_idx], ytr[sub_idx]

    print(f"\n=== N={N} (k={K}, layout={INITIAL_LAYOUT}) ===")
    t0 = perf_counter()
    Ktt_ideal = kernel_noiseless(XN, XN)
    Kte_ideal = kernel_noiseless(Xte_k, XN)
    noiseless = fit_metrics(Ktt_ideal, Kte_ideal, yN, yte)
    t1 = perf_counter()

    t2 = perf_counter()
    Ktt_noisy = kernel_noisy(XN, XN, shots=SHOTS_NOISY)
    Kte_noisy = kernel_noisy(Xte_k, XN, shots=SHOTS_NOISY)
    noisy = fit_metrics(Ktt_noisy, Kte_noisy, yN, yte)
    t3 = perf_counter()

    print(f"  Noiseless  : AUC={noiseless['auc']:.4f}  Acc={noiseless['acc']:.4f}  F1={noiseless['f1']:.4f}  "
          f"TP={noiseless['tp']} FP={noiseless['fp']} TN={noiseless['tn']} FN={noiseless['fn']}  "
          f"({t1-t0:.1f}s)")
    print(f"  Noisy (Aer): AUC={noisy['auc']:.4f}  Acc={noisy['acc']:.4f}  F1={noisy['f1']:.4f}  "
          f"TP={noisy['tp']} FP={noisy['fp']} TN={noisy['tn']} FN={noisy['fn']}  "
          f"({t3-t2:.1f}s, shots={SHOTS_NOISY})")

    results.append((N, noiseless, noisy))

# 6) Pick smallest N* within tolerance of best NOISY AUC
noisy_aucs = [r[2]['auc'] for r in results]
best_noisy_auc = np.nanmax(noisy_aucs)
candidates = [N for (N, _, noisy) in results if noisy['auc'] >= best_noisy_auc - TOLERANCE]
N_star = min(candidates) if candidates else N_SET[-1]

print("\n=== Selection ===")
print(f"Best NOISY AUC = {best_noisy_auc:.4f}")
print(f"N* (smallest within {TOLERANCE*100:.1f} pp of best NOISY AUC) = {N_star}")

print("\nSummary:")
for (N, ideal, noisy) in results:
    print(f"N={N:<4}  "
          f"ideal AUC={ideal['auc']:.4f} Acc={ideal['acc']:.4f} F1={ideal['f1']:.4f} "
          f"TP={ideal['tp']} FP={ideal['fp']} TN={ideal['tn']} FN={ideal['fn']} | "
          f"noisy AUC={noisy['auc']:.4f} Acc={noisy['acc']:.4f} F1={noisy['f1']:.4f} "
          f"TP={noisy['tp']} FP={noisy['fp']} TN={noisy['tn']} FN={noisy['fn']}")


In [ ]:
# ====== Minimal-training-size finder (WBCD, k=7) with your backend+layout ======
# Uses:
#   - Noiseless kernel (statevector)
#   - Noisy kernel (AerSimulator.from_backend(ibm_brisbane)), transpiled with your initial_layout
# Prints TP/FP/TN/FN, Acc, F1, AUC per N, then selects the smallest N* within tol of best NOISY AUC.

import numpy as np
from time import perf_counter
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix
from sklearn.svm import SVC

from qiskit.circuit import QuantumCircuit
from qiskit.circuit.library import z_feature_map
from qiskit.quantum_info import Statevector
from qiskit import transpile

from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import QiskitRuntimeService

# --------------------------- your choices ---------------------------
BACKEND_FOR_NOISE = "ibm_brisbane"      # or "ibm_torino"
INITIAL_LAYOUT    = [3, 4, 5, 6, 7, 8, 16]
RANDOM_STATE      = 42
TEST_SIZE         = 0.20
K                 = 7
REPS              = 1
ENTANGL           = "linear"
N_SET             = [ 64, 96, 128]   # tweak for speed if needed
SHOTS_NOISY       = 256 #1000                         # bump later to 2000–5000
TOLERANCE         = 0.005                        # 0.5 pp
CHUNK_CIRCS       = 256
# -------------------------------------------------------------------

# 0) Data: split, scale, RFE->K (train-only fit)
X, y = load_breast_cancer(return_X_y=True)
Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)
scaler = StandardScaler().fit(Xtr)
Xtr_s = scaler.transform(Xtr)
Xte_s = scaler.transform(Xte)

lr = LogisticRegression(max_iter=500, solver="lbfgs", random_state=RANDOM_STATE)
rfe = RFE(lr, n_features_to_select=K).fit(Xtr_s, ytr)
Xtr_k = rfe.transform(Xtr_s)
Xte_k = rfe.transform(Xte_s)

# 1) Feature map (modern function API)
fm = z_feature_map(feature_dimension=K, reps=REPS, entanglement=ENTANGL)

def _bind_feature_map(x):
    return fm.assign_parameters({p: float(val) for p, val in zip(fm.parameters, x)}).to_instruction()

# 2) Noiseless (statevector) kernel
def kernel_noiseless(A, B):
    sA = []
    for x in A:
        qc = QuantumCircuit(K); qc.append(_bind_feature_map(x), range(K))
        sA.append(Statevector.from_instruction(qc).data)
    sB = []
    for x in B:
        qc = QuantumCircuit(K); qc.append(_bind_feature_map(x), range(K))
        sB.append(Statevector.from_instruction(qc).data)
    sA, sB = np.asarray(sA), np.asarray(sB)
    return np.abs(sA @ np.conjugate(sB.T)) ** 2

# 3) Noisy (Aer) kernel — simulator cloned from the real backend, transpiled with your layout
service  = QiskitRuntimeService()
backend  = service.backend(BACKEND_FOR_NOISE)
sim      = AerSimulator.from_backend(backend)     # <-- V2 path, no deprecation warning
sim.set_options(seed_simulator=RANDOM_STATE)

def _fidelity_circuit(xi, xj):
    qc = QuantumCircuit(K)
    U_i = _bind_feature_map(xi)
    U_j = _bind_feature_map(xj)
    qc.append(U_i, range(K))
    qc.append(U_j.inverse(), range(K))
    qc.measure_all()
    return qc

def kernel_noisy(A, B, shots=SHOTS_NOISY, chunk=CHUNK_CIRCS):
    same = A is B or np.array_equal(A, B)
    nA, nB = len(A), len(B)
    Kmat = np.zeros((nA, nB), dtype=float)

    def run_batch(circs, pairs):
        # Transpile *to the simulator cloned from backend* but enforce your INITIAL_LAYOUT
        tcircs = transpile(
            circs,
            backend=sim,
            initial_layout=INITIAL_LAYOUT,
            layout_method="sabre",
            routing_method="sabre",
            optimization_level=0,
        )
        res = sim.run(tcircs, shots=shots).result()
        for r, (i, j) in enumerate(pairs):
            counts = res.get_counts(r)
            Kmat[i, j] = counts.get("0"*K, 0) / shots

    circs, pairs = [], []
    if same:
        for i in range(nA):
            Kmat[i, i] = 1.0  # exact by construction
            for j in range(i+1, nB):
                circs.append(_fidelity_circuit(A[i], B[j]))
                pairs.append((i, j))
                if len(circs) >= chunk:
                    run_batch(circs, pairs); circs, pairs = [], []
        if circs: run_batch(circs, pairs)
        # mirror upper->lower
        iu, ju = np.triu_indices(nA, 1)
        Kmat[ju, iu] = Kmat[iu, ju]
    else:
        for i in range(nA):
            for j in range(nB):
                circs.append(_fidelity_circuit(A[i], B[j]))
                pairs.append((i, j))
                if len(circs) >= chunk:
                    run_batch(circs, pairs); circs, pairs = [], []
        if circs: run_batch(circs, pairs)
    return Kmat

# 4) Train & metrics on precomputed kernel
def fit_metrics(Ktt, Kte, y_train, y_test):
    svc = SVC(kernel="precomputed")
    svc.fit(Ktt, y_train)
    y_hat = svc.predict(Kte)
    acc = accuracy_score(y_test, y_hat)
    f1  = f1_score(y_test, y_hat)
    tn, fp, fn, tp = confusion_matrix(y_test, y_hat, labels=[0,1]).ravel()
    try:
        scores = svc.decision_function(Kte)
        auc = roc_auc_score(y_test, scores)
    except Exception:
        auc = np.nan
    return dict(acc=acc, f1=f1, auc=auc, tp=int(tp), fp=int(fp), tn=int(tn), fn=int(fn))

# 5) Learning curve over N
results = []
for N in N_SET:
    sss = StratifiedShuffleSplit(n_splits=1, train_size=N, random_state=RANDOM_STATE)
    (sub_idx, _), = sss.split(Xtr_k, ytr)
    XN, yN = Xtr_k[sub_idx], ytr[sub_idx]

    print(f"\n=== N={N} (k={K}, layout={INITIAL_LAYOUT}) ===")
    t0 = perf_counter()
    Ktt_ideal = kernel_noiseless(XN, XN)
    Kte_ideal = kernel_noiseless(Xte_k, XN)
    noiseless = fit_metrics(Ktt_ideal, Kte_ideal, yN, yte)
    t1 = perf_counter()

    t2 = perf_counter()
    Ktt_noisy = kernel_noisy(XN, XN, shots=SHOTS_NOISY)
    Kte_noisy = kernel_noisy(Xte_k, XN, shots=SHOTS_NOISY)
    noisy = fit_metrics(Ktt_noisy, Kte_noisy, yN, yte)
    t3 = perf_counter()

    print(f"  Noiseless  : AUC={noiseless['auc']:.4f}  Acc={noiseless['acc']:.4f}  F1={noiseless['f1']:.4f}  "
          f"TP={noiseless['tp']} FP={noiseless['fp']} TN={noiseless['tn']} FN={noiseless['fn']}  "
          f"({t1-t0:.1f}s)")
    print(f"  Noisy (Aer): AUC={noisy['auc']:.4f}  Acc={noisy['acc']:.4f}  F1={noisy['f1']:.4f}  "
          f"TP={noisy['tp']} FP={noisy['fp']} TN={noisy['tn']} FN={noisy['fn']}  "
          f"({t3-t2:.1f}s, shots={SHOTS_NOISY})")

    results.append((N, noiseless, noisy))

# 6) Pick smallest N* within tolerance of best NOISY AUC
noisy_aucs = [r[2]['auc'] for r in results]
best_noisy_auc = np.nanmax(noisy_aucs)
candidates = [N for (N, _, noisy) in results if noisy['auc'] >= best_noisy_auc - TOLERANCE]
N_star = min(candidates) if candidates else N_SET[-1]

print("\n=== Selection ===")
print(f"Best NOISY AUC = {best_noisy_auc:.4f}")
print(f"N* (smallest within {TOLERANCE*100:.1f} pp of best NOISY AUC) = {N_star}")

print("\nSummary:")
for (N, ideal, noisy) in results:
    print(f"N={N:<4}  "
          f"ideal AUC={ideal['auc']:.4f} Acc={ideal['acc']:.4f} F1={ideal['f1']:.4f} "
          f"TP={ideal['tp']} FP={ideal['fp']} TN={ideal['tn']} FN={ideal['fn']} | "
          f"noisy AUC={noisy['auc']:.4f} Acc={noisy['acc']:.4f} F1={noisy['f1']:.4f} "
          f"TP={noisy['tp']} FP={noisy['fp']} TN={noisy['tn']} FN={noisy['fn']}")


This cell:

uses FidelityQuantumKernel with a ZFeatureMap (7 qubits, reps=1),

evaluates CV AUC at shots = [256, 512, 1024, 2048],

prints the mean AUC so you can pick the smallest shots where AUC stabilizes.

Works locally with Aer Sampler (no IBM account). If you later run on real hardware, replace the sampler creation with an IBM SamplerV2 and set default_shots there.

In [ ]:
# Works with: qiskit==2.2.1, qiskit-aer==0.17.2, qiskit-ibm-runtime==0.42.0, qiskit-machine-learning==0.8.2
# Goal: Noisy-Aer shots loop (256→512→1024→2048) using ibm_brisbane noise, N=96, 7 named features, saving K and timings.

import os, time
import numpy as np
import pandas as pd
from collections import Counter

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score

from qiskit import QuantumCircuit, ClassicalRegister, transpile
from qiskit.circuit import Parameter
from qiskit.circuit.library import z_feature_map  # function API (no deprecation warning)

from qiskit_aer import AerSimulator
from qiskit_aer.primitives import SamplerV2 as AerSamplerV2  # Aer V2 primitive

from qiskit_ibm_runtime import QiskitRuntimeService  # only to fetch backend props for noise

# -------------------------------
# Settings
# -------------------------------
BACKEND_NAME    = "ibm_brisbane"
INITIAL_LAYOUT  = [3, 4, 5, 6, 7, 8, 16]
SHOTS_GRID      = [128 ,256, 512, 1024, 2048]
N               = 96
N_SPLITS        = 3
RANDOM_SEED     = 42
OUTDIR          = "./kernels_noisy_aer_brisbane"
os.makedirs(OUTDIR, exist_ok=True)

FEATURES = [
    'mean texture',
    'mean concave points',
    'worst radius',
    'worst texture',
    'worst smoothness',
    'worst concave points',
    'worst symmetry',
]

# -------------------------------
# 1) Data: pick EXACT 7 features and fix N=96
# -------------------------------
X_df, y_sr = load_breast_cancer(return_X_y=True, as_frame=True)
missing = [n for n in FEATURES if n not in X_df.columns]
if missing:
    raise ValueError(f"Missing features: {missing}")

X_all = X_df[FEATURES].to_numpy()
y_all = (y_sr.values == 1).astype(int)

scaler = StandardScaler().fit(X_all)
X_all = scaler.transform(X_all)

sss = StratifiedShuffleSplit(n_splits=1, train_size=N, random_state=RANDOM_SEED)
(train_idx, _), = sss.split(X_all, y_all)
X = X_all[train_idx]
y = y_all[train_idx]
print(f"[INFO] Using N={N} samples. Class balance: {Counter(int(v) for v in y)}")

# -------------------------------
# 2) Noisy Aer simulator from real backend properties (local sim; needs saved account)
#    If you cannot connect, replace the next 3 lines with:
#    from qiskit_ibm_runtime.fake_provider import FakeBrisbane
#    fake = FakeBrisbane(); aer_sim = AerSimulator.from_backend(fake)
#    coupling_map = fake.coupling_map
# -------------------------------
service = QiskitRuntimeService()
backend = service.backend(BACKEND_NAME)
aer_sim = AerSimulator.from_backend(backend)     # includes basis, coupling, and a basic noise model
coupling_map = backend.coupling_map

# -------------------------------
# 3) Compute–uncompute kernel circuit (Z feature map, reps=1) WITHOUT parameter_map
#    U(z) • U(x)^\dagger, with distinct param prefixes 'z' and 'x'
# -------------------------------
nq = 7
fm_z = z_feature_map(feature_dimension=nq, reps=1, parameter_prefix='z')
fm_x = z_feature_map(feature_dimension=nq, reps=1, parameter_prefix='x')  # we will invert it

qc = QuantumCircuit(nq, name="fid")
qc.compose(fm_z, qubits=range(nq), inplace=True)
qc.compose(fm_x.inverse(), qubits=range(nq), inplace=True)

cr = ClassicalRegister(nq, "meas")
qc.add_register(cr)
qc.measure(range(nq), cr)

# Transpile once to Brisbane ISA with your initial layout (so sim mirrors device routing)
t0_build = time.perf_counter()
qc_t = transpile(
    qc,
    backend=aer_sim,               # uses simulator basis/ISA (incl. noise-backed basis)
    coupling_map=coupling_map,
    initial_layout=INITIAL_LAYOUT,
    optimization_level=1,
)
t1_build = time.perf_counter()
print(f"[INFO] Transpiled for Brisbane-like ISA with layout {INITIAL_LAYOUT}. Time: {t1_build - t0_build:.2f}s")

# Helper: extract transpiled Parameter objects by prefix
def extract_params_by_prefix(circ: QuantumCircuit, prefix: str):
    plist = [p for p in circ.parameters if p.name.startswith(f"{prefix}[")]
    # names like 'x[0]', 'z[3]' -> sort by integer index
    plist.sort(key=lambda p: int(p.name[p.name.find('[')+1:-1]))
    return plist

params_x = extract_params_by_prefix(qc_t, "x")
params_z = extract_params_by_prefix(qc_t, "z")

# -------------------------------
# 4) SamplerV2 helpers (noisy Aer)
# -------------------------------
def make_sampler(shots: int) -> AerSamplerV2:
    try:
        return AerSamplerV2.from_backend(aer_sim, default_shots=shots)
    except TypeError:
        return AerSamplerV2(backend=aer_sim, default_shots=shots)

def pubs_for_cross(A, B):
    """List of PUBs (circuit, param_dict) for all pairs in A x B."""
    pubs = []
    for i in range(A.shape[0]):
        for j in range(B.shape[0]):
            pmap = {}
            # assign U(z) params from B[j], U(x) params from A[i]
            for k in range(nq):
                pmap[params_z[k]] = float(B[j, k])
                pmap[params_x[k]] = float(A[i, k])
            pubs.append((qc_t, pmap))
    return pubs

def kernel_cross(sampler: AerSamplerV2, A, B, shots, chunk=256):
    """Compute raw K(A,B) via SamplerV2; chunked submission."""
    K = np.zeros((A.shape[0], B.shape[0]), dtype=float)
    pubs = pubs_for_cross(A, B)
    idx = 0
    for start in range(0, len(pubs), chunk):
        sub = pubs[start:start+chunk]
        job = sampler.run(sub)  # shots provided by default_shots
        res = job.result()
        for r in res:
            # V2 result -> combined counts
            try:
                counts = r.join_data().get_counts()
            except Exception:
                # conservative fallback
                counts = next(iter(r.data.values())).get_counts()
            p0 = counts.get("0"*nq, 0) / shots
            i = idx // B.shape[0]
            j = idx %  B.shape[0]
            K[i, j] = p0
            idx += 1
    return K

def kernel_diag(sampler: AerSamplerV2, A, shots, chunk=256):
    """Self-overlaps (diagonal) for A: returns vector d with raw K_ii."""
    pubs = []
    for i in range(A.shape[0]):
        pmap = {}
        for k in range(nq):
            # both z and x set to the same A[i] for ⟨φ(A[i])|φ(A[i])⟩
            pmap[params_z[k]] = float(A[i, k])
            pmap[params_x[k]] = float(A[i, k])
        pubs.append((qc_t, pmap))
    d = np.zeros(A.shape[0], dtype=float)
    for start in range(0, len(pubs), chunk):
        sub = pubs[start:start+chunk]
        job = sampler.run(sub)
        res = job.result()
        for offs, r in enumerate(res):
            try:
                counts = r.join_data().get_counts()
            except Exception:
                counts = next(iter(r.data.values())).get_counts()
            d[start + offs] = counts.get("0"*nq, 0) / shots
    return d

def normalize_square_from_diag(K_raw, d):
    """Normalize square kernel using its own diagonal; force symmetry & diag=1."""
    Kn = K_raw / np.sqrt(np.outer(d, d))
    Kn = 0.5 * (Kn + Kn.T)
    np.fill_diagonal(Kn, 1.0)
    return Kn

def normalize_cross_from_diag(K_raw, d_test, d_train):
    """Normalize cross kernel using test and train diagonals."""
    denom = np.sqrt(np.outer(d_test, d_train))
    return K_raw / denom

# -------------------------------
# 5) CV loop over shots, save matrices and timings
# -------------------------------
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_SEED)
prev_mean = None
TOL = 0.005  # 0.5 percentage point

for shots in SHOTS_GRID:
    sampler = make_sampler(shots)
    fold_aucs, fold_ksecs, fold_tsecs = [], [], []
    print(f"\n=== SHOTS = {shots} ===")

    for fold_id, (tr_idx, te_idx) in enumerate(cv.split(X, y), 1):
        X_tr, X_te = X[tr_idx], X[te_idx]
        y_tr, y_te = y[tr_idx], y[te_idx]

        t0 = time.perf_counter()

        # --- Train square kernel (raw), its diagonal, then normalize
        t0k = time.perf_counter()
        K_tr_raw = kernel_cross(sampler, X_tr, X_tr, shots)
        d_tr     = np.clip(np.diag(K_tr_raw), 1e-12, None)
        K_tr     = normalize_square_from_diag(K_tr_raw, d_tr)

        # --- Test diagonal (raw self-overlaps)
        d_te = kernel_diag(sampler, X_te, shots)

        # --- Test-vs-train cross (raw), then normalize with d_te & d_tr
        K_te_raw = kernel_cross(sampler, X_te, X_tr, shots)
        K_te     = normalize_cross_from_diag(K_te_raw, d_te, d_tr)

        t1k = time.perf_counter()

        # Save matrices
        pd.DataFrame(K_tr).to_csv(os.path.join(OUTDIR, f"K_train_shots{shots}_fold{fold_id}.csv"), index=False)
        pd.DataFrame(K_te).to_csv(os.path.join(OUTDIR, f"K_test_shots{shots}_fold{fold_id}.csv"), index=False)

        # Train SVM (precomputed kernel)
        clf = SVC(kernel="precomputed", C=1.0)
        clf.fit(K_tr, y_tr)
        scores = clf.decision_function(K_te)
        auc = roc_auc_score(y_te, scores)

        t1 = time.perf_counter()

        fold_aucs.append(auc)
        fold_ksecs.append(t1k - t0k)
        fold_tsecs.append(t1 - t0)
        print(f"  fold {fold_id}: AUC={auc:.4f} | kernel={fold_ksecs[-1]:.2f}s | total={fold_tsecs[-1]:.2f}s")

    mean_auc, std_auc = float(np.mean(fold_aucs)), float(np.std(fold_aucs))
    mean_k, mean_t    = float(np.mean(fold_ksecs)), float(np.mean(fold_tsecs))
    delta             = None if prev_mean is None else mean_auc - prev_mean
    print(f"== shots={shots} | mean AUC={mean_auc:.4f} (+/- {std_auc:.4f}) "
          f"| mean kernel/fold={mean_k:.2f}s | mean total/fold={mean_t:.2f}s"
          + ("" if delta is None else f" | Δ vs prev={delta:+.4f}"))
    if prev_mean is not None and abs(delta) < TOL:
        print(f"--> AUC stabilized (|Δ|<{TOL}); keep shots={shots}.")
        break
    prev_mean = mean_auc

print(f"\n[SAVED] CSVs in: {os.path.abspath(OUTDIR)}")


In [ ]:
# Qiskit pins: qiskit==2.2.1, qiskit-aer==0.17.2, qiskit-ibm-runtime==0.42.0, qiskit-machine-learning==0.8.2
# Mode: Full QSVM (N=96), no Nyström. M3 mitigation calibrated on AER (local), optional DD (local),
# adaptive shots (256 -> top-up ≤ 1025). No SamplerV2 (we use AerSimulator.run counts-only to avoid MemError).
# Saves: K_train/K_test CSVs + per-entry long-form CSV with p0_mitigated and shots_total.
# Prints per-fold AUC/Acc/F1/TP/FP/TN/FN and a summary.

import os, time, math
import numpy as np
import pandas as pd
from collections import Counter

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, confusion_matrix

from qiskit import QuantumCircuit, ClassicalRegister, transpile
from qiskit.circuit.library import z_feature_map
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import QiskitRuntimeService

# --- Optional measurement mitigation (M3) ---
try:
    import mthree as m3
    M3_AVAILABLE = True
except ModuleNotFoundError:
    M3_AVAILABLE = False

# -------------------------------
# Toggles / configuration
# -------------------------------
BACKEND_NAME    = "ibm_brisbane"
INITIAL_LAYOUT  = [3, 4, 5, 6, 7, 8, 16]

USE_M3          = True        # enable measurement mitigation
M3_SOURCE       = "aer"       # "aer" (local; recommended for sim) or "hardware" (queues jobs)
USE_DD          = True        # optional: schedule + pad with DD
USE_ADAPTIVE    = True        # two-pass adaptive shots per K entry

BASE_SHOTS      = 256         # base shots
SIGMA_TARGET    = 0.0125      # target SE (~1.25%); set None to disable top-up
MAX_SHOTS       = 1025        # cap per entry (base + top-up)

# Execution batching (small to avoid RAM spikes)
RUN_BATCH       = 32          # circuits per backend.run() call
KERNEL_CHUNK    = 256         # pubs per outer chunk

N               = 96
N_SPLITS        = 3
RANDOM_SEED     = 42
OUTDIR          = "./kernels_noisy_aer_brisbane"
SAVE_PER_ENTRY_DETAILS = True
os.makedirs(OUTDIR, exist_ok=True)

FEATURES = [
    'mean texture',
    'mean concave points',
    'worst radius',
    'worst texture',
    'worst smoothness',
    'worst concave points',
    'worst symmetry',
]

# -------------------------------
# 1) Data (7 features, N=96)
# -------------------------------
X_df, y_sr = load_breast_cancer(return_X_y=True, as_frame=True)
missing = [n for n in FEATURES if n not in X_df.columns]
if missing:
    raise ValueError(f"Missing features: {missing}")

X_all = X_df[FEATURES].to_numpy()
y_all = (y_sr.values == 1).astype(int)

scaler = StandardScaler().fit(X_all)
X_all = scaler.transform(X_all)

sss = StratifiedShuffleSplit(n_splits=1, train_size=N, random_state=RANDOM_SEED)
(train_idx, _), = sss.split(X_all, y_all)
X = X_all[train_idx]
y = y_all[train_idx]
print(f"[INFO] Using N={N} samples. Class balance: {Counter(int(v) for v in y)}")

# -------------------------------
# 2) Device-like noisy simulator (local)
# -------------------------------
service = QiskitRuntimeService()                # fetch backend props (no jobs)
backend = service.backend(BACKEND_NAME)
aer_sim = AerSimulator.from_backend(backend)    # basis, coupling, basic device noise

# Conservative parallelism to avoid RAM pressure
try:
    import multiprocessing as mp
    aer_sim.set_options(
        max_parallel_experiments=1,
        max_parallel_shots=1,
        max_parallel_threads=min(4, mp.cpu_count()),
    )
except Exception:
    pass

# -------------------------------
# 3) Kernel circuit: U(z) • U(x)^† (7 qubits, reps=1)
# -------------------------------
nq = 7
fm_z = z_feature_map(feature_dimension=nq, reps=1, parameter_prefix='z')
fm_x = z_feature_map(feature_dimension=nq, reps=1, parameter_prefix='x')  # will invert

qc = QuantumCircuit(nq, name="fid")
qc.compose(fm_z, qubits=range(nq), inplace=True)
qc.compose(fm_x.inverse(), qubits=range(nq), inplace=True)
cr = ClassicalRegister(nq, "meas")
qc.add_register(cr)
qc.measure(range(nq), cr)

# Transpile (no coupling_map/basis_gates when backend is provided)
t0_build = time.perf_counter()
qc_t = transpile(
    qc,
    backend=aer_sim,
    initial_layout=INITIAL_LAYOUT,
    optimization_level=1,
)

# Optional: DD (schedule with device durations; still local)
if USE_DD:
    from qiskit.transpiler import InstructionDurations, PassManager
    from qiskit.transpiler.passes import ALAPScheduleAnalysis, PadDynamicalDecoupling
    from qiskit.circuit.library import XGate
    durations = InstructionDurations.from_backend(backend)
    pm = PassManager([
        ALAPScheduleAnalysis(durations),
        PadDynamicalDecoupling(durations, [XGate(), XGate()])
    ])
    qc_t = pm.run(qc_t)

t1_build = time.perf_counter()
print(f"[INFO] Prepared circuit (transpile{' + DD' if USE_DD else ''}). Time: {t1_build - t0_build:.2f}s")

# -------------------------------
# 4) Parameters in transpiled circuit
# -------------------------------
def extract_params_by_prefix(circ, prefix: str):
    plist = [p for p in circ.parameters if p.name.startswith(f"{prefix}[")]
    plist.sort(key=lambda p: int(p.name[p.name.find('[')+1:-1]))
    return plist
params_x = extract_params_by_prefix(qc_t, "x")
params_z = extract_params_by_prefix(qc_t, "z")

# -------------------------------
# 5) M3 mitigator + execution utilities
# -------------------------------
if USE_M3 and not M3_AVAILABLE:
    print("[WARN] 'mthree' not installed — running WITHOUT measurement mitigation. pip install mthree")
if USE_M3 and M3_AVAILABLE:
    if M3_SOURCE.lower() == "hardware":
        mit = m3.M3Mitigation(backend)          # WARNING: queues a real calibration job
        print("[WARN] M3_SOURCE='hardware' will queue a calibration job on the device.")
    else:
        mit = m3.M3Mitigation(aer_sim)          # LOCAL calibration on the simulator (no queue)
    physical_qubits = INITIAL_LAYOUT[:7]
    mit.cals_from_system(physical_qubits)
else:
    mit = None
    physical_qubits = None

def counts_to_p0(counts_dict):
    shots_used = sum(counts_dict.values())
    return counts_dict.get("0"*nq, 0) / max(1, shots_used)

def _p0_from_quasi(quasi, nq):
    # robust across mthree versions: prefer integer key 0; fallbacks to '000...0', list-like, or .probabilities().
    try:
        if 0 in quasi:
            return float(quasi.get(0, 0.0))
    except Exception:
        pass
    bit0 = "0"*nq
    if hasattr(quasi, "get"):
        val = quasi.get(0, quasi.get(bit0, None))
        if val is not None:
            return float(val)
    try:
        return float(quasi[0])
    except Exception:
        pass
    try:
        probs = quasi.probabilities()
        try:
            return float(probs[0])
        except Exception:
            return float(probs.get(0, probs.get(bit0, 0.0)))
    except Exception:
        pass
    return 0.0

def mitigate_p0(counts_dict):
    if mit is None:
        return counts_to_p0(counts_dict)
    quasi = mit.apply_correction(counts_dict, physical_qubits)  # single object, no unpacking
    return _p0_from_quasi(quasi, nq)

def pub_for_pair(Arow, Brow):
    pmap = {}
    for k in range(nq):
        pmap[params_z[k]] = float(Brow[k])
        pmap[params_x[k]] = float(Arow[k])
    return (qc_t, pmap)

def run_counts_pubs(pubs, shots, batch=RUN_BATCH, seed=1234):
    """Execute pubs in small batches using AerSimulator.run; return list of counts dicts."""
    out = []
    for start in range(0, len(pubs), batch):
        sub = pubs[start:start+batch]
        # FIX: use assign_parameters (bind_parameters may not exist in your build)
        bound = [circ.assign_parameters(pmap, inplace=False) for (circ, pmap) in sub]
        job = aer_sim.run(bound, shots=shots, seed_simulator=seed)
        res = job.result()
        counts = res.get_counts()
        if isinstance(counts, dict):  # single circuit case
            out.append(counts)
        else:
            out.extend(counts)
    return out

# Adaptive per-chunk, returns (p0, shots_total) for each pub
def p0s_with_adaptive_and_shots(pubs, base_shots, sigma, shots_max):
    # Pass 1
    counts1 = run_counts_pubs(pubs, shots=base_shots)
    if sigma is None:
        return [ (mitigate_p0(c), sum(c.values())) for c in counts1 ]

    # Decide top-ups from raw p-hat
    p_hat = [counts_to_p0(c) for c in counts1]
    adds = []
    for ph in p_hat:
        var = ph*(1-ph)
        nstar = math.ceil(min(0.25, var) / (sigma**2))
        extra = max(0, nstar - base_shots)
        if shots_max is not None:
            extra = min(extra, max(0, shots_max - base_shots))
        adds.append(extra)

    out = []
    # Prepare extras only for those that need it
    to_run_pubs, to_run_idx, to_run_shots = [], [], []
    for idx, extra in enumerate(adds):
        if extra > 0:
            to_run_pubs.append(pubs[idx])
            to_run_idx.append(idx)
            to_run_shots.append(extra)

    # Run extras grouped by shot count
    extras_counts = {}
    if to_run_pubs:
        from collections import defaultdict
        groups = defaultdict(list)
        for i, shots in zip(to_run_idx, to_run_shots):
            groups[shots].append(i)
        for shots, indices in groups.items():
            sub_pubs = [pubs[i] for i in indices]
            sub_counts = run_counts_pubs(sub_pubs, shots=shots)
            for i, cnt in zip(indices, sub_counts):
                extras_counts[i] = cnt

    # Aggregate and mitigate
    for idx, (c0, extra) in enumerate(zip(counts1, adds)):
        if extra > 0:
            c1 = extras_counts[idx]
            c_tot = dict(c0)
            for k, v in c1.items():
                c_tot[k] = c_tot.get(k, 0) + v
            out.append( (mitigate_p0(c_tot), sum(c_tot.values())) )
        else:
            out.append( (mitigate_p0(c0), sum(c0.values())) )
    return out

# Upper-triangle train kernel (raw -> normalized), logs per-entry CSV rows
def kernel_square_upper(A, base_shots, sigma, shots_max, chunk=KERNEL_CHUNK, details=None):
    n = A.shape[0]
    K = np.zeros((n, n), dtype=float)
    idx_pairs, pubs = [], []
    for i in range(n):
        for j in range(i, n):
            idx_pairs.append((i, j))
            pubs.append(pub_for_pair(A[i], A[j]))

    for start in range(0, len(pubs), chunk):
        sub = pubs[start:start+chunk]
        vals = p0s_with_adaptive_and_shots(sub, base_shots, sigma, shots_max)
        for (p0, shots_total), (i, j) in zip(vals, idx_pairs[start:start+chunk]):
            K[i, j] = p0;  K[j, i] = p0
            if details is not None:
                details.append({"block":"train", "i":int(i), "j":int(j),
                                "p0_mitigated":float(p0), "shots_total":int(shots_total)})
    return K

def kernel_cross(A, B, base_shots, sigma, shots_max, chunk=KERNEL_CHUNK, details=None):
    K = np.zeros((A.shape[0], B.shape[0]), dtype=float)
    pubs = [pub_for_pair(A[i], B[j]) for i in range(A.shape[0]) for j in range(B.shape[0])]
    idx = 0
    for start in range(0, len(pubs), chunk):
        sub = pubs[start:start+chunk]
        vals = p0s_with_adaptive_and_shots(sub, base_shots, sigma, shots_max)
        for (p0, shots_total) in vals:
            i = idx // B.shape[0]; j = idx % B.shape[0]
            K[i, j] = p0
            if details is not None:
                details.append({"block":"testxtrain", "i":int(i), "j":int(j),
                                "p0_mitigated":float(p0), "shots_total":int(shots_total)})
            idx += 1
    return K

def kernel_diag(A, base_shots, sigma, shots_max, chunk=KERNEL_CHUNK, details=None):
    pubs = [pub_for_pair(A[i], A[i]) for i in range(A.shape[0])]
    d = np.zeros(A.shape[0], dtype=float)
    for start in range(0, len(pubs), chunk):
        sub = pubs[start:start+chunk]
        vals = p0s_with_adaptive_and_shots(sub, base_shots, sigma, shots_max)
        for offs, (p0, shots_total) in enumerate(vals):
            d[start + offs] = p0
            if details is not None:
                details.append({"block":"diag", "i":int(start+offs), "j":int(start+offs),
                                "p0_mitigated":float(p0), "shots_total":int(shots_total)})
    return d

def normalize_square_from_diag(K_raw, d):
    Kn = K_raw / np.sqrt(np.outer(d, d))
    Kn = 0.5 * (Kn + Kn.T)
    np.fill_diagonal(Kn, 1.0)
    return Kn

def normalize_cross_from_diag(K_raw, d_test, d_train):
    denom = np.sqrt(np.outer(d_test, d_train))
    return K_raw / denom

# -------------------------------
# 6) CV run (base=256; adaptive top-up ≤ 1025)
# -------------------------------
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_SEED)

print(f"[INFO] Circuit ready | M3={'ON' if (USE_M3 and M3_AVAILABLE) else 'OFF'} (source={M3_SOURCE}) "
      f"| DD={'ON' if USE_DD else 'OFF'} | Adaptive={'ON' if USE_ADAPTIVE else 'OFF'}")
print(f"[INFO] Base shots={BASE_SHOTS}, SigmaTarget={SIGMA_TARGET}, Max shots={MAX_SHOTS} | "
      f"RUN_BATCH={RUN_BATCH}, KERNEL_CHUNK={KERNEL_CHUNK}")

fold_aucs, fold_accs, fold_f1s = [], [], []
agg_tn = agg_fp = agg_fn = agg_tp = 0
fold_ksecs, fold_tsecs = [], []

for fold_id, (tr_idx, te_idx) in enumerate(cv.split(X, y), 1):
    X_tr, X_te = X[tr_idx], X[te_idx]
    y_tr, y_te = y[tr_idx], y[te_idx]

    DETAILS = [] if SAVE_PER_ENTRY_DETAILS else None

    t0 = time.perf_counter()

    # Train square (upper triangle) -> diag -> normalize
    t0k = time.perf_counter()
    K_tr_raw = kernel_square_upper(X_tr, BASE_SHOTS,
                                   (SIGMA_TARGET if USE_ADAPTIVE else None), MAX_SHOTS, details=DETAILS)
    d_tr     = np.clip(np.diag(K_tr_raw), 1e-12, None)
    K_tr     = normalize_square_from_diag(K_tr_raw, d_tr)

    # Test diag and cross, then normalize
    d_te     = kernel_diag(X_te, BASE_SHOTS,
                           (SIGMA_TARGET if USE_ADAPTIVE else None), MAX_SHOTS, details=DETAILS)
    K_te_raw = kernel_cross(X_te, X_tr, BASE_SHOTS,
                            (SIGMA_TARGET if USE_ADAPTIVE else None), MAX_SHOTS, details=DETAILS)
    K_te     = normalize_cross_from_diag(K_te_raw, d_te, d_tr)
    t1k = time.perf_counter()

    # Save matrices
    pd.DataFrame(K_tr).to_csv(os.path.join(OUTDIR, f"K_train_shots{BASE_SHOTS}_fold{fold_id}.csv"), index=False)
    pd.DataFrame(K_te).to_csv(os.path.join(OUTDIR, f"K_test_shots{BASE_SHOTS}_fold{fold_id}.csv"), index=False)
    if SAVE_PER_ENTRY_DETAILS and DETAILS:
        pd.DataFrame(DETAILS).to_csv(os.path.join(OUTDIR, f"K_entries_shots{BASE_SHOTS}_fold{fold_id}.csv"), index=False)

    # QSVM classifier
    clf = SVC(kernel="precomputed", C=1.0)
    clf.fit(K_tr, y_tr)
    scores = clf.decision_function(K_te)
    y_pred = (scores >= 0).astype(int)

    # Metrics
    auc = roc_auc_score(y_te, scores)
    acc = accuracy_score(y_te, y_pred)
    f1  = f1_score(y_te, y_pred, average="binary", pos_label=1)
    tn, fp, fn, tp = confusion_matrix(y_te, y_pred, labels=[0,1]).ravel()

    t1 = time.perf_counter()

    fold_aucs.append(auc)
    fold_accs.append(acc)
    fold_f1s.append(f1)
    agg_tn += tn; agg_fp += fp; agg_fn += fn; agg_tp += tp
    fold_ksecs.append(t1k - t0k)
    fold_tsecs.append(t1 - t0)

    print(f"  fold {fold_id}: AUC={auc:.4f}  Acc={acc:.4f}  F1={f1:.4f}  "
          f"TP={tp} FP={fp} TN={tn} FN={fn}  | kernel={fold_ksecs[-1]:.2f}s | total={fold_tsecs[-1]:.2f}s")

# Summary
mean_auc = float(np.mean(fold_aucs)); std_auc = float(np.std(fold_aucs))
mean_acc = float(np.mean(fold_accs)); std_acc = float(np.std(fold_accs))
mean_f1  = float(np.mean(fold_f1s));  std_f1  = float(np.std(fold_f1s))
mean_k   = float(np.mean(fold_ksecs)); mean_t = float(np.mean(fold_tsecs))

print("\n=== Summary ===")
print(f"Mean AUC={mean_auc:.4f} (+/- {std_auc:.4f})  "
      f"Mean Acc={mean_acc:.4f} (+/- {std_acc:.4f})  "
      f"Mean F1={mean_f1:.4f} (+/- {std_f1:.4f})")
print(f"Totals: TP={agg_tp}  FP={agg_fp}  TN={agg_tn}  FN={agg_fn}")
print(f"Timing: mean kernel/fold={mean_k:.2f}s  |  mean total/fold={mean_t:.2f}s")
print(f"[SAVED] CSVs in: {os.path.abspath(OUTDIR)}")


# ADAPTIVE WHEN NEEDED 

In [ ]:
# QSVM fidelity kernel, N=96, 3-fold CV
# Device-like simulator cloned from your real backend (Brisbane by default)
# Base shots=256 + adaptive top-up for ambiguous pairs; robust diag normalization
# Metrics: AUC, Acc, F1, TP/FP/TN/FN, Sensitivity, Specificity, Precision, Recall
# Qiskit pins: qiskit==2.2.1, qiskit-aer==0.17.2, qiskit-ibm-runtime==0.42.0

import os, time, math, numpy as np, pandas as pd
from collections import Counter

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, confusion_matrix, precision_score, recall_score

from qiskit import QuantumCircuit, ClassicalRegister, transpile
from qiskit.circuit.library import z_feature_map
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import QiskitRuntimeService

# ------------------------------- USER CONFIG -------------------------------
BACKEND_NAME    = "ibm_brisbane"          # change to "ibm_torino" if needed
INITIAL_LAYOUT  = [3, 4, 5, 6, 7, 8, 16]  # k=7 chain we’ll map onto
SUBSET_STRATEGY = "stratified"            # or "fps"

N               = 96
N_SPLITS        = 3
RANDOM_SEED     = 42

# Shots policy
BASE_SHOTS      = 256
USE_ADAPTIVE    = True
BAND            = (0.40, 0.60)            # only top-up when p0 in this band
SIGMA_TARGET    = 0.0125                  # target stddev for Bernoulli estimator
MAX_SHOTS       = 1025                    # per pair cap (including base shots)

# Throughput
RUN_BATCH       = 64                      # circuits per run()
KERNEL_CHUNK    = 512                     # pubs per outer chunk

# Output
OUTDIR          = "./kernels_noisy_aer_brisbane"
SAVE_PER_ENTRY_DETAILS = True
os.makedirs(OUTDIR, exist_ok=True)

# QSVM feature map settings
nq = 7
fm_z = z_feature_map(feature_dimension=nq, reps=1, entanglement="linear", parameter_prefix='z')
fm_x = z_feature_map(feature_dimension=nq, reps=1, entanglement="linear", parameter_prefix='x')

# Manual 7 features
FEATURES = [
    'mean texture',
    'mean concave points',
    'worst radius',
    'worst texture',
    'worst smoothness',
    'worst concave points',
    'worst symmetry',
]

# Grid for C (inner CV on precomputed kernel)
C_GRID = np.logspace(-2, 2, 9)  # 0.01 ... 100
# ---------------------------------------------------------------------------

# ------------------------------- DATA (N=96) -------------------------------
X_df, y_sr = load_breast_cancer(return_X_y=True, as_frame=True)
missing = [c for c in FEATURES if c not in X_df.columns]
if missing:
    raise ValueError(f"Missing features in dataset: {missing}")

X_all = X_df[FEATURES].to_numpy()
y_all = (y_sr.values == 1).astype(int)

# Choose exactly N=96 indices
if SUBSET_STRATEGY == "fps":
    rng = np.random.RandomState(RANDOM_SEED)
    pos, neg = np.where(y_all==1)[0], np.where(y_all==0)[0]
    i0, j0 = rng.choice(pos), rng.choice(neg)
    chosen = [int(i0), int(j0)]
    d2 = np.sum((X_all[:,None,:]-X_all[None,:,:])**2, axis=2)
    mind2 = np.minimum(d2[i0], d2[j0])
    while len(chosen) < N:
        nxt = int(np.argmax(mind2)); chosen.append(nxt)
        mind2 = np.minimum(mind2, d2[nxt])
    train_idx = np.array(chosen, dtype=int)
else:
    sss = StratifiedShuffleSplit(n_splits=1, train_size=N, random_state=RANDOM_SEED)
    (train_idx, _), = sss.split(X_all, y_all)

# We’ll fit scaler on ALL X_all (your requested variant)
scaler = StandardScaler().fit(X_all)
X_all_scaled = scaler.transform(X_all)

X = X_all_scaled[train_idx]
y = y_all[train_idx]

print(f"[INFO] Using N={N} samples. Class balance: {Counter(int(v) for v in y)}")
print(f"[INFO] Strategy={SUBSET_STRATEGY} | Adaptive={'ON' if USE_ADAPTIVE else 'OFF'} | "
      f"Base shots={BASE_SHOTS} | Band={BAND} | sigma={SIGMA_TARGET} | Max shots={MAX_SHOTS}")

# ----------------- LIVE CONNECTION BANNER (no hardware submission here) ----
def connection_banner(name: str):
    try:
        service = QiskitRuntimeService()
        backend = service.backend(name)
        st = backend.status()
        print(f"[CONNECTED] backend={backend.name} | operational={st.operational} | "
              f"pending_jobs={st.pending_jobs} | status='{st.status_msg}'")
        return service, backend
    except Exception as e:
        print(f"[WARN] Could not query backend status for '{name}': {e}")
        return None, None

service, backend = connection_banner(BACKEND_NAME)

# ------------------ ROBUST SNAPSHOT / CLONE WITH RETRIES -------------------
def get_simulator_with_snapshot(service, backend, retries=5, wait_s=3.0):
    """
    Try to clone a device-like simulator from 'backend'.
    If AerSimulator.from_backend(...) keeps failing, fall back to:
      - AerSimulator(noise_model=NoiseModel.from_backend(properties))
      - Plain AerSimulator (finite-shot only)
    Prints clear messages about what happened.
    """
    from qiskit_aer.noise import NoiseModel

    last_err = None
    props = None
    # Try to clone with retries
    for attempt in range(1, retries+1):
        try:
            sim = AerSimulator.from_backend(backend)
            sim.set_options(seed_simulator=RANDOM_SEED)
            # Try to fetch properties timestamp for the banner
            try:
                props = backend.properties()
                ts = getattr(props, "last_update_date", None)
                print(f"[SNAPSHOT] Cloned from {backend.name} on attempt {attempt}/{retries} "
                      f"(properties timestamp={ts})")
            except Exception as e_props:
                print(f"[SNAPSHOT] Cloned from {backend.name} (could not read properties timestamp: {e_props})")
            return sim, props
        except Exception as e:
            print(f"[SNAPSHOT-RETRY {attempt}/{retries}] clone failed: {e}")
            last_err = e
            time.sleep(wait_s)

    # Fallback path: build noise model from properties
    try:
        props = backend.properties()
    except Exception as e2:
        print(f"[SNAPSHOT] backend.properties() unavailable: {e2}")
        props = None

    try:
        if props is not None:
            nm = NoiseModel.from_backend(props)
            sim = AerSimulator(noise_model=nm)
            sim.set_options(seed_simulator=RANDOM_SEED)
            ts = getattr(props, "last_update_date", None)
            print(f"[SNAPSHOT] Using AerSimulator(noise_model=from_backend(properties)) "
                  f"(properties timestamp={ts})")
        else:
            sim = AerSimulator()
            sim.set_options(seed_simulator=RANDOM_SEED)
            print("[SNAPSHOT] Using plain AerSimulator (finite-shot only)")
        return sim, props
    except Exception as e3:
        print(f"[SNAPSHOT] Final fallback to plain AerSimulator due to error: {e3}")
        sim = AerSimulator()
        sim.set_options(seed_simulator=RANDOM_SEED)
        return sim, props

# Build simulator (with snapshot) — if service/backend missing, fall back to plain Aer
if service is not None and backend is not None:
    aer_sim, props = get_simulator_with_snapshot(service, backend, retries=5, wait_s=2.5)
else:
    print("[SNAPSHOT] No backend handle; using plain AerSimulator()")
    aer_sim, props = AerSimulator(), None
    aer_sim.set_options(seed_simulator=RANDOM_SEED)

# Optional: dump properties text snapshot for reproducibility
if props is not None:
    try:
        with open(os.path.join(OUTDIR, "backend_properties.txt"), "w", encoding="utf-8") as f:
            f.write(str(props))
        print(f"[SNAPSHOT] Saved backend properties to {os.path.abspath(os.path.join(OUTDIR,'backend_properties.txt'))}")
    except Exception as e:
        print(f"[SNAPSHOT] Could not save properties: {e}")

# Conservative threading (tune if you like)
try:
    import multiprocessing as mp
    aer_sim.set_options(
        max_parallel_experiments=1,
        max_parallel_shots=1,
        max_parallel_threads=min(4, mp.cpu_count()),
    )
except Exception:
    pass

# ------------------------------- TEMPLATE CIRCUIT --------------------------
qc = QuantumCircuit(nq, name="fid")
qc.compose(fm_z, qubits=range(nq), inplace=True)
qc.compose(fm_x.inverse(), qubits=range(nq), inplace=True)
cr = ClassicalRegister(nq, "m")
qc.add_register(cr)
qc.measure(range(nq), cr)

t0_build = time.perf_counter()
qc_t = transpile(
    qc,
    backend=aer_sim,                # IMPORTANT: only pass backend (the sim), not basis_gates/coupling_map
    initial_layout=INITIAL_LAYOUT,  # your chosen chain
    optimization_level=1,
)
t1_build = time.perf_counter()
print(f"[INFO] Prepared circuit (transpile). Time: {t1_build - t0_build:.2f}s")

def extract_params_by_prefix(circ, prefix: str):
    plist = [p for p in circ.parameters if p.name.startswith(f"{prefix}[")]
    plist.sort(key=lambda p: int(p.name[p.name.find('[')+1:-1]))
    return plist
params_x = extract_params_by_prefix(qc_t, "x")
params_z = extract_params_by_prefix(qc_t, "z")

# ------------------------------- HELPERS -----------------------------------
STATS = {"adapted_pairs": 0, "total_pairs": 0, "max_shots_seen": BASE_SHOTS}

def counts_to_p0(counts_dict):
    shots_used = sum(counts_dict.values())
    return counts_dict.get("0"*nq, 0) / max(1, shots_used), shots_used

def pub_for_pair(Arow, Brow):
    pmap = {}
    for k in range(nq):
        pmap[params_z[k]] = float(Brow[k])
        pmap[params_x[k]] = float(Arow[k])
    return (qc_t, pmap)

def run_counts_pubs(pubs, shots, batch=RUN_BATCH, seed=1234):
    out = []
    for start in range(0, len(pubs), batch):
        sub = pubs[start:start+batch]
        bound = [circ.assign_parameters(pmap, inplace=False) for (circ, pmap) in sub]
        job = aer_sim.run(bound, shots=shots, seed_simulator=seed)
        res = job.result()
        counts = res.get_counts()
        if isinstance(counts, dict): out.append(counts)
        else: out.extend(counts)
    return out

def needed_shots_for_sigma(p_hat, sigma):
    p = max(1e-6, min(1-1e-6, p_hat))
    return int(math.ceil(p*(1-p)/(sigma*sigma)))

def p0s_with_adaptive(pubs, base_shots=BASE_SHOTS, band=BAND, sigma=SIGMA_TARGET, max_shots=MAX_SHOTS, seed=1234):
    counts1 = run_counts_pubs(pubs, shots=base_shots, seed=seed)
    p0 = np.zeros(len(pubs), dtype=float)
    shots_tot = np.zeros(len(pubs), dtype=int)
    for i, c in enumerate(counts1):
        p, s = counts_to_p0(c); p0[i] = p; shots_tot[i] = s

    lo, hi = band
    need = []
    for i, p in enumerate(p0):
        if lo <= p <= hi:
            req = needed_shots_for_sigma(p, sigma)
            add = max(0, min(max_shots - shots_tot[i], req - shots_tot[i]))
            if add > 0: need.append((i, add))

    if need:
        from collections import defaultdict
        groups = defaultdict(list)
        for idx, add in need: groups[add].append(idx)
        for add, idxs in groups.items():
            pubs_batch = [pubs[i] for i in idxs]
            counts2 = run_counts_pubs(pubs_batch, shots=add, seed=seed+1)
            for j, c2 in enumerate(counts2):
                i = idxs[j]
                p_add, s_add = counts_to_p0(c2)
                s1 = shots_tot[i]; p1 = p0[i]
                s_sum = s1 + s_add
                p_comb = (p1*s1 + p_add*s_add) / max(1, s_sum)
                p0[i] = p_comb; shots_tot[i] = s_sum

    STATS["total_pairs"] += len(pubs)
    STATS["adapted_pairs"] += int(np.sum(shots_tot > base_shots))
    STATS["max_shots_seen"] = max(STATS["max_shots_seen"], int(shots_tot.max()))
    return [(float(p0[i]), int(shots_tot[i])) for i in range(len(pubs))]

def p0s(pubs):
    return p0s_with_adaptive(pubs) if USE_ADAPTIVE else [(counts_to_p0(c))[0] for c in run_counts_pubs(pubs, shots=BASE_SHOTS)]

def kernel_square_upper(A, chunk=KERNEL_CHUNK, details=None):
    n = A.shape[0]
    K_raw = np.zeros((n, n), dtype=float)
    idx_pairs, pubs = [], []
    for i in range(n):
        for j in range(i, n):
            idx_pairs.append((i, j))
            pubs.append(pub_for_pair(A[i], A[j]))
    for start in range(0, len(pubs), chunk):
        sub = pubs[start:start+chunk]
        vals = p0s(sub)
        for (p0_val, s), (i, j) in zip(vals, idx_pairs[start:start+chunk]):
            K_raw[i, j] = p0_val;  K_raw[j, i] = p0_val
            if details is not None:
                details.append({"block":"train", "i":int(i), "j":int(j),
                                "p0":float(p0_val), "shots_total":int(s)})
    return K_raw

def kernel_cross(A, B, chunk=KERNEL_CHUNK, details=None):
    K_raw = np.zeros((A.shape[0], B.shape[0]), dtype=float)
    pubs = [pub_for_pair(A[i], B[j]) for i in range(A.shape[0]) for j in range(B.shape[0])]
    idx = 0
    for start in range(0, len(pubs), chunk):
        sub = pubs[start:start+chunk]
        vals = p0s(sub)
        for (p0_val, s) in vals:
            i = idx // B.shape[0]; j = idx % B.shape[0]
            K_raw[i, j] = p0_val
            if details is not None:
                details.append({"block":"testxtrain", "i":int(i), "j":int(j),
                                "p0":float(p0_val), "shots_total":int(s)})
            idx += 1
    return K_raw

def kernel_diag(A, chunk=KERNEL_CHUNK, details=None):
    pubs = [pub_for_pair(A[i], A[i]) for i in range(A.shape[0])]
    d = np.zeros(A.shape[0], dtype=float)
    for start in range(0, len(pubs), chunk):
        sub = pubs[start:start+chunk]
        vals = p0s(sub)
        for offs, (p0_val, s) in enumerate(vals):
            d[start + offs] = p0_val
            if details is not None:
                details.append({"block":"diag", "i":int(start+offs), "j":int(start+offs),
                                "p0":float(p0_val), "shots_total":int(s)})
    return d

def normalize_square_from_diag(K_raw, d):
    d = np.clip(d, 1e-12, None)
    Kn = K_raw / np.sqrt(np.outer(d, d))
    Kn = 0.5 * (Kn + Kn.T)
    np.fill_diagonal(Kn, 1.0)
    return Kn

def normalize_cross_from_diag(K_raw, d_test, d_train):
    d_test  = np.clip(d_test,  1e-12, None)
    d_train = np.clip(d_train, 1e-12, None)
    return K_raw / np.sqrt(np.outer(d_test, d_train))

def pick_best_C(K_tr, y_tr, Cs=C_GRID, seed=RANDOM_SEED):
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=seed)
    best_auc, best_C = -np.inf, Cs[0]
    n = len(y_tr)
    for C in Cs:
        aucs = []
        for tr2, va2 in skf.split(np.arange(n), y_tr):
            K_tt = K_tr[np.ix_(tr2, tr2)]
            K_tv = K_tr[np.ix_(va2, tr2)]
            clf2 = SVC(kernel="precomputed", C=C)
            clf2.fit(K_tt, y_tr[tr2])
            scores2 = clf2.decision_function(K_tv)
            aucs.append(roc_auc_score(y_tr[va2], scores2))
        m = float(np.mean(aucs))
        if m > best_auc: best_auc, best_C = m, C
    return best_C, best_auc

# ------------------------------- 3-FOLD CV ---------------------------------
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_SEED)
print(f"[INFO] AER run | Adaptive={'ON' if USE_ADAPTIVE else 'OFF'} | Base shots={BASE_SHOTS} | "
      f"RUN_BATCH={RUN_BATCH} | KERNEL_CHUNK={KERNEL_CHUNK}")

fold_aucs, fold_accs, fold_f1s = [], [], []
fold_sens, fold_spec, fold_prec, fold_rec = [], [], [], []
agg_tn = agg_fp = agg_fn = agg_tp = 0
fold_ksecs, fold_tsecs = [], []

for fold_id, (tr_idx, te_idx) in enumerate(cv.split(X, y), 1):
    X_tr, X_te = X[tr_idx], X[te_idx]
    y_tr, y_te = y[tr_idx], y[te_idx]

    DETAILS = [] if SAVE_PER_ENTRY_DETAILS else None
    t0 = time.perf_counter()

    # Train square -> diag -> normalize
    t0k = time.perf_counter()
    K_tr_raw = kernel_square_upper(X_tr, details=DETAILS)
    d_tr     = np.diag(K_tr_raw).copy()
    K_tr     = normalize_square_from_diag(K_tr_raw, d_tr)

    # Test diag + cross -> normalize
    d_te     = kernel_diag(X_te, details=DETAILS)
    K_te_raw = kernel_cross(X_te, X_tr, details=DETAILS)
    K_te     = normalize_cross_from_diag(K_te_raw, d_te, d_tr)
    t1k = time.perf_counter()

    # Save kernels & (optional) details
    pd.DataFrame(K_tr).to_csv(os.path.join(OUTDIR, f"K_train_shots{BASE_SHOTS}_fold{fold_id}.csv"), index=False)
    pd.DataFrame(K_te).to_csv(os.path.join(OUTDIR, f"K_test_shots{BASE_SHOTS}_fold{fold_id}.csv"), index=False)
    if SAVE_PER_ENTRY_DETAILS and DETAILS:
        pd.DataFrame(DETAILS).to_csv(os.path.join(OUTDIR, f"K_entries_shots{BASE_SHOTS}_fold{fold_id}.csv"), index=False)

    # Inner CV over C on precomputed kernel
    C_best, _ = pick_best_C(K_tr, y_tr)

    # QSVM (precomputed) with the best C
    clf = SVC(kernel="precomputed", C=C_best)
    clf.fit(K_tr, y_tr)
    scores = clf.decision_function(K_te)
    y_pred = (scores >= 0).astype(int)

    # Metrics
    auc = roc_auc_score(y_te, scores)
    acc = accuracy_score(y_te, y_pred)
    f1  = f1_score(y_te, y_pred, average="binary", pos_label=1)
    tn, fp, fn, tp = confusion_matrix(y_te, y_pred, labels=[0,1]).ravel()
    sens = tp / max(1, tp+fn)             # sensitivity / recall / TPR
    spec = tn / max(1, tn+fp)             # specificity / TNR
    prec = tp / max(1, tp+fp)             # precision
    rec  = sens

    t1 = time.perf_counter()

    fold_aucs.append(auc); fold_accs.append(acc); fold_f1s.append(f1)
    fold_sens.append(sens); fold_spec.append(spec); fold_prec.append(prec); fold_rec.append(rec)
    agg_tn += tn; agg_fp += fp; agg_fn += fn; agg_tp += tp
    fold_ksecs.append(t1k - t0k); fold_tsecs.append(t1 - t0)

    print(f"  fold {fold_id}: AUC={auc:.4f}  Acc={acc:.4f}  F1={f1:.4f}  "
          f"TP={tp} FP={fp} TN={tn} FN={fn}  "
          f"Sens={sens:.4f} Spec={spec:.4f} Prec={prec:.4f} Rec={rec:.4f}  "
          f"| C*={C_best:g} | kernel={fold_ksecs[-1]:.2f}s | total={fold_tsecs[-1]:.2f}s")

# Summary
mean_auc = float(np.mean(fold_aucs)); std_auc = float(np.std(fold_aucs))
mean_acc = float(np.mean(fold_accs)); std_acc = float(np.std(fold_accs))
mean_f1  = float(np.mean(fold_f1s));  std_f1  = float(np.std(fold_f1s))
mean_sens = float(np.mean(fold_sens)); std_sens = float(np.std(fold_sens))
mean_spec = float(np.mean(fold_spec)); std_spec = float(np.std(fold_spec))
mean_prec = float(np.mean(fold_prec)); std_prec = float(np.std(fold_prec))
mean_rec  = float(np.mean(fold_rec));  std_rec  = float(np.std(fold_rec))
mean_k   = float(np.mean(fold_ksecs)); mean_t = float(np.mean(fold_tsecs))

print("\n=== Summary ===")
print(f"Mean AUC={mean_auc:.4f} (+/- {std_auc:.4f})  "
      f"Mean Acc={mean_acc:.4f} (+/- {std_acc:.4f})  "
      f"Mean F1={mean_f1:.4f} (+/- {std_f1:.4f})")
print(f"Mean Sens={mean_sens:.4f} (+/- {std_sens:.4f})  "
      f"Mean Spec={mean_spec:.4f} (+/- {std_spec:.4f})  "
      f"Mean Prec={mean_prec:.4f} (+/- {std_prec:.4f})  "
      f"Mean Rec={mean_rec:.4f} (+/- {std_rec:.4f})")
print(f"Totals: TP={agg_tp}  FP={agg_fp}  TN={agg_tn}  FN={agg_fn}")
print(f"Timing: mean kernel/fold={mean_k:.2f}s  |  mean total/fold={mean_t:.2f}s")

# Adaptive shots stats
print(f"\n[ADAPTIVE] pairs={STATS['total_pairs']}  adapted={STATS['adapted_pairs']}  "
      f"({100.0*STATS['adapted_pairs']/max(1,STATS['total_pairs']):.1f}%)  "
      f"max_shots_seen={STATS['max_shots_seen']}")
print(f"[SAVED] CSVs in: {os.path.abspath(OUTDIR)}")


In [ ]:
# QSVM_fidelity_kernel_cv.py
# Qiskit pins: qiskit==2.2.1, qiskit-aer==0.17.2, qiskit-ibm-runtime==0.42.0

import os, time, math, numpy as np, pandas as pd
from collections import Counter

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import (
    roc_auc_score, accuracy_score, f1_score,
    confusion_matrix, precision_score, recall_score
)

from qiskit import QuantumCircuit, ClassicalRegister, transpile
# FIXED: use the z_feature_map() function (ZFeatureMap class is deprecated)
try:
    from qiskit.circuit.library import z_feature_map
except ImportError:
    # Fallback for some distributions
    from qiskit.circuit.library.data_preparation import z_feature_map

from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import QiskitRuntimeService

# ------------------------------- USER CONFIG -------------------------------
BACKEND_NAME    = "ibm_brisbane"          # change to "ibm_torino" if needed
INITIAL_LAYOUT  = [3, 4, 5, 6, 7, 8, 16]  # k=7 chain we’ll map onto
SUBSET_STRATEGY = "stratified"            # or "fps"

N               = 96
N_SPLITS        = 3
RANDOM_SEED     = 42

# Shots policy
BASE_SHOTS      = 256
USE_ADAPTIVE    = True
BAND            = (0.40, 0.60)            # only top-up when p0 in this band
SIGMA_TARGET    = 0.0125                  # target stddev for Bernoulli estimator
MAX_SHOTS       = 1025                    # per pair cap (including base shots)

# Throughput
RUN_BATCH       = 64                      # circuits per run()
KERNEL_CHUNK    = 512                     # pubs per outer chunk

# Output
OUTDIR          = "./kernels_noisy_aer_brisbane"
SAVE_PER_ENTRY_DETAILS = True
os.makedirs(OUTDIR, exist_ok=True)

# QSVM feature map settings
nq = 7
# NOTE: ZFeatureMap class is deprecated in Qiskit 2.x; use the function below.
# 'entanglement' arg is not used for Z feature maps, so it's removed.
fm_z = z_feature_map(feature_dimension=nq, reps=1, parameter_prefix='z')
fm_x = z_feature_map(feature_dimension=nq, reps=1, parameter_prefix='x')

# Manual 7 features
FEATURES = [
    'mean texture',
    'mean concave points',
    'worst radius',
    'worst texture',
    'worst smoothness',
    'worst concave points',
    'worst symmetry',
]

# Grid for C (inner CV on precomputed kernel)
C_GRID = np.logspace(-2, 2, 9)  # 0.01 ... 100
# ---------------------------------------------------------------------------

# ------------------------------- DATA (N=96) -------------------------------
X_df, y_sr = load_breast_cancer(return_X_y=True, as_frame=True)
missing = [c for c in FEATURES if c not in X_df.columns]
if missing:
    raise ValueError(f"Missing features in dataset: {missing}")

X_all = X_df[FEATURES].to_numpy()
y_all = (y_sr.values == 1).astype(int)

# Choose exactly N=96 indices
if SUBSET_STRATEGY == "fps":
    rng = np.random.RandomState(RANDOM_SEED)
    pos, neg = np.where(y_all==1)[0], np.where(y_all==0)[0]
    i0, j0 = rng.choice(pos), rng.choice(neg)
    chosen = [int(i0), int(j0)]
    d2 = np.sum((X_all[:,None,:]-X_all[None,:,:])**2, axis=2)
    mind2 = np.minimum(d2[i0], d2[j0])
    while len(chosen) < N:
        nxt = int(np.argmax(mind2)); chosen.append(nxt)
        mind2 = np.minimum(mind2, d2[nxt])
    train_idx = np.array(chosen, dtype=int)
else:
    sss = StratifiedShuffleSplit(n_splits=1, train_size=N, random_state=RANDOM_SEED)
    (train_idx, _), = sss.split(X_all, y_all)

# We’ll fit scaler on ALL X_all (your requested variant)
scaler = StandardScaler().fit(X_all)
X_all_scaled = scaler.transform(X_all)

X = X_all_scaled[train_idx]
y = y_all[train_idx]

print(f"[INFO] Using N={N} samples. Class balance: {Counter(int(v) for v in y)}")
print(f"[INFO] Strategy={SUBSET_STRATEGY} | Adaptive={'ON' if USE_ADAPTIVE else 'OFF'} | "
      f"Base shots={BASE_SHOTS} | Band={BAND} | sigma={SIGMA_TARGET} | Max shots={MAX_SHOTS}")

# ----------------- LIVE CONNECTION BANNER (no hardware submission here) ----
def connection_banner(name: str):
    try:
        service = QiskitRuntimeService()
        backend = service.backend(name)
        st = backend.status()
        print(f"[CONNECTED] backend={backend.name} | operational={st.operational} | "
              f"pending_jobs={st.pending_jobs} | status='{st.status_msg}'")
        return service, backend
    except Exception as e:
        print(f"[WARN] Could not query backend status for '{name}': {e}")
        return None, None

service, backend = connection_banner(BACKEND_NAME)

# ------------------ ROBUST SNAPSHOT / CLONE WITH RETRIES -------------------
def get_simulator_with_snapshot(service, backend, retries=5, wait_s=3.0):
    """
    Try to clone a device-like simulator from 'backend'.
    If AerSimulator.from_backend(...) keeps failing, fall back to:
      - AerSimulator(noise_model=NoiseModel.from_backend(properties))
      - Plain AerSimulator (finite-shot only)
    Prints clear messages about what happened.
    """
    from qiskit_aer.noise import NoiseModel

    last_err = None
    props = None
    # Try to clone with retries
    for attempt in range(1, retries+1):
        try:
            sim = AerSimulator.from_backend(backend)
            sim.set_options(seed_simulator=RANDOM_SEED)
            # Try to fetch properties timestamp for the banner
            try:
                props = backend.properties()
                ts = getattr(props, "last_update_date", None)
                print(f"[SNAPSHOT] Cloned from {backend.name} on attempt {attempt}/{retries} "
                      f"(properties timestamp={ts})")
            except Exception as e_props:
                print(f"[SNAPSHOT] Cloned from {backend.name} (could not read properties timestamp: {e_props})")
            return sim, props
        except Exception as e:
            print(f"[SNAPSHOT-RETRY {attempt}/{retries}] clone failed: {e}")
            last_err = e
            time.sleep(wait_s)

    # Fallback path: build noise model from properties
    try:
        props = backend.properties()
    except Exception as e2:
        print(f"[SNAPSHOT] backend.properties() unavailable: {e2}")
        props = None

    try:
        if props is not None:
            nm = NoiseModel.from_backend(props)
            sim = AerSimulator(noise_model=nm)
            sim.set_options(seed_simulator=RANDOM_SEED)
            ts = getattr(props, "last_update_date", None)
            print(f"[SNAPSHOT] Using AerSimulator(noise_model=from_backend(properties)) "
                  f"(properties timestamp={ts})")
        else:
            sim = AerSimulator()
            sim.set_options(seed_simulator=RANDOM_SEED)
            print("[SNAPSHOT] Using plain AerSimulator (finite-shot only)")
        return sim, props
    except Exception as e3:
        print(f"[SNAPSHOT] Final fallback to plain AerSimulator due to error: {e3}")
        sim = AerSimulator()
        sim.set_options(seed_simulator=RANDOM_SEED)
        return sim, props

# Build simulator (with snapshot) — if service/backend missing, fall back to plain Aer
if service is not None and backend is not None:
    aer_sim, props = get_simulator_with_snapshot(service, backend, retries=5, wait_s=2.5)
else:
    print("[SNAPSHOT] No backend handle; using plain AerSimulator()")
    aer_sim, props = AerSimulator(), None
    aer_sim.set_options(seed_simulator=RANDOM_SEED)

# Optional: dump properties text snapshot for reproducibility
if props is not None:
    try:
        with open(os.path.join(OUTDIR, "backend_properties.txt"), "w", encoding="utf-8") as f:
            f.write(str(props))
        print(f"[SNAPSHOT] Saved backend properties to {os.path.abspath(os.path.join(OUTDIR,'backend_properties.txt'))}")
    except Exception as e:
        print(f"[SNAPSHOT] Could not save properties: {e}")

# Conservative threading (tune if you like)
try:
    import multiprocessing as mp
    aer_sim.set_options(
        max_parallel_experiments=1,
        max_parallel_shots=1,
        max_parallel_threads=min(4, mp.cpu_count()),
    )
except Exception:
    pass

# ------------------------------- TEMPLATE CIRCUIT --------------------------
qc = QuantumCircuit(nq, name="fid")
qc.compose(fm_z, qubits=range(nq), inplace=True)
qc.compose(fm_x.inverse(), qubits=range(nq), inplace=True)
cr = ClassicalRegister(nq, "m")
qc.add_register(cr)
qc.measure(range(nq), cr)

t0_build = time.perf_counter()
qc_t = transpile(
    qc,
    backend=aer_sim,                # IMPORTANT: only pass backend (the sim), not basis_gates/coupling_map
    initial_layout=INITIAL_LAYOUT,  # your chosen chain
    optimization_level=1,
)
t1_build = time.perf_counter()
print(f"[INFO] Prepared circuit (transpile). Time: {t1_build - t0_build:.2f}s")

def extract_params_by_prefix(circ, prefix: str):
    plist = [p for p in circ.parameters if p.name.startswith(f"{prefix}[")]
    plist.sort(key=lambda p: int(p.name[p.name.find('[')+1:-1]))
    return plist
params_x = extract_params_by_prefix(qc_t, "x")
params_z = extract_params_by_prefix(qc_t, "z")

# ------------------------------- HELPERS -----------------------------------
STATS = {"adapted_pairs": 0, "total_pairs": 0, "max_shots_seen": BASE_SHOTS}

def counts_to_p0(counts_dict):
    shots_used = sum(counts_dict.values())
    return counts_dict.get("0"*nq, 0) / max(1, shots_used), shots_used

def pub_for_pair(Arow, Brow):
    pmap = {}
    for k in range(nq):
        pmap[params_z[k]] = float(Brow[k])
        pmap[params_x[k]] = float(Arow[k])
    return (qc_t, pmap)

def run_counts_pubs(pubs, shots, batch=RUN_BATCH, seed=1234):
    out = []
    for start in range(0, len(pubs), batch):
        sub = pubs[start:start+batch]
        bound = [circ.assign_parameters(pmap, inplace=False) for (circ, pmap) in sub]
        job = aer_sim.run(bound, shots=shots, seed_simulator=seed)
        res = job.result()
        counts = res.get_counts()
        if isinstance(counts, dict): out.append(counts)
        else: out.extend(counts)
    return out

def needed_shots_for_sigma(p_hat, sigma):
    p = max(1e-6, min(1-1e-6, p_hat))
    return int(math.ceil(p*(1-p)/(sigma*sigma)))

def p0s_with_adaptive(pubs, base_shots=BASE_SHOTS, band=BAND, sigma=SIGMA_TARGET, max_shots=MAX_SHOTS, seed=1234):
    counts1 = run_counts_pubs(pubs, shots=base_shots, seed=seed)
    p0 = np.zeros(len(pubs), dtype=float)
    shots_tot = np.zeros(len(pubs), dtype=int)
    for i, c in enumerate(counts1):
        p, s = counts_to_p0(c); p0[i] = p; shots_tot[i] = s

    lo, hi = band
    need = []
    for i, p in enumerate(p0):
        if lo <= p <= hi:
            req = needed_shots_for_sigma(p, sigma)
            add = max(0, min(max_shots - shots_tot[i], req - shots_tot[i]))
            if add > 0: need.append((i, add))

    if need:
        from collections import defaultdict
        groups = defaultdict(list)
        for idx, add in need: groups[add].append(idx)
        for add, idxs in groups.items():
            pubs_batch = [pubs[i] for i in idxs]
            counts2 = run_counts_pubs(pubs_batch, shots=add, seed=seed+1)
            for j, c2 in enumerate(counts2):
                i = idxs[j]
                p_add, s_add = counts_to_p0(c2)
                s1 = shots_tot[i]; p1 = p0[i]
                s_sum = s1 + s_add
                p_comb = (p1*s1 + p_add*s_add) / max(1, s_sum)
                p0[i] = p_comb; shots_tot[i] = s_sum

    STATS["total_pairs"] += len(pubs)
    STATS["adapted_pairs"] += int(np.sum(shots_tot > base_shots))
    STATS["max_shots_seen"] = max(STATS["max_shots_seen"], int(shots_tot.max()))
    return [(float(p0[i]), int(shots_tot[i])) for i in range(len(pubs))]

def p0s(pubs):
    return p0s_with_adaptive(pubs) if USE_ADAPTIVE else [(counts_to_p0(c))[0] for c in run_counts_pubs(pubs, shots=BASE_SHOTS)]

def kernel_square_upper(A, chunk=KERNEL_CHUNK, details=None):
    n = A.shape[0]
    K_raw = np.zeros((n, n), dtype=float)
    idx_pairs, pubs = [], []
    for i in range(n):
        for j in range(i, n):
            idx_pairs.append((i, j))
            pubs.append(pub_for_pair(A[i], A[j]))
    for start in range(0, len(pubs), chunk):
        sub = pubs[start:start+chunk]
        vals = p0s(sub)
        for (p0_val, s), (i, j) in zip(vals, idx_pairs[start:start+chunk]):
            K_raw[i, j] = p0_val;  K_raw[j, i] = p0_val
            if details is not None:
                details.append({"block":"train", "i":int(i), "j":int(j),
                                "p0":float(p0_val), "shots_total":int(s)})
    return K_raw

def kernel_cross(A, B, chunk=KERNEL_CHUNK, details=None):
    K_raw = np.zeros((A.shape[0], B.shape[0]), dtype=float)
    pubs = [pub_for_pair(A[i], B[j]) for i in range(A.shape[0]) for j in range(B.shape[0])]
    idx = 0
    for start in range(0, len(pubs), chunk):
        sub = pubs[start:start+chunk]
        vals = p0s(sub)
        for (p0_val, s) in vals:
            i = idx // B.shape[0]; j = idx % B.shape[0]
            K_raw[i, j] = p0_val
            if details is not None:
                details.append({"block":"testxtrain", "i":int(i), "j":int(j),
                                "p0":float(p0_val), "shots_total":int(s)})
            idx += 1
    return K_raw

def kernel_diag(A, chunk=KERNEL_CHUNK, details=None):
    pubs = [pub_for_pair(A[i], A[i]) for i in range(A.shape[0])]
    d = np.zeros(A.shape[0], dtype=float)
    for start in range(0, len(pubs), chunk):
        sub = pubs[start:start+chunk]
        vals = p0s(sub)
        for offs, (p0_val, s) in enumerate(vals):
            d[start + offs] = p0_val
            if details is not None:
                details.append({"block":"diag", "i":int(start+offs), "j":int(start+offs),
                                "p0":float(p0_val), "shots_total":int(s)})
    return d

def normalize_square_from_diag(K_raw, d):
    d = np.clip(d, 1e-12, None)
    Kn = K_raw / np.sqrt(np.outer(d, d))
    Kn = 0.5 * (Kn + Kn.T)
    np.fill_diagonal(Kn, 1.0)
    return Kn

def normalize_cross_from_diag(K_raw, d_test, d_train):
    d_test  = np.clip(d_test,  1e-12, None)
    d_train = np.clip(d_train, 1e-12, None)
    return K_raw / np.sqrt(np.outer(d_test, d_train))

def pick_best_C(K_tr, y_tr, Cs=C_GRID, seed=RANDOM_SEED):
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=seed)
    best_auc, best_C = -np.inf, Cs[0]
    n = len(y_tr)
    for C in Cs:
        aucs = []
        for tr2, va2 in skf.split(np.arange(n), y_tr):
            K_tt = K_tr[np.ix_(tr2, tr2)]
            K_tv = K_tr[np.ix_(va2, tr2)]
            clf2 = SVC(kernel="precomputed", C=C)
            clf2.fit(K_tt, y_tr[tr2])
            scores2 = clf2.decision_function(K_tv)
            aucs.append(roc_auc_score(y_tr[va2], scores2))
        m = float(np.mean(aucs))
        if m > best_auc: best_auc, best_C = m, C
    return best_C, best_auc

# ------------------------------- 3-FOLD CV ---------------------------------
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_SEED)
print(f"[INFO] AER run | Adaptive={'ON' if USE_ADAPTIVE else 'OFF'} | Base shots={BASE_SHOTS} | "
      f"RUN_BATCH={RUN_BATCH} | KERNEL_CHUNK={KERNEL_CHUNK}")

fold_aucs, fold_accs, fold_f1s = [], [], []
fold_sens, fold_spec, fold_prec, fold_rec = [], [], [], []
agg_tn = agg_fp = agg_fn = agg_tp = 0
fold_ksecs, fold_tsecs = [], []

for fold_id, (tr_idx, te_idx) in enumerate(cv.split(X, y), 1):
    X_tr, X_te = X[tr_idx], X[te_idx]
    y_tr, y_te = y[tr_idx], y[te_idx]

    DETAILS = [] if SAVE_PER_ENTRY_DETAILS else None
    t0 = time.perf_counter()

    # Train square -> diag -> normalize
    t0k = time.perf_counter()
    K_tr_raw = kernel_square_upper(X_tr, details=DETAILS)
    d_tr     = np.diag(K_tr_raw).copy()
    K_tr     = normalize_square_from_diag(K_tr_raw, d_tr)

    # Test diag + cross -> normalize
    d_te     = kernel_diag(X_te, details=DETAILS)
    K_te_raw = kernel_cross(X_te, X_tr, details=DETAILS)
    K_te     = normalize_cross_from_diag(K_te_raw, d_te, d_tr)
    t1k = time.perf_counter()

    # Save kernels & (optional) details
    pd.DataFrame(K_tr).to_csv(os.path.join(OUTDIR, f"K_train_shots{BASE_SHOTS}_fold{fold_id}.csv"), index=False)
    pd.DataFrame(K_te).to_csv(os.path.join(OUTDIR, f"K_test_shots{BASE_SHOTS}_fold{fold_id}.csv"), index=False)
    if SAVE_PER_ENTRY_DETAILS and DETAILS:
        pd.DataFrame(DETAILS).to_csv(os.path.join(OUTDIR, f"K_entries_shots{BASE_SHOTS}_fold{fold_id}.csv"), index=False)

    # Inner CV over C on precomputed kernel
    C_best, _ = pick_best_C(K_tr, y_tr)

    # QSVM (precomputed) with the best C
    clf = SVC(kernel="precomputed", C=C_best)
    clf.fit(K_tr, y_tr)
    scores = clf.decision_function(K_te)
    y_pred = (scores >= 0).astype(int)

    # Metrics
    auc = roc_auc_score(y_te, scores)
    acc = accuracy_score(y_te, y_pred)
    f1  = f1_score(y_te, y_pred, average="binary", pos_label=1)
    tn, fp, fn, tp = confusion_matrix(y_te, y_pred, labels=[0,1]).ravel()
    sens = tp / max(1, tp+fn)             # sensitivity / recall / TPR
    spec = tn / max(1, tn+fp)             # specificity / TNR
    prec = tp / max(1, tp+fp)             # precision
    rec  = sens

    t1 = time.perf_counter()

    fold_aucs.append(auc); fold_accs.append(acc); fold_f1s.append(f1)
    fold_sens.append(sens); fold_spec.append(spec); fold_prec.append(prec); fold_rec.append(rec)
    agg_tn += tn; agg_fp += fp; agg_fn += fn; agg_tp += tp
    fold_ksecs.append(t1k - t0k); fold_tsecs.append(t1 - t0)

    print(f"  fold {fold_id}: AUC={auc:.4f}  Acc={acc:.4f}  F1={f1:.4f}  "
          f"TP={tp} FP={fp} TN={tn} FN={fn}  "
          f"Sens={sens:.4f} Spec={spec:.4f} Prec={prec:.4f} Rec={rec:.4f}  "
          f"| C*={C_best:g} | kernel={t1k - t0k:.2f}s | total={t1 - t0:.2f}s")

# Summary
mean_auc = float(np.mean(fold_aucs)); std_auc = float(np.std(fold_aucs))
mean_acc = float(np.mean(fold_accs)); std_acc = float(np.std(fold_accs))
mean_f1  = float(np.mean(fold_f1s));  std_f1  = float(np.std(fold_f1s))
mean_sens = float(np.mean(fold_sens)); std_sens = float(np.std(fold_sens))
mean_spec = float(np.mean(fold_spec)); std_spec = float(np.std(fold_spec))
mean_prec = float(np.mean(fold_prec)); std_prec = float(np.std(fold_prec))
mean_rec  = float(np.mean(fold_rec));  std_rec  = float(np.std(fold_rec))
mean_k   = float(np.mean(fold_ksecs)); mean_t = float(np.mean(fold_tsecs))

print("\n=== Summary ===")
print(f"Mean AUC={mean_auc:.4f} (+/- {std_auc:.4f})  "
      f"Mean Acc={mean_acc:.4f} (+/- {std_acc:.4f})  "
      f"Mean F1={mean_f1:.4f} (+/- {std_f1:.4f})")
print(f"Mean Sens={mean_sens:.4f} (+/- {std_sens:.4f})  "
      f"Mean Spec={mean_spec:.4f} (+/- {std_spec:.4f})  "
      f"Mean Prec={mean_prec:.4f} (+/- {std_prec:.4f})  "
      f"Mean Rec={mean_rec:.4f} (+/- {std_rec:.4f})")
print(f"Totals: TP={agg_tp}  FP={agg_fp}  TN={agg_tn}  FN={agg_fn}")
print(f"Timing: mean kernel/fold={mean_k:.2f}s  |  mean total/fold={mean_t:.2f}s")

# Adaptive shots stats
print(f"\n[ADAPTIVE] pairs={STATS['total_pairs']}  adapted={STATS['adapted_pairs']}  "
      f"({100.0*STATS['adapted_pairs']/max(1,STATS['total_pairs']):.1f}%)  "
      f"max_shots_seen={STATS['max_shots_seen']}")
print(f"[SAVED] CSVs in: {os.path.abspath(OUTDIR)}")


kernal centring 

In [ ]:
# === QSVM fidelity kernel, N=96, 3-fold CV, comparing MANUAL-7 vs RFE-7 ===
# - Device-like simulator cloned from your real backend (Brisbane) with retry
# - Base shots=256 + adaptive top-up (to 1025) for ambiguous pairs
# - Kernel *centering* + SVC(class_weight="balanced") + inner CV over C
# - Prints full metrics per fold and summary for each feature set
#
# Pins tested with: qiskit==2.2.1, qiskit-aer==0.17.2, qiskit-ibm-runtime==0.42.0

import os, time, math, numpy as np, pandas as pd
from collections import Counter

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import RFE
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, confusion_matrix

from qiskit import QuantumCircuit, ClassicalRegister, transpile
from qiskit.circuit.library import z_feature_map
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel
from qiskit_ibm_runtime import QiskitRuntimeService

# ------------------------------- USER CONFIG -------------------------------
BACKEND_NAME    = "ibm_brisbane"          # change to "ibm_torino" if needed
INITIAL_LAYOUT  = [3, 4, 5, 6, 7, 8, 16]  # qubit chain for k=7
N               = 96
N_SPLITS        = 3
RANDOM_SEED     = 42

# Shots & adaptive policy
BASE_SHOTS      = 256
USE_ADAPTIVE    = True
BAND            = (0.40, 0.60)     # top-up only if p0 in this band
SIGMA_TARGET    = 0.0125
MAX_SHOTS       = 1025

# Throughput
RUN_BATCH       = 64               # circuits per run()
KERNEL_CHUNK    = 512              # pubs per outer chunk

# Output
OUTDIR          = "./kernels_noisy_aer_brisbane"
SAVE_PER_ENTRY_DETAILS = False     # set True to dump per-pair logs
os.makedirs(OUTDIR, exist_ok=True)

# Manual 7 features (your list)
MANUAL_FEATURES = [
    'mean texture',
    'mean concave points',
    'worst radius',
    'worst texture',
    'worst smoothness',
    'worst concave points',
    'worst symmetry',
]

# Grid for C (inner CV on precomputed kernel)
C_GRID = np.logspace(-2, 2, 9)  # 0.01 ... 100

# QSVM feature map settings
nq = 7
fm_z = z_feature_map(feature_dimension=nq, reps=1, entanglement="linear", parameter_prefix='z')
fm_x = z_feature_map(feature_dimension=nq, reps=1, entanglement="linear", parameter_prefix='x')

np.random.seed(RANDOM_SEED)

# ------------------------------- DATA (N=96) -------------------------------
X_df, y_sr = load_breast_cancer(return_X_y=True, as_frame=True)
X_full = X_df.to_numpy()
y_full = (y_sr.values == 1).astype(int)

# pick the SAME N=96 indices for both feature sets (fair comparison)
sss = StratifiedShuffleSplit(n_splits=1, train_size=N, random_state=RANDOM_SEED)
(train_idx, _), = sss.split(X_full, y_full)

# scaler on ALL data (as you requested)
scaler_all = StandardScaler().fit(X_full)
X_full_scaled = scaler_all.transform(X_full)

# --- build MANUAL-7 matrix with same rows ---
missing = [c for c in MANUAL_FEATURES if c not in X_df.columns]
if missing:
    raise ValueError(f"Missing manual features: {missing}")
X_manual_all = X_df[MANUAL_FEATURES].to_numpy()
X_manual_all = StandardScaler().fit_transform(X_manual_all)  # scale manual features on ALL
X_manual = X_manual_all[train_idx]
y_manual = y_full[train_idx]

# --- build RFE-7 matrix with same rows ---
logreg = LogisticRegression(max_iter=500, solver="lbfgs", random_state=RANDOM_SEED)
rfe = RFE(logreg, n_features_to_select=nq).fit(X_full_scaled, y_full)
rfe_mask = rfe.support_
rfe_names = list(X_df.columns[rfe_mask])
X_rfe_all = X_full_scaled[:, rfe_mask]
X_rfe = X_rfe_all[train_idx]
y_rfe = y_full[train_idx]

print(f"[INFO] Using N={N} samples. Class balance: {Counter(int(v) for v in y_manual)}")
print(f"[INFO] RFE-7 features: {rfe_names}")

# ----------------- LIVE CONNECTION + SNAPSHOT (retry) ----------------------
def clone_backend_with_retry(name, tries=5, save_dir=OUTDIR):
    service = QiskitRuntimeService()
    try:
        bk = service.backend(name)
        st = bk.status()
        print(f"[CONNECTED] backend={bk.name} | operational={st.operational} | pending_jobs={st.pending_jobs} | status='{st.status_msg}'")
    except Exception as e:
        print(f"[WARN] Could not query backend status for '{name}': {e}")

    last_err = None
    for a in range(1, tries+1):
        try:
            backend = service.backend(name)
            sim = AerSimulator.from_backend(backend)
            sim.set_options(seed_simulator=RANDOM_SEED)
            ts = None
            # save properties snapshot if available
            try:
                props = backend.properties()
                try:
                    ts = props.last_update_date
                except Exception:
                    ts = None
                with open(os.path.join(save_dir, "backend_properties.txt"), "w", encoding="utf-8") as f:
                    f.write(str(props))
            except Exception as e_props:
                # attempt to build NoiseModel from props (may fail on V2)
                try:
                    nm = NoiseModel.from_backend(props)  # may warn if V1 deprecations
                except Exception:
                    pass
                with open(os.path.join(save_dir, "backend_properties.txt"), "w", encoding="utf-8") as f:
                    f.write(f"[no properties available] {e_props}")
            print(f"[SNAPSHOT] Cloned from {name} on attempt {a}/{tries}"
                  + (f" (properties timestamp={ts})" if ts else ""))
            return sim
        except Exception as e:
            last_err = e
            print(f"[SNAPSHOT-RETRY {a}/{tries}] clone failed: {e}")
    print(f"[SNAPSHOT] Final fallback to plain AerSimulator due to error: {last_err}")
    sim = AerSimulator()
    sim.set_options(seed_simulator=RANDOM_SEED)
    # still dump something for reproducibility
    with open(os.path.join(save_dir, "backend_properties.txt"), "w", encoding="utf-8") as f:
        f.write(f"[fallback] could not clone {name}: {last_err}")
    return sim

aer_sim = clone_backend_with_retry(BACKEND_NAME)

# conservative threading
try:
    import multiprocessing as mp
    aer_sim.set_options(
        max_parallel_experiments=1,
        max_parallel_shots=1,
        max_parallel_threads=min(4, mp.cpu_count()),
    )
except Exception:
    pass

# ------------------------------- TEMPLATE CIRCUIT --------------------------
qc = QuantumCircuit(nq, name="fid")
qc.compose(fm_z, qubits=range(nq), inplace=True)
qc.compose(fm_x.inverse(), qubits=range(nq), inplace=True)
cr = ClassicalRegister(nq, "m")
qc.add_register(cr)
qc.measure(range(nq), cr)

t0_build = time.perf_counter()
qc_t = transpile(
    qc,
    backend=aer_sim,                # IMPORTANT: pass only backend
    initial_layout=INITIAL_LAYOUT,  # your chain
    optimization_level=1,
)
t1_build = time.perf_counter()
print(f"[INFO] Prepared circuit (transpile). Time: {t1_build - t0_build:.2f}s")

def extract_params_by_prefix(circ, prefix: str):
    plist = [p for p in circ.parameters if p.name.startswith(f"{prefix}[")]
    plist.sort(key=lambda p: int(p.name[p.name.find('[')+1:-1]))
    return plist
params_x = extract_params_by_prefix(qc_t, "x")
params_z = extract_params_by_prefix(qc_t, "z")

# ------------------------------- KERNEL HELPERS ----------------------------
STATS = {"adapted_pairs": 0, "total_pairs": 0, "max_shots_seen": BASE_SHOTS}

def counts_to_p0(counts_dict):
    shots_used = sum(counts_dict.values())
    return counts_dict.get("0"*nq, 0) / max(1, shots_used), shots_used

def pub_for_pair(Arow, Brow):
    pmap = {}
    for k in range(nq):
        pmap[params_z[k]] = float(Brow[k])
        pmap[params_x[k]] = float(Arow[k])
    return (qc_t, pmap)

def run_counts_pubs(pubs, shots, batch=RUN_BATCH, seed=1234):
    out = []
    for start in range(0, len(pubs), batch):
        sub = pubs[start:start+batch]
        bound = [circ.assign_parameters(pmap, inplace=False) for (circ, pmap) in sub]
        job = aer_sim.run(bound, shots=shots, seed_simulator=seed)
        res = job.result()
        counts = res.get_counts()
        if isinstance(counts, dict): out.append(counts)
        else: out.extend(counts)
    return out

def needed_shots_for_sigma(p_hat, sigma):
    p = max(1e-6, min(1-1e-6, p_hat))
    return int(math.ceil(p*(1-p)/(sigma*sigma)))

def p0s_with_adaptive(pubs, base_shots=BASE_SHOTS, band=BAND, sigma=SIGMA_TARGET, max_shots=MAX_SHOTS, seed=1234):
    counts1 = run_counts_pubs(pubs, shots=base_shots, seed=seed)
    p0 = np.zeros(len(pubs), dtype=float)
    shots_tot = np.zeros(len(pubs), dtype=int)
    for i, c in enumerate(counts1):
        p, s = counts_to_p0(c); p0[i] = p; shots_tot[i] = s

    lo, hi = band
    need = []
    for i, p in enumerate(p0):
        if lo <= p <= hi:
            req = needed_shots_for_sigma(p, sigma)
            add = max(0, min(max_shots - shots_tot[i], req - shots_tot[i]))
            if add > 0:
                need.append((i, add))

    if need:
        from collections import defaultdict
        groups = defaultdict(list)
        for idx, add in need: groups[add].append(idx)
        for add, idxs in groups.items():
            add_counts = run_counts_pubs([pubs[i] for i in idxs], shots=add, seed=seed+1)
            for j, c2 in enumerate(add_counts):
                i = idxs[j]
                p_add, s_add = counts_to_p0(c2)
                s1 = shots_tot[i]; p1 = p0[i]
                s_sum = s1 + s_add
                p_comb = (p1*s1 + p_add*s_add) / max(1, s_sum)
                p0[i] = p_comb; shots_tot[i] = s_sum

    STATS["total_pairs"] += len(pubs)
    STATS["adapted_pairs"] += int(np.sum(shots_tot > base_shots))
    STATS["max_shots_seen"] = max(STATS["max_shots_seen"], int(shots_tot.max()))
    return [(float(p0[i]), int(shots_tot[i])) for i in range(len(pubs))]

def p0s(pubs):
    return p0s_with_adaptive(pubs) if USE_ADAPTIVE else [(counts_to_p0(c))[0] for c in run_counts_pubs(pubs, shots=BASE_SHOTS)]

def kernel_square_upper(A, chunk=KERNEL_CHUNK, details=None):
    n = A.shape[0]
    K_raw = np.zeros((n, n), dtype=float)
    idx_pairs, pubs = [], []
    for i in range(n):
        for j in range(i, n):
            idx_pairs.append((i, j))
            pubs.append(pub_for_pair(A[i], A[j]))
    for start in range(0, len(pubs), chunk):
        sub = pubs[start:start+chunk]
        vals = p0s(sub)
        for (p0_val, s), (i, j) in zip(vals, idx_pairs[start:start+chunk]):
            K_raw[i, j] = p0_val;  K_raw[j, i] = p0_val
            if details is not None:
                details.append({"block":"train", "i":int(i), "j":int(j), "p0":float(p0_val), "shots_total":int(s)})
    return K_raw

def kernel_cross(A, B, chunk=KERNEL_CHUNK, details=None):
    K_raw = np.zeros((A.shape[0], B.shape[0]), dtype=float)
    pubs = [pub_for_pair(A[i], B[j]) for i in range(A.shape[0]) for j in range(B.shape[0])]
    idx = 0
    for start in range(0, len(pubs), chunk):
        sub = pubs[start:start+chunk]
        vals = p0s(sub)
        for (p0_val, s) in vals:
            i = idx // B.shape[0]; j = idx % B.shape[0]
            K_raw[i, j] = p0_val
            if details is not None:
                details.append({"block":"testxtrain", "i":int(i), "j":int(j), "p0":float(p0_val), "shots_total":int(s)})
            idx += 1
    return K_raw

def kernel_diag(A, chunk=KERNEL_CHUNK, details=None):
    pubs = [pub_for_pair(A[i], A[i]) for i in range(A.shape[0])]
    d = np.zeros(A.shape[0], dtype=float)
    for start in range(0, len(pubs), chunk):
        sub = pubs[start:start+chunk]
        vals = p0s(sub)
        for offs, (p0_val, s) in enumerate(vals):
            d[start + offs] = p0_val
            if details is not None:
                details.append({"block":"diag", "i":int(start+offs), "j":int(start+offs), "p0":float(p0_val), "shots_total":int(s)})
    return d

def normalize_square_from_diag(K_raw, d):
    d = np.clip(d, 1e-12, None)
    Kn = K_raw / np.sqrt(np.outer(d, d))
    Kn = 0.5 * (Kn + Kn.T)
    np.fill_diagonal(Kn, 1.0)
    return Kn

def normalize_cross_from_diag(K_raw, d_test, d_train):
    d_test  = np.clip(d_test,  1e-12, None)
    d_train = np.clip(d_train, 1e-12, None)
    return K_raw / np.sqrt(np.outer(d_test, d_train))

# ---- NEW: kernel centering (train & test) ---------------------------------
def center_train_kernel(K):
    r = K.mean(axis=1, keepdims=True)
    c = K.mean(axis=0, keepdims=True)
    g = K.mean()
    return K - r - c + g

def center_test_kernel(K_te, K_tr):
    c_tr = K_tr.mean(axis=0, keepdims=True)
    g_tr = K_tr.mean()
    r_te = K_te.mean(axis=1, keepdims=True)
    return K_te - c_tr - r_te + g_tr

# ---- Inner CV over C on precomputed kernel --------------------------------
def pick_best_C(K_tr, y_tr, Cs=C_GRID, seed=RANDOM_SEED):
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=seed)
    best_auc, best_C = -np.inf, Cs[0]
    n = len(y_tr)
    for C in Cs:
        aucs = []
        for tr2, va2 in skf.split(np.arange(n), y_tr):
            K_tt = K_tr[np.ix_(tr2, tr2)]
            K_tv = K_tr[np.ix_(va2, tr2)]
            clf2 = SVC(kernel="precomputed", C=C, class_weight="balanced")
            clf2.fit(K_tt, y_tr[tr2])
            scores2 = clf2.decision_function(K_tv)
            aucs.append(roc_auc_score(y_tr[va2], scores2))
        m = float(np.mean(aucs))
        if m > best_auc: best_auc, best_C = m, C
    return best_C, best_auc

# ------------------------------- CV RUNNER ---------------------------------
def run_cv(label, X, y):
    print(f"\n===== {label} =====")
    print(f"[INFO] AER run | Adaptive={'ON' if USE_ADAPTIVE else 'OFF'} | Base shots={BASE_SHOTS} | RUN_BATCH={RUN_BATCH} | KERNEL_CHUNK={KERNEL_CHUNK}")

    cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_SEED)
    fold_aucs, fold_accs, fold_f1s = [], [], []
    fold_sens, fold_spec, fold_prec, fold_rec = [], [], [], []
    agg_tn = agg_fp = agg_fn = agg_tp = 0
    fold_ksecs, fold_tsecs = [], []

    for fold_id, (tr_idx, te_idx) in enumerate(cv.split(X, y), 1):
        X_tr, X_te = X[tr_idx], X[te_idx]
        y_tr, y_te = y[tr_idx], y[te_idx]
        DETAILS = [] if SAVE_PER_ENTRY_DETAILS else None

        t0 = time.perf_counter()

        # Train square -> diag -> normalize -> CENTER
        t0k = time.perf_counter()
        K_tr_raw = kernel_square_upper(X_tr, details=DETAILS)
        d_tr     = np.diag(K_tr_raw).copy()
        K_tr     = normalize_square_from_diag(K_tr_raw, d_tr)
        K_tr     = center_train_kernel(K_tr)

        # Test diag + cross -> normalize -> CENTER(using train stats)
        d_te     = kernel_diag(X_te, details=DETAILS)
        K_te_raw = kernel_cross(X_te, X_tr, details=DETAILS)
        K_te     = normalize_cross_from_diag(K_te_raw, d_te, d_tr)
        K_te     = center_test_kernel(K_te, K_tr)
        t1k = time.perf_counter()

        # (optional) save kernels
        pd.DataFrame(K_tr).to_csv(os.path.join(OUTDIR, f"{label}_K_train_f{fold_id}.csv"), index=False)
        pd.DataFrame(K_te).to_csv(os.path.join(OUTDIR, f"{label}_K_test_f{fold_id}.csv"), index=False)

        # Inner CV over C
        C_best, _ = pick_best_C(K_tr, y_tr)

        # Train final SVC (balanced) and score
        clf = SVC(kernel="precomputed", C=C_best, class_weight="balanced")
        clf.fit(K_tr, y_tr)
        scores = clf.decision_function(K_te)
        y_pred = (scores >= 0).astype(int)   # default threshold; centering should make this ok

        # Metrics
        auc = roc_auc_score(y_te, scores)
        acc = accuracy_score(y_te, y_pred)
        f1  = f1_score(y_te, y_pred, average="binary", pos_label=1)
        tn, fp, fn, tp = confusion_matrix(y_te, y_pred, labels=[0,1]).ravel()
        sens = tp / max(1, tp+fn)
        spec = tn / max(1, tn+fp)
        prec = tp / max(1, tp+fp)
        rec  = sens

        t1 = time.perf_counter()

        fold_aucs.append(auc); fold_accs.append(acc); fold_f1s.append(f1)
        fold_sens.append(sens); fold_spec.append(spec); fold_prec.append(prec); fold_rec.append(rec)
        agg_tn += tn; agg_fp += fp; agg_fn += fn; agg_tp += tp
        fold_ksecs.append(t1k - t0k); fold_tsecs.append(t1 - t0)

        print(f"  fold {fold_id}: AUC={auc:.4f}  Acc={acc:.4f}  F1={f1:.4f}  "
              f"TP={tp} FP={fp} TN={tn} FN={fn}  "
              f"Sens={sens:.4f} Spec={spec:.4f} Prec={prec:.4f} Rec={rec:.4f}  "
              f"| C*={C_best:g} | kernel={fold_ksecs[-1]:.2f}s | total={fold_tsecs[-1]:.2f}s")

    # Summary
    mean_auc = float(np.mean(fold_aucs)); std_auc = float(np.std(fold_aucs))
    mean_acc = float(np.mean(fold_accs)); std_acc = float(np.std(fold_accs))
    mean_f1  = float(np.mean(fold_f1s));  std_f1  = float(np.std(fold_f1s))
    mean_sens = float(np.mean(fold_sens)); std_sens = float(np.std(fold_sens))
    mean_spec = float(np.mean(fold_spec)); std_spec = float(np.std(fold_spec))
    mean_prec = float(np.mean(fold_prec)); std_prec = float(np.std(fold_prec))
    mean_rec  = float(np.mean(fold_rec));  std_rec  = float(np.std(fold_rec))
    mean_k   = float(np.mean(fold_ksecs)); mean_t = float(np.mean(fold_tsecs))

    print("\n=== Summary:", label, "===")
    print(f"Mean AUC={mean_auc:.4f} (+/- {std_auc:.4f})  "
          f"Mean Acc={mean_acc:.4f} (+/- {std_acc:.4f})  "
          f"Mean F1={mean_f1:.4f} (+/- {std_f1:.4f})")
    print(f"Mean Sens={mean_sens:.4f} (+/- {std_sens:.4f})  "
          f"Mean Spec={mean_spec:.4f} (+/- {std_spec:.4f})  "
          f"Mean Prec={mean_prec:.4f} (+/- {std_prec:.4f})  "
          f"Mean Rec={mean_rec:.4f} (+/- {std_rec:.4f})")
    print(f"Totals: TP={agg_tp}  FP={agg_fp}  TN={agg_tn}  FN={agg_fn}")
    print(f"Timing: mean kernel/fold={mean_k:.2f}s  |  mean total/fold={mean_t:.2f}s")

# ------------------------------- RUN BOTH ----------------------------------
run_cv("MANUAL7", X_manual, y_manual)
run_cv("RFE7",    X_rfe,    y_rfe)

print(f"\n[ADAPTIVE] pairs={STATS['total_pairs']}  adapted={STATS['adapted_pairs']}  "
      f"({100.0*STATS['adapted_pairs']/max(1,STATS['total_pairs']):.1f}%)  "
      f"max_shots_seen={STATS['max_shots_seen']}")
print(f"[SAVED] kernels in: {os.path.abspath(OUTDIR)}")


In [ ]:
# === QSVM fidelity kernel (RFE-7), N=96, 3-fold CV ===
# - Snapshot clone of real backend (Brisbane by default), with retry & logging
# - Auto-select best 7-qubit chain (INITIAL_LAYOUT) from snapshot (2Q + RO error)
# - Base shots=256 with adaptive top-up (to 1025) only for ambiguous pairs
# - Diagonal normalization + double-centering
# - SVC(class_weight="balanced") with inner CV over trimmed C-grid
# - Threshold calibration (Youden’s J) on train scores per fold
# - Prints a "QPU-READY CHECKLIST", chosen layout, and full metrics
#
# Pins tested with: qiskit==2.2.1, qiskit-aer==0.17.2, qiskit-ibm-runtime==0.42.0

import os, time, math, numpy as np, pandas as pd
from collections import Counter, defaultdict

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, confusion_matrix

from qiskit import QuantumCircuit, ClassicalRegister, transpile
from qiskit.circuit.library import z_feature_map
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel
from qiskit_ibm_runtime import QiskitRuntimeService

# ------------------------------- USER CONFIG -------------------------------
BACKEND_NAME    = "ibm_brisbane"           # change to "ibm_torino" etc. if you want
N               = 96
N_SPLITS        = 3
RANDOM_SEED     = 42

# Shots & adaptive policy
BASE_SHOTS      = 256
USE_ADAPTIVE    = True
BAND            = (0.40, 0.60)     # top-up only if p0 in this band
SIGMA_TARGET    = 0.0125
MAX_SHOTS       = 1025

# Throughput
RUN_BATCH       = 64               # circuits per run()
KERNEL_CHUNK    = 512              # pubs per outer chunk

# Output
OUTDIR          = "./kernels_noisy_aer_brisbane"
SAVE_PER_ENTRY_DETAILS = False
os.makedirs(OUTDIR, exist_ok=True)

# Fixed RFE-7 (your chosen list)
RFE7_FEATURES = [
    'mean concave points',
    'radius error',
    'worst radius',
    'worst texture',
    'worst perimeter',
    'worst area',
    'worst concave points',
]

# Trimmed C-grid (avoid very tiny C that created all-positive folds)
C_GRID = np.logspace(-1.5, 2, 8)  # 0.0316 ... 100

# QSVM feature map settings
nq = 7
fm_z = z_feature_map(feature_dimension=nq, reps=1, entanglement="linear", parameter_prefix='z')
fm_x = z_feature_map(feature_dimension=nq, reps=1, entanglement="linear", parameter_prefix='x')

np.random.seed(RANDOM_SEED)

# ------------------------------- DATA (N=96, RFE-7 fixed) ------------------
X_df, y_sr = load_breast_cancer(return_X_y=True, as_frame=True)

missing = [c for c in RFE7_FEATURES if c not in X_df.columns]
if missing:
    raise ValueError(f"Missing RFE-7 features: {missing}")

X_sel_all = X_df[RFE7_FEATURES].to_numpy()
y_all     = (y_sr.values == 1).astype(int)

# scaler on ALL rows (your requested variant)
scaler_all = StandardScaler().fit(X_sel_all)
X_sel_all  = scaler_all.transform(X_sel_all)

# choose the SAME N=96 indices (stratified)
sss = StratifiedShuffleSplit(n_splits=1, train_size=N, random_state=RANDOM_SEED)
(train_idx, _), = sss.split(X_sel_all, y_all)
X = X_sel_all[train_idx]
y = y_all[train_idx]

print(f"[INFO] Using N={N} samples. Class balance: {Counter(int(v) for v in y)}")

# ----------------- LIVE CONNECTION + SNAPSHOT (retry) ----------------------
def clone_backend_with_retry(name, tries=5, save_dir=OUTDIR):
    """Try to clone a device-like Aer sim from a live backend; save snapshot text; retry on errors."""
    service = QiskitRuntimeService()
    snapshot_ok = False
    status_blob = {}
    # Query backend status banner
    try:
        bk = service.backend(name)
        st = bk.status()
        status_blob = {
            "name": bk.name,
            "operational": getattr(st, "operational", None),
            "pending_jobs": getattr(st, "pending_jobs", None),
            "status_msg": getattr(st, "status_msg", None),
        }
        print(f"[CONNECTED] backend={bk.name} | operational={st.operational} | "
              f"pending_jobs={st.pending_jobs} | status='{st.status_msg}'")
    except Exception as e:
        print(f"[WARN] Could not query backend status for '{name}': {e}")

    last_err = None
    props_saved = False
    for a in range(1, tries+1):
        try:
            backend = service.backend(name)
            # Save a text snapshot of properties (if available)
            try:
                props = backend.properties()
                with open(os.path.join(save_dir, "backend_properties.txt"), "w", encoding="utf-8") as f:
                    f.write(str(props))
                ts = getattr(props, "last_update_date", None)
                props_saved = True
            except Exception as e_props:
                with open(os.path.join(save_dir, "backend_properties.txt"), "w", encoding="utf-8") as f:
                    f.write(f"[no properties available] {e_props}")
                ts = None

            sim = AerSimulator.from_backend(backend)
            sim.set_options(seed_simulator=RANDOM_SEED)
            print(f"[SNAPSHOT] Cloned from {name} on attempt {a}/{tries}"
                  + (f" (properties timestamp={ts})" if ts else ""))
            snapshot_ok = True
            return sim, backend, snapshot_ok, status_blob, props_saved
        except Exception as e:
            last_err = e
            print(f"[SNAPSHOT-RETRY {a}/{tries}] clone failed: {e}")

    print(f"[SNAPSHOT] Final fallback to plain AerSimulator due to error: {last_err}")
    sim = AerSimulator()
    sim.set_options(seed_simulator=RANDOM_SEED)
    # still dump something for reproducibility
    with open(os.path.join(save_dir, "backend_properties.txt"), "w", encoding="utf-8") as f:
        f.write(f"[fallback] could not clone {name}: {last_err}")
    return sim, None, snapshot_ok, status_blob, props_saved

aer_sim, live_backend, SNAPSHOT_OK, STATUS, PROPS_SAVED = clone_backend_with_retry(BACKEND_NAME)

# conservative threading
try:
    import multiprocessing as mp
    aer_sim.set_options(
        max_parallel_experiments=1,
        max_parallel_shots=1,
        max_parallel_threads=min(4, mp.cpu_count()),
    )
except Exception:
    pass

# ----------------- AUTO-SELECT BEST 7-QUBIT CHAIN (INITIAL_LAYOUT) ---------
def _readout_error_from_props(props, q):
    # Try direct readout_error, else average of confusion terms, else small default
    try:
        return float(props.readout_error(q))
    except Exception:
        try:
            p01 = props.qubit_property(q, "prob_meas1_prep0")[0].value
            p10 = props.qubit_property(q, "prob_meas0_prep1")[0].value
            return float(0.5*(p01 + p10))
        except Exception:
            return 0.02  # default if unknown

def _gate_error_from_props(props, pair):
    (q0, q1) = pair
    for g in ("ecr", "cx"):  # IBM heavy-hex is usually 'ecr'; else 'cx'
        try:
            return float(props.gate_error(g, [q0, q1]))
        except Exception:
            try:
                return float(props.gate_error(g, [q1, q0]))
            except Exception:
                pass
    return 0.01  # default small if unknown

def _get_coupling_edges(backend):
    # Try modern attribute first
    try:
        cm = backend.coupling_map
        if cm:
            return [tuple(e) for e in cm]
    except Exception:
        pass
    # Try legacy configuration()
    try:
        cfg = backend.configuration()
        if hasattr(cfg, "coupling_map") and cfg.coupling_map:
            return [tuple(e) for e in cfg.coupling_map]
    except Exception:
        pass
    return []

def pick_best_chain_7(backend):
    """Heuristic: among all simple greedy chains of length 7, minimize combined (2Q edge + RO) error."""
    edges = _get_coupling_edges(backend)
    if not edges:
        return None, None  # no info

    # Build undirected adjacency
    adj = defaultdict(set)
    for a, b in edges:
        adj[a].add(b); adj[b].add(a)

    # Try to get properties
    try:
        props = backend.properties()
    except Exception:
        props = None

    # Precompute readout error per qubit
    ro = defaultdict(lambda: 0.02)
    if props is not None:
        qs = set([q for e in edges for q in e])
        for q in qs:
            ro[q] = _readout_error_from_props(props, q)

    # Precompute 2Q error per undirected edge (min of both directions)
    eerr = {}
    if props is not None:
        for a, b in edges:
            eerr[(a, b)] = _gate_error_from_props(props, (a, b))
            eerr[(b, a)] = eerr[(a, b)]
    else:
        for a, b in edges:
            eerr[(a, b)] = eerr[(b, a)] = 0.01

    # Greedy search from every start: at each step pick neighbor with smallest (edge_err + 0.5*(ro_i+ro_j))
    best_chain, best_score = None, float("inf")
    for start in adj.keys():
        chain = [start]
        score = ro[start]  # include readout of first node
        ok = True
        while len(chain) < 7:
            cur = chain[-1]
            # candidates not visited
            cand = [nb for nb in adj[cur] if nb not in chain]
            if not cand:
                ok = False
                break
            # score each candidate step
            step_scores = []
            for nb in cand:
                step_scores.append((eerr.get((cur, nb), 0.02) + 0.5*(ro[cur] + ro[nb]), nb))
            smin, nb_best = min(step_scores, key=lambda t: t[0])
            chain.append(nb_best)
            score += smin + ro[nb_best]
        if ok and score < best_score:
            best_score, best_chain = score, chain

    return best_chain, best_score

# Decide INITIAL_LAYOUT
AUTO_LAYOUT = None
if live_backend is not None:
    AUTO_LAYOUT, LAYOUT_SCORE = pick_best_chain_7(live_backend)
if AUTO_LAYOUT is None or len(AUTO_LAYOUT) != 7:
    print("[LAYOUT] Auto-selection failed; will use a safe fallback chain [3,4,5,6,7,8,16].")
    INITIAL_LAYOUT = [3, 4, 5, 6, 7, 8, 16]
else:
    INITIAL_LAYOUT = list(AUTO_LAYOUT)
    print(f"[LAYOUT] Selected best 7-qubit chain: {INITIAL_LAYOUT} | score={LAYOUT_SCORE:.6f}")

# ------------------------------- TEMPLATE CIRCUIT --------------------------
qc = QuantumCircuit(nq, name="fid")
qc.compose(fm_z, qubits=range(nq), inplace=True)
qc.compose(fm_x.inverse(), qubits=range(nq), inplace=True)
cr = ClassicalRegister(nq, "m")
qc.add_register(cr)
qc.measure(range(nq), cr)

t0_build = time.perf_counter()
qc_t = transpile(
    qc,
    backend=aer_sim,                # IMPORTANT: pass only backend
    initial_layout=INITIAL_LAYOUT,  # auto-selected chain (or fallback)
    optimization_level=1,
)
t1_build = time.perf_counter()
print(f"[INFO] Prepared circuit (transpile). Time: {t1_build - t0_build:.2f}s")

def extract_params_by_prefix(circ, prefix: str):
    plist = [p for p in circ.parameters if p.name.startswith(f"{prefix}[")]
    plist.sort(key=lambda p: int(p.name[p.name.find('[')+1:-1]))
    return plist
params_x = extract_params_by_prefix(qc_t, "x")
params_z = extract_params_by_prefix(qc_t, "z")

# ------------------------------- KERNEL HELPERS ----------------------------
STATS = {"adapted_pairs": 0, "total_pairs": 0, "max_shots_seen": BASE_SHOTS}

def counts_to_p0(counts_dict):
    shots_used = sum(counts_dict.values())
    return counts_dict.get("0"*nq, 0) / max(1, shots_used), shots_used

def pub_for_pair(Arow, Brow):
    pmap = {}
    for k in range(nq):
        pmap[params_z[k]] = float(Brow[k])
        pmap[params_x[k]] = float(Arow[k])
    return (qc_t, pmap)

def run_counts_pubs(pubs, shots, batch=RUN_BATCH, seed=1234):
    out = []
    for start in range(0, len(pubs), batch):
        sub = pubs[start:start+batch]
        bound = [circ.assign_parameters(pmap, inplace=False) for (circ, pmap) in sub]
        job = aer_sim.run(bound, shots=shots, seed_simulator=seed)
        res = job.result()
        counts = res.get_counts()
        if isinstance(counts, dict): out.append(counts)
        else: out.extend(counts)
    return out

def needed_shots_for_sigma(p_hat, sigma):
    p = max(1e-6, min(1-1e-6, p_hat))
    return int(math.ceil(p*(1-p)/(sigma*sigma)))

def p0s_with_adaptive(pubs, base_shots=BASE_SHOTS, band=BAND, sigma=SIGMA_TARGET, max_shots=MAX_SHOTS, seed=1234):
    counts1 = run_counts_pubs(pubs, shots=base_shots, seed=seed)
    p0 = np.zeros(len(pubs), dtype=float)
    shots_tot = np.zeros(len(pubs), dtype=int)
    for i, c in enumerate(counts1):
        p, s = counts_to_p0(c); p0[i] = p; shots_tot[i] = s

    lo, hi = band
    need = []
    for i, p in enumerate(p0):
        if lo <= p <= hi:
            req = needed_shots_for_sigma(p, sigma)
            add = max(0, min(max_shots - shots_tot[i], req - shots_tot[i]))
            if add > 0:
                need.append((i, add))

    if need:
        groups = defaultdict(list)
        for idx, add in need: groups[add].append(idx)
        for add, idxs in groups.items():
            add_counts = run_counts_pubs([pubs[i] for i in idxs], shots=add, seed=seed+1)
            for j, c2 in enumerate(add_counts):
                i = idxs[j]
                p_add, s_add = counts_to_p0(c2)
                s1 = shots_tot[i]; p1 = p0[i]
                s_sum = s1 + s_add
                p_comb = (p1*s1 + p_add*s_add) / max(1, s_sum)
                p0[i] = p_comb; shots_tot[i] = s_sum

    STATS["total_pairs"] += len(pubs)
    STATS["adapted_pairs"] += int(np.sum(shots_tot > base_shots))
    STATS["max_shots_seen"] = max(STATS["max_shots_seen"], int(shots_tot.max()))
    return [(float(p0[i]), int(shots_tot[i])) for i in range(len(pubs))]

def p0s(pubs):
    return p0s_with_adaptive(pubs) if USE_ADAPTIVE else [(counts_to_p0(c))[0] for c in run_counts_pubs(pubs, shots=BASE_SHOTS)]

def kernel_square_upper(A, chunk=KERNEL_CHUNK, details=None):
    n = A.shape[0]
    K_raw = np.zeros((n, n), dtype=float)
    idx_pairs, pubs = [], []
    for i in range(n):
        for j in range(i, n):
            idx_pairs.append((i, j))
            pubs.append(pub_for_pair(A[i], A[j]))
    for start in range(0, len(pubs), chunk):
        sub = pubs[start:start+chunk]
        vals = p0s(sub)
        for (p0_val, s), (i, j) in zip(vals, idx_pairs[start:start+chunk]):
            K_raw[i, j] = p0_val;  K_raw[j, i] = p0_val
            if details is not None:
                details.append({"block":"train", "i":int(i), "j":int(j), "p0":float(p0_val), "shots_total":int(s)})
    return K_raw

def kernel_cross(A, B, chunk=KERNEL_CHUNK, details=None):
    K_raw = np.zeros((A.shape[0], B.shape[0]), dtype=float)
    pubs = [pub_for_pair(A[i], B[j]) for i in range(A.shape[0]) for j in range(B.shape[0])]
    idx = 0
    for start in range(0, len(pubs), chunk):
        sub = pubs[start:start+chunk]
        vals = p0s(sub)
        for (p0_val, s) in vals:
            i = idx // B.shape[0]; j = idx % B.shape[0]
            K_raw[i, j] = p0_val
            if details is not None:
                details.append({"block":"testxtrain", "i":int(i), "j":int(j), "p0":float(p0_val), "shots_total":int(s)})
            idx += 1
    return K_raw

def kernel_diag(A, chunk=KERNEL_CHUNK, details=None):
    pubs = [pub_for_pair(A[i], A[i]) for i in range(A.shape[0])]
    d = np.zeros(A.shape[0], dtype=float)
    for start in range(0, len(pubs), chunk):
        sub = pubs[start:start+chunk]
        vals = p0s(sub)
        for offs, (p0_val, s) in enumerate(vals):
            d[start + offs] = p0_val
            if details is not None:
                details.append({"block":"diag", "i":int(start+offs), "j":int(start+offs), "p0":float(p0_val), "shots_total":int(s)})
    return d

def normalize_square_from_diag(K_raw, d):
    d = np.clip(d, 1e-12, None)
    Kn = K_raw / np.sqrt(np.outer(d, d))
    Kn = 0.5 * (Kn + Kn.T)
    np.fill_diagonal(Kn, 1.0)
    return Kn

def normalize_cross_from_diag(K_raw, d_test, d_train):
    d_test  = np.clip(d_test,  1e-12, None)
    d_train = np.clip(d_train, 1e-12, None)
    return K_raw / np.sqrt(np.outer(d_test, d_train))

# ---- Double-centering (train & test) --------------------------------------
def center_train_kernel(K):
    r = K.mean(axis=1, keepdims=True)
    c = K.mean(axis=0, keepdims=True)
    g = K.mean()
    return K - r - c + g

def center_test_kernel(K_te, K_tr):
    c_tr = K_tr.mean(axis=0, keepdims=True)
    g_tr = K_tr.mean()
    r_te = K_te.mean(axis=1, keepdims=True)
    return K_te - c_tr - r_te + g_tr

# ---- Inner CV over C on precomputed kernel --------------------------------
def pick_best_C(K_tr, y_tr, Cs=C_GRID, seed=RANDOM_SEED):
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=seed)
    best_auc, best_C = -np.inf, Cs[0]
    n = len(y_tr)
    for C in Cs:
        aucs = []
        for tr2, va2 in skf.split(np.arange(n), y_tr):
            K_tt = K_tr[np.ix_(tr2, tr2)]
            K_tv = K_tr[np.ix_(va2, tr2)]
            clf2 = SVC(kernel="precomputed", C=C, class_weight="balanced")
            clf2.fit(K_tt, y_tr[tr2])
            scores2 = clf2.decision_function(K_tv)
            aucs.append(roc_auc_score(y_tr[va2], scores2))
        m = float(np.mean(aucs))
        if m > best_auc: best_auc, best_C = m, C
    return best_C, best_auc

# ---- Threshold calibration (Youden’s J) -----------------------------------
def youden_threshold(scores, y_true):
    # pick threshold maximizing TPR - FPR
    order = np.argsort(scores)
    sc = scores[order]; yt = y_true[order]
    P = np.sum(yt == 1); Nn = np.sum(yt == 0)
    if P == 0 or Nn == 0:
        return 0.0  # degenerate
    # scan thresholds between consecutive unique scores
    best_J, best_t = -1.0, 0.0
    # cumulative positives to the right if we predict >= t as positive
    # precompute TPR/FPR at each cutoff
    for k in range(len(sc)):
        t = sc[k]
        y_pred = (scores >= t).astype(int)
        tp = np.sum((y_pred == 1) & (y_true == 1))
        fp = np.sum((y_pred == 1) & (y_true == 0))
        fn = P - tp
        tn = Nn - fp
        tpr = tp / max(1, P)
        fpr = fp / max(1, Nn)
        J = tpr - fpr
        if J > best_J:
            best_J, best_t = J, t
    return float(best_t)

# ------------------------------- 3-FOLD CV ---------------------------------
print(f"[INFO] AER run | Adaptive={'ON' if USE_ADAPTIVE else 'OFF'} | Base shots={BASE_SHOTS} | "
      f"RUN_BATCH={RUN_BATCH} | KERNEL_CHUNK={KERNEL_CHUNK}")
print(f"[INFO] C-grid (trimmed): {C_GRID}")

fold_aucs, fold_accs, fold_f1s = [], [], []
fold_sens, fold_spec, fold_prec, fold_rec = [], [], [], []
agg_tn = agg_fp = agg_fn = agg_tp = 0
fold_ksecs, fold_tsecs = [], []
THRESHOLDS = []

cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_SEED)
for fold_id, (tr_idx, te_idx) in enumerate(cv.split(X, y), 1):
    X_tr, X_te = X[tr_idx], X[te_idx]
    y_tr, y_te = y[tr_idx], y[te_idx]
    DETAILS = [] if SAVE_PER_ENTRY_DETAILS else None

    t0 = time.perf_counter()

    # Train square -> diag -> normalize -> CENTER
    t0k = time.perf_counter()
    K_tr_raw = kernel_square_upper(X_tr, details=DETAILS)
    d_tr     = np.diag(K_tr_raw).copy()
    K_tr     = normalize_square_from_diag(K_tr_raw, d_tr)
    K_tr     = center_train_kernel(K_tr)

    # Test diag + cross -> normalize -> CENTER(using train stats)
    d_te     = kernel_diag(X_te, details=DETAILS)
    K_te_raw = kernel_cross(X_te, X_tr, details=DETAILS)
    K_te     = normalize_cross_from_diag(K_te_raw, d_te, d_tr)
    K_te     = center_test_kernel(K_te, K_tr)
    t1k = time.perf_counter()

    # Save kernels
    pd.DataFrame(K_tr).to_csv(os.path.join(OUTDIR, f"RFE7_K_train_f{fold_id}.csv"), index=False)
    pd.DataFrame(K_te).to_csv(os.path.join(OUTDIR, f"RFE7_K_test_f{fold_id}.csv"), index=False)

    # Inner CV over C
    C_best, _ = pick_best_C(K_tr, y_tr)

    # Train final SVC (balanced)
    clf = SVC(kernel="precomputed", C=C_best, class_weight="balanced")
    clf.fit(K_tr, y_tr)

    # Scores
    scores_tr = clf.decision_function(K_tr)
    thr = youden_threshold(scores_tr, y_tr)
    THRESHOLDS.append(thr)
    scores_te = clf.decision_function(K_te)
    y_pred = (scores_te >= thr).astype(int)

    # Metrics
    auc = roc_auc_score(y_te, scores_te)
    acc = accuracy_score(y_te, y_pred)
    f1  = f1_score(y_te, y_pred, average="binary", pos_label=1)
    tn, fp, fn, tp = confusion_matrix(y_te, y_pred, labels=[0,1]).ravel()
    sens = tp / max(1, tp+fn)
    spec = tn / max(1, tn+fp)
    prec = tp / max(1, tp+fp)
    rec  = sens

    t1 = time.perf_counter()

    fold_aucs.append(auc); fold_accs.append(acc); fold_f1s.append(f1)
    fold_sens.append(sens); fold_spec.append(spec); fold_prec.append(prec); fold_rec.append(rec)
    agg_tn += tn; agg_fp += fp; agg_fn += fn; agg_tp += tp
    fold_ksecs.append(t1k - t0k); fold_tsecs.append(t1 - t0)

    print(f"  fold {fold_id}: AUC={auc:.4f}  Acc={acc:.4f}  F1={f1:.4f}  "
          f"TP={tp} FP={fp} TN={tn} FN={fn}  "
          f"Sens={sens:.4f} Spec={spec:.4f} Prec={prec:.4f} Rec={rec:.4f}  "
          f"| C*={C_best:g} | thr*={thr:.6f} | kernel={fold_ksecs[-1]:.2f}s | total={fold_tsecs[-1]:.2f}s")

# Summary
mean_auc = float(np.mean(fold_aucs)); std_auc = float(np.std(fold_aucs))
mean_acc = float(np.mean(fold_accs)); std_acc = float(np.std(fold_accs))
mean_f1  = float(np.mean(fold_f1s));  std_f1  = float(np.std(fold_f1s))
mean_sens = float(np.mean(fold_sens)); std_sens = float(np.std(fold_sens))
mean_spec = float(np.mean(fold_spec)); std_spec = float(np.std(fold_spec))
mean_prec = float(np.mean(fold_prec)); std_prec = float(np.std(fold_prec))
mean_rec  = float(np.mean(fold_rec));  std_rec  = float(np.std(fold_rec))
mean_k   = float(np.mean(fold_ksecs)); mean_t = float(np.mean(fold_tsecs))
mean_thr = float(np.mean(THRESHOLDS)); std_thr = float(np.std(THRESHOLDS))

print("\n=== Summary: RFE-7 (centered, balanced, trimmed C, calibrated thr) ===")
print(f"Mean AUC={mean_auc:.4f} (+/- {std_auc:.4f})  "
      f"Mean Acc={mean_acc:.4f} (+/- {std_acc:.4f})  "
      f"Mean F1={mean_f1:.4f} (+/- {std_f1:.4f})")
print(f"Mean Sens={mean_sens:.4f} (+/- {std_sens:.4f})  "
      f"Mean Spec={mean_spec:.4f} (+/- {std_spec:.4f})  "
      f"Mean Prec={mean_prec:.4f} (+/- {std_prec:.4f})  "
      f"Mean Rec={mean_rec:.4f} (+/- {std_rec:.4f})")
print(f"Threshold (Youden) mean={mean_thr:.6f} (+/- {std_thr:.6f})")
print(f"Totals: TP={agg_tp}  FP={agg_fp}  TN={agg_tn}  FN={agg_fn}")
print(f"Timing: mean kernel/fold={mean_k:.2f}s  |  mean total/fold={mean_t:.2f}s")

# Adaptive shots stats
print(f"\n[ADAPTIVE] pairs={STATS['total_pairs']}  adapted={STATS['adapted_pairs']}  "
      f"({100.0*STATS['adapted_pairs']/max(1,STATS['total_pairs']):.1f}%)  "
      f"max_shots_seen={STATS['max_shots_seen']}")

# QPU-ready checklist
print("\n=== QPU-READY CHECKLIST ===")
print(f"Snapshot cloned from '{BACKEND_NAME}': {SNAPSHOT_OK}")
print(f"Backend status: {STATUS if STATUS else 'unavailable'}")
print(f"Properties saved to: {os.path.abspath(os.path.join(OUTDIR,'backend_properties.txt'))}  (saved={PROPS_SAVED})")
print(f"Auto INITIAL_LAYOUT (7 qubits): {INITIAL_LAYOUT}  (auto={'yes' if SNAPSHOT_OK else 'no / fallback'})")
print(f"Centering: ON  | class_weight: balanced  | C-grid: {C_GRID[0]:.4g}..{C_GRID[-1]:.4g}  | threshold: Youden per fold")
print(f"Shots: base={BASE_SHOTS}  adaptive up to={MAX_SHOTS} in band={BAND}")
print(f"[SAVED] kernels in: {os.path.abspath(OUTDIR)}")


ADING THE M3 AND DD 

In [ ]:
# === QSVM fidelity kernel, N=96, 3-fold CV (RFE-7), with DD + optional M3 ===
# - Snapshot-cloned Aer from your backend (retry) + auto best 7-qubit chain
# - Base shots=256 + adaptive top-up to 1025 for ambiguous pairs
# - Kernel centering + class_weight="balanced" + trimmed C grid
# - Threshold calibration (Youden) per fold
# - DD: robust (XY4 if 'y' is timed; else CPMG/XX)  + optional M3 readout mitigation
#
# Pins: qiskit==2.2.1, qiskit-aer==0.17.2, qiskit-ibm-runtime==0.42.0, mthree (optional)

import os, time, math, numpy as np, pandas as pd
from collections import Counter

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import RFE
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, confusion_matrix

from qiskit import QuantumCircuit, ClassicalRegister, transpile
from qiskit.circuit.library import z_feature_map, XGate, YGate
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import QiskitRuntimeService

# DD imports (critical)
from qiskit.transpiler import PassManager, InstructionDurations
from qiskit.transpiler.passes import ALAPScheduleAnalysis, PadDynamicalDecoupling

# Optional M3
try:
    import mthree as m3
    _M3_AVAILABLE = True
except Exception:
    _M3_AVAILABLE = False

# ------------------------------- USER CONFIG -------------------------------
BACKEND_NAME    = "ibm_brisbane"
USE_AUTO_LAYOUT = True              # pick best 7-qubit chain automatically (fallback to INITIAL_LAYOUT)
INITIAL_LAYOUT  = [3, 4, 5, 6, 7, 8, 16]  # used only if auto layout fails

N               = 96
N_SPLITS        = 3
RANDOM_SEED     = 42

# Shots & adaptive policy
BASE_SHOTS      = 256
USE_ADAPTIVE    = True
BAND            = (0.40, 0.60)
SIGMA_TARGET    = 0.0125
MAX_SHOTS       = 1025

# Throughput
RUN_BATCH       = 64
KERNEL_CHUNK    = 512

# Output
OUTDIR          = "./kernels_noisy_aer_brisbane"
os.makedirs(OUTDIR, exist_ok=True)

# QSVM feature map
nq = 7
fm_z = z_feature_map(feature_dimension=nq, reps=1, entanglement="linear", parameter_prefix='z')
fm_x = z_feature_map(feature_dimension=nq, reps=1, entanglement="linear", parameter_prefix='x')

# Trimmed C grid (drop too-small Cs)
C_GRID = np.logspace(-1.5, 2, 8)  # ~[0.0316, 0.1, ..., 100]

# Flags
USE_DD = True           # robust DD (XY4 if 'y' timed; else CPMG/XX)
USE_M3 = True           # M3 readout mitigation (optional; will auto-disable if unavailable)

np.random.seed(RANDOM_SEED)

# ------------------------------- DATA (RFE-7, N=96) ------------------------
X_df, y_sr = load_breast_cancer(return_X_y=True, as_frame=True)
X_full = X_df.to_numpy()
y_full = (y_sr.values == 1).astype(int)

sss = StratifiedShuffleSplit(n_splits=1, train_size=N, random_state=RANDOM_SEED)
(train_idx, _), = sss.split(X_full, y_full)

scaler_all = StandardScaler().fit(X_full)
X_full_scaled = scaler_all.transform(X_full)

logreg = LogisticRegression(max_iter=500, solver="lbfgs", random_state=RANDOM_SEED)
rfe = RFE(logreg, n_features_to_select=nq).fit(X_full_scaled, y_full)
rfe_mask = rfe.support_
rfe_names = list(X_df.columns[rfe_mask])
X_rfe_all = X_full_scaled[:, rfe_mask]
X = X_rfe_all[train_idx]
y = y_full[train_idx]

print(f"[INFO] Using N={N} samples. Class balance: {Counter(int(v) for v in y)}")
print(f"[INFO] RFE-7 features: {rfe_names}")

# ----------------- LIVE CONNECTION + SNAPSHOT (retry) ----------------------
def clone_backend_with_retry(name, tries=5, save_dir=OUTDIR):
    service = QiskitRuntimeService()
    try:
        bk = service.backend(name)
        st = bk.status()
        print(f"[CONNECTED] backend={bk.name} | operational={st.operational} | pending_jobs={st.pending_jobs} | status='{st.status_msg}'")
    except Exception as e:
        print(f"[WARN] Could not query backend status for '{name}': {e}")

    last_err = None
    for a in range(1, tries+1):
        try:
            backend = service.backend(name)
            sim = AerSimulator.from_backend(backend)
            sim.set_options(seed_simulator=RANDOM_SEED)
            # save properties snapshot (string repr; V2 backends may not expose everything)
            ts = None
            try:
                props = backend.properties()
                try: ts = props.last_update_date
                except: ts = None
                with open(os.path.join(save_dir, "backend_properties.txt"), "w", encoding="utf-8") as f:
                    f.write(str(props))
            except Exception as e_props:
                # also try to dump target summary
                with open(os.path.join(save_dir, "backend_properties.txt"), "w", encoding="utf-8") as f:
                    f.write(f"[no properties available] {e_props}\nTarget summary:\n{backend.target}")
            print(f"[SNAPSHOT] Cloned from {name} on attempt {a}/{tries}" + (f" (properties timestamp={ts})" if ts else ""))
            return sim, backend
        except Exception as e:
            last_err = e
            print(f"[SNAPSHOT-RETRY {a}/{tries}] clone failed: {e}")
    print(f"[SNAPSHOT] Final fallback to plain AerSimulator due to error: {last_err}")
    sim = AerSimulator()
    sim.set_options(seed_simulator=RANDOM_SEED)
    return sim, None

aer_sim, real_backend = clone_backend_with_retry(BACKEND_NAME)

# conservative threading
try:
    import multiprocessing as mp
    aer_sim.set_options(
        max_parallel_experiments=1,
        max_parallel_shots=1,
        max_parallel_threads=min(4, mp.cpu_count()),
    )
except Exception:
    pass

# ----------------- (Optional) auto-pick best 7-qubit chain -----------------
def pick_best_chain(backend, k=7):
    try:
        try:
            cmap = backend.coupling_map
        except Exception:
            cmap = backend.target.build_coupling_map()
        edges = set(tuple(sorted(e)) for e in cmap)
        ro = {}
        e2q = {}
        try:
            targ = backend.target
            # readout errors
            for q in range(targ.num_qubits):
                try:
                    ro[q] = float(targ.get_instruction("measure").get_properties((q,)).error)
                except Exception:
                    ro[q] = 0.02
            # 2q gate errors
            for (a,b) in edges:
                try:
                    e2q[(a,b)] = float(targ.get_instruction("ecr").get_properties((a,b)).error)
                except Exception:
                    e2q[(a,b)] = 0.02
        except Exception:
            return None
        adj = {}
        for a,b in edges:
            adj.setdefault(a, set()).add(b)
            adj.setdefault(b, set()).add(a)
        best = None
        def dfs(path):
            nonlocal best
            if len(path)==k:
                score = sum(ro[q] for q in path) + sum(e2q[tuple(sorted((path[i], path[i+1])))] for i in range(k-1))
                if best is None or score < best[0]:
                    best = (score, path.copy())
                return
            for nb in adj[path[-1]]:
                if nb in path: continue
                path.append(nb); dfs(path); path.pop()
        for s in adj:
            dfs([s])
        return best[1] if best else None
    except Exception:
        return None

if USE_AUTO_LAYOUT and real_backend is not None:
    best_chain = pick_best_chain(real_backend, k=nq)
else:
    best_chain = None

if best_chain is None:
    layout = INITIAL_LAYOUT
    print(f"[LAYOUT] Auto-pick failed → using INITIAL_LAYOUT: {layout}")
else:
    layout = best_chain
    print(f"[LAYOUT] Selected best 7-qubit chain: {layout}")

# ------------------------------- TEMPLATE CIRCUIT --------------------------
qc = QuantumCircuit(nq, name="fid")
qc.compose(fm_z, qubits=range(nq), inplace=True)
qc.compose(fm_x.inverse(), qubits=range(nq), inplace=True)
cr = ClassicalRegister(nq, "m")
qc.add_register(cr)
qc.measure(range(nq), cr)

t0_build = time.perf_counter()
qc_t = transpile(
    qc,
    backend=aer_sim,                # IMPORTANT: pass only backend
    initial_layout=layout,          # final layout
    optimization_level=1,
)

# ----------------- DD scheduling with durations (robust) -------------------
if USE_DD:
    durations = None
    try:
        durations = InstructionDurations.from_backend(real_backend) if real_backend is not None else None
    except Exception:
        durations = None

    if durations is None:
        print("[DD] WARNING: Could not build InstructionDurations; DD disabled.")
    else:
        # Determine if 'y' is both native and timed; else fall back to CPMG (X,X)
        supports_y = False
        try:
            op_names = set(real_backend.target.operation_names) if real_backend is not None else set()
        except Exception:
            op_names = set()
        if "y" in op_names:
            try:
                _ = durations.get("y", (0,), unit="dt")  # will raise if not present
                supports_y = True
            except Exception:
                supports_y = False

        if supports_y:
            dd_seq = [XGate(), YGate(), XGate(), YGate()]  # XY4
            print("[DD] Using XY4 (X–Y–X–Y).")
        else:
            dd_seq = [XGate(), XGate()]  # CPMG; fully native on IBM backends
            print("[DD] 'y' not native/timed; using CPMG sequence (X–X).")

        pm = PassManager([
            ALAPScheduleAnalysis(durations=durations),
            PadDynamicalDecoupling(durations=durations, dd_sequence=dd_seq)
        ])
        qc_t = pm.run(qc_t)

t1_build = time.perf_counter()
print(f"[INFO] Prepared circuit (transpile{' + DD' if USE_DD else ''}). Time: {t1_build - t0_build:.2f}s")

# Parameter handles
def extract_params_by_prefix(circ, prefix: str):
    plist = [p for p in circ.parameters if p.name.startswith(f"{prefix}[")]
    plist.sort(key=lambda p: int(p.name[p.name.find('[')+1:-1]))
    return plist
params_x = extract_params_by_prefix(qc_t, "x")
params_z = extract_params_by_prefix(qc_t, "z")

# ------------------------------- M3 mitigation -----------------------------
M3_READY = False
MEAS_QUBITS = layout  # physical order we measure on hardware
if USE_M3 and _M3_AVAILABLE and real_backend is not None:
    try:
        mit = m3.M3Mitigation(real_backend)
        mit.cals_from_system(MEAS_QUBITS)
        M3_READY = True
        print(f"[M3] Calibrated on qubits: {MEAS_QUBITS}")
    except Exception as e:
        print(f"[M3] Disabled (setup failed): {e}")
        M3_READY = False
else:
    if USE_M3 and not _M3_AVAILABLE:
        print("[M3] Package 'mthree' not installed; mitigation disabled.")

# ------------------------------- HELPERS -----------------------------------
STATS = {"adapted_pairs": 0, "total_pairs": 0, "max_shots_seen": BASE_SHOTS}

def counts_to_p0(counts_dict):
    shots_used = int(sum(counts_dict.values()))
    return counts_dict.get("0"*nq, 0) / max(1, shots_used), shots_used

def mitigate_p0(counts_dict):
    """Return (p0_mitigated, shots). Falls back to raw counts if M3 not ready."""
    p_raw, s = counts_to_p0(counts_dict)
    if not M3_READY:
        return p_raw, s
    try:
        q = mit.apply_correction(counts_dict, MEAS_QUBITS)
        if isinstance(q, dict):
            p0 = float(q.get("0"*nq, p_raw))
        else:
            try:
                p0 = float(q[0].get("0"*nq, p_raw))
            except Exception:
                p0 = p_raw
        return p0, s
    except Exception as e:
        print(f"[M3] apply_correction failed once: {e} → using raw counts.")
        return p_raw, s

def pub_for_pair(Arow, Brow):
    pmap = {}
    for k in range(nq):
        pmap[params_z[k]] = float(Brow[k])
        pmap[params_x[k]] = float(Arow[k])
    return (qc_t, pmap)

def run_counts_pubs(pubs, shots, batch=RUN_BATCH, seed=1234):
    out = []
    for start in range(0, len(pubs), batch):
        sub = pubs[start:start+batch]
        bound = [circ.assign_parameters(pmap, inplace=False) for (circ, pmap) in sub]
        job = aer_sim.run(bound, shots=shots, seed_simulator=seed)
        res = job.result()
        counts = res.get_counts()
        out.extend(counts if isinstance(counts, list) else [counts])
    return out

def needed_shots_for_sigma(p_hat, sigma):
    p = max(1e-6, min(1-1e-6, p_hat))
    return int(math.ceil(p*(1-p)/(sigma*sigma)))

def p0s_with_adaptive(pubs, base_shots=BASE_SHOTS, band=BAND, sigma=SIGMA_TARGET, max_shots=MAX_SHOTS, seed=1234):
    counts1 = run_counts_pubs(pubs, shots=base_shots, seed=seed)
    p0 = np.zeros(len(pubs), dtype=float)
    shots_tot = np.zeros(len(pubs), dtype=int)
    for i, c in enumerate(counts1):
        p, s = mitigate_p0(c)
        p0[i] = p; shots_tot[i] = s

    lo, hi = band
    need = []
    for i, p in enumerate(p0):
        if lo <= p <= hi:
            req = needed_shots_for_sigma(p, sigma)
            add = max(0, min(max_shots - shots_tot[i], req - shots_tot[i]))
            if add > 0:
                need.append((i, add))

    if need:
        from collections import defaultdict
        groups = defaultdict(list)
        for idx, add in need: groups[add].append(idx)
        for add, idxs in groups.items():
            add_counts = run_counts_pubs([pubs[i] for i in idxs], shots=add, seed=seed+1)
            for j, c2 in enumerate(add_counts):
                i = idxs[j]
                p_add, s_add = mitigate_p0(c2)
                s1 = shots_tot[i]; p1 = p0[i]
                s_sum = s1 + s_add
                p_comb = (p1*s1 + p_add*s_add) / max(1, s_sum)
                p0[i] = p_comb; shots_tot[i] = s_sum

    STATS["total_pairs"] += len(pubs)
    STATS["adapted_pairs"] += int(np.sum(shots_tot > base_shots))
    STATS["max_shots_seen"] = max(STATS["max_shots_seen"], int(shots_tot.max()))
    return [(float(p0[i]), int(shots_tot[i])) for i in range(len(pubs))]

def p0s(pubs):
    return p0s_with_adaptive(pubs) if USE_ADAPTIVE else [
        (mitigate_p0(c))[0] for c in run_counts_pubs(pubs, shots=BASE_SHOTS)
    ]

def kernel_square_upper(A, chunk=KERNEL_CHUNK, details=None):
    n = A.shape[0]
    K_raw = np.zeros((n, n), dtype=float)
    idx_pairs, pubs = [], []
    for i in range(n):
        for j in range(i, n):
            idx_pairs.append((i, j))
            pubs.append(pub_for_pair(A[i], A[j]))
    for start in range(0, len(pubs), chunk):
        sub = pubs[start:start+chunk]
        vals = p0s(sub)
        for (p0_val, s), (i, j) in zip(vals, idx_pairs[start:start+chunk]):
            K_raw[i, j] = p0_val;  K_raw[j, i] = p0_val
            if details is not None:
                details.append({"block":"train", "i":int(i), "j":int(j), "p0":float(p0_val), "shots_total":int(s)})
    return K_raw

def kernel_cross(A, B, chunk=KERNEL_CHUNK, details=None):
    K_raw = np.zeros((A.shape[0], B.shape[0]), dtype=float)
    pubs = [pub_for_pair(A[i], B[j]) for i in range(A.shape[0]) for j in range(B.shape[0])]
    idx = 0
    for start in range(0, len(pubs), chunk):
        sub = pubs[start:start+chunk]
        vals = p0s(sub)
        for (p0_val, s) in vals:
            i = idx // B.shape[0]; j = idx % B.shape[0]
            K_raw[i, j] = p0_val
            if details is not None:
                details.append({"block":"testxtrain", "i":int(i), "j":int(j), "p0":float(p0_val), "shots_total":int(s)})
            idx += 1
    return K_raw

def kernel_diag(A, chunk=KERNEL_CHUNK, details=None):
    pubs = [pub_for_pair(A[i], A[i]) for i in range(A.shape[0])]
    d = np.zeros(A.shape[0], dtype=float)
    for start in range(0, len(pubs), chunk):
        sub = pubs[start:start+chunk]
        vals = p0s(sub)
        for offs, (p0_val, s) in enumerate(vals):
            d[start + offs] = p0_val
            if details is not None:
                details.append({"block":"diag", "i":int(start+offs), "j":int(start+offs), "p0":float(p0_val), "shots_total":int(s)})
    return d

def normalize_square_from_diag(K_raw, d):
    d = np.clip(d, 1e-12, None)
    Kn = K_raw / np.sqrt(np.outer(d, d))
    Kn = 0.5 * (Kn + Kn.T)
    np.fill_diagonal(Kn, 1.0)
    return Kn

def normalize_cross_from_diag(K_raw, d_test, d_train):
    d_test  = np.clip(d_test,  1e-12, None)
    d_train = np.clip(d_train, 1e-12, None)
    return K_raw / np.sqrt(np.outer(d_test, d_train))

# Centering
def center_train_kernel(K):
    r = K.mean(axis=1, keepdims=True); c = K.mean(axis=0, keepdims=True); g = K.mean()
    return K - r - c + g
def center_test_kernel(K_te, K_tr):
    c_tr = K_tr.mean(axis=0, keepdims=True); g_tr = K_tr.mean(); r_te = K_te.mean(axis=1, keepdims=True)
    return K_te - c_tr - r_te + g_tr

# Inner CV over C
def pick_best_C(K_tr, y_tr, Cs=C_GRID, seed=RANDOM_SEED):
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=seed)
    best_auc, best_C = -np.inf, Cs[0]
    n = len(y_tr)
    for C in Cs:
        aucs = []
        for tr2, va2 in skf.split(np.arange(n), y_tr):
            K_tt = K_tr[np.ix_(tr2, tr2)]
            K_tv = K_tr[np.ix_(va2, tr2)]
            clf2 = SVC(kernel="precomputed", C=C, class_weight="balanced")
            clf2.fit(K_tt, y_tr[tr2])
            scores2 = clf2.decision_function(K_tv)
            aucs.append(roc_auc_score(y_tr[va2], scores2))
        m = float(np.mean(aucs))
        if m > best_auc: best_auc, best_C = m, C
    return best_C, best_auc

# Youden threshold from validation ROC
def youden_threshold(scores, y_true):
    from sklearn.metrics import roc_curve
    fpr, tpr, thr = roc_curve(y_true, scores)
    j = tpr - fpr
    i = int(np.argmax(j))
    return float(thr[i])

# ------------------------------- CV RUNNER ---------------------------------
print(f"[INFO] AER run | Adaptive={'ON' if USE_ADAPTIVE else 'OFF'} | Base shots={BASE_SHOTS} | RUN_BATCH={RUN_BATCH} | KERNEL_CHUNK={KERNEL_CHUNK}")
print(f"[INFO] C-grid (trimmed): {C_GRID}")

cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_SEED)
fold_aucs, fold_accs, fold_f1s = [], [], []
fold_sens, fold_spec, fold_prec, fold_rec = [], [], [], []
thr_list = []
agg_tn = agg_fp = agg_fn = agg_tp = 0
fold_ksecs, fold_tsecs = [], []

for fold_id, (tr_idx, te_idx) in enumerate(cv.split(X, y), 1):
    X_tr, X_te = X[tr_idx], X[te_idx]
    y_tr, y_te = y[tr_idx], y[te_idx]

    DETAILS = None
    t0 = time.perf_counter()

    # Train square -> diag -> normalize -> center
    t0k = time.perf_counter()
    K_tr_raw = kernel_square_upper(X_tr, details=DETAILS)
    d_tr     = np.diag(K_tr_raw).copy()
    K_tr     = normalize_square_from_diag(K_tr_raw, d_tr)
    K_tr     = center_train_kernel(K_tr)

    # Test diag + cross -> normalize -> center (using train stats)
    d_te     = kernel_diag(X_te, details=DETAILS)
    K_te_raw = kernel_cross(X_te, X_tr, details=DETAILS)
    K_te     = normalize_cross_from_diag(K_te_raw, d_te, d_tr)
    K_te     = center_test_kernel(K_te, K_tr)
    t1k = time.perf_counter()

    # save kernels
    pd.DataFrame(K_tr).to_csv(os.path.join(OUTDIR, f"RFE7_K_train_f{fold_id}.csv"), index=False)
    pd.DataFrame(K_te).to_csv(os.path.join(OUTDIR, f"RFE7_K_test_f{fold_id}.csv"), index=False)

    # inner-CV pick
    C_best, _ = pick_best_C(K_tr, y_tr)

    # Train final SVC
    clf = SVC(kernel="precomputed", C=C_best, class_weight="balanced")
    clf.fit(K_tr, y_tr)

    # Calibrate threshold (Youden) using TRAIN scores
    train_scores = clf.decision_function(K_tr)
    thr_star = youden_threshold(train_scores, y_tr)
    thr_list.append(thr_star)

    scores = clf.decision_function(K_te)
    y_pred = (scores >= thr_star).astype(int)

    # Metrics
    auc = roc_auc_score(y_te, scores)
    acc = accuracy_score(y_te, y_pred)
    f1  = f1_score(y_te, y_pred, average="binary", pos_label=1)
    tn, fp, fn, tp = confusion_matrix(y_te, y_pred, labels=[0,1]).ravel()
    sens = tp / max(1, tp+fn)
    spec = tn / max(1, tn+fp)
    prec = tp / max(1, tp+fp)
    rec  = sens

    t1 = time.perf_counter()

    fold_aucs.append(auc); fold_accs.append(acc); fold_f1s.append(f1)
    fold_sens.append(sens); fold_spec.append(spec); fold_prec.append(prec); fold_rec.append(rec)
    agg_tn += tn; agg_fp += fp; agg_fn += fn; agg_tp += tp
    fold_ksecs.append(t1k - t0k); fold_tsecs.append(t1 - t0)

    print(f"  fold {fold_id}: AUC={auc:.4f}  Acc={acc:.4f}  F1={f1:.4f}  "
          f"TP={tp} FP={fp} TN={tn} FN={fn}  "
          f"Sens={sens:.4f} Spec={spec:.4f} Prec={prec:.4f} Rec={rec:.4f}  "
          f"| C*={C_best:g} | thr*={thr_star:.6f} | kernel={fold_ksecs[-1]:.2f}s | total={fold_tsecs[-1]:.2f}s")

# Summary
mean_auc = float(np.mean(fold_aucs)); std_auc = float(np.std(fold_aucs))
mean_acc = float(np.mean(fold_accs)); std_acc = float(np.std(fold_accs))
mean_f1  = float(np.mean(fold_f1s));  std_f1  = float(np.std(fold_f1s))
mean_sens = float(np.mean(fold_sens)); std_sens = float(np.std(fold_sens))
mean_spec = float(np.mean(fold_spec)); std_spec = float(np.std(fold_spec))
mean_prec = float(np.mean(fold_prec)); std_prec = float(np.std(fold_prec))
mean_rec  = float(np.mean(fold_rec));  std_rec  = float(np.std(fold_rec))
mean_thr  = float(np.mean(thr_list));  std_thr  = float(np.std(thr_list))
mean_k   = float(np.mean(fold_ksecs)); mean_t = float(np.mean(fold_tsecs))

print("\n=== Summary: RFE-7 (centered, balanced, trimmed C, calibrated thr, DD, M3-if-available) ===")
print(f"Mean AUC={mean_auc:.4f} (+/- {std_auc:.4f})  "
      f"Mean Acc={mean_acc:.4f} (+/- {std_acc:.4f})  "
      f"Mean F1={mean_f1:.4f} (+/- {std_f1:.4f})")
print(f"Mean Sens={mean_sens:.4f} (+/- {std_sens:.4f})  "
      f"Mean Spec={mean_spec:.4f} (+/- {std_spec:.4f})  "
      f"Mean Prec={mean_prec:.4f} (+/- {std_prec:.4f})  "
      f"Mean Rec={mean_rec:.4f} (+/- {std_rec:.4f})")
print(f"Threshold (Youden) mean={mean_thr:.6f} (+/- {std_thr:.6f})")
print(f"Totals: TP={agg_tp}  FP={agg_fp}  TN={agg_tn}  FN={agg_fn}")
print(f"Timing: mean kernel/fold={mean_k:.2f}s  |  mean total/fold={mean_t:.2f}s")

print(f"\n[ADAPTIVE] pairs={STATS['total_pairs']}  adapted={STATS['adapted_pairs']}  "
      f"({100.0*STATS['adapted_pairs']/max(1,STATS['total_pairs']):.1f}%)  "
      f"max_shots_seen={STATS['max_shots_seen']}")

print("\n=== QPU-READY CHECKLIST ===")
print(f"Snapshot cloned from '{BACKEND_NAME}': {real_backend is not None}")
if real_backend is not None:
    st = real_backend.status()
    print(f"Backend status: {{'name': '{real_backend.name}', 'operational': {st.operational}, 'pending_jobs': {st.pending_jobs}, 'status_msg': '{st.status_msg}'}}")
print(f"Properties saved to: {os.path.abspath(os.path.join(OUTDIR, 'backend_properties.txt'))}")
print(f"FINAL INITIAL_LAYOUT (7 qubits): {layout}")
print(f"Centering: ON  | class_weight: balanced  | C-grid: {C_GRID[0]}..{C_GRID[-1]}  | threshold: Youden per fold")
print(f"Shots: base={BASE_SHOTS}  adaptive up to={MAX_SHOTS} in band={BAND}")
print(f"DD: {'ON' if USE_DD else 'OFF'}  | M3: {'ON (calibrated)' if ('M3_READY' in globals() and M3_READY) else 'OFF'}")
print(f"[SAVED] kernels in: {os.path.abspath(OUTDIR)}")


In [ ]:
# === QSVM fidelity kernel on REAL QPU (Sampler V2, no Session) =============
# - RFE-7 features, N=96, 3-fold CV
# - SamplerV2 bound directly to your instance backend (no Session)
# - ASAP scheduling + DD (XY4 if Y timed; else CPMG/XX)
# - Adaptive shots (base=256; top up to 1025 inside band)
# - Optional M3 readout mitigation (if mthree installed)
# - Kernel centering + class_weight="balanced" + trimmed C grid
# - Full persistence (per-entry CSV, kernels, summaries) + robust RESUME
#
# Compatible with:
#   qiskit ~= 2.2.x
#   qiskit-ibm-runtime ~= 0.42.x  (V2 primitives)
#
# IBM docs showing the V2 usage pattern (SamplerV2 + get_counts):
# https://quantum.cloud.ibm.com/docs/en/guides/visualize-results

import os, json, time, math, numpy as np, pandas as pd
from collections import Counter, defaultdict
from datetime import datetime

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import RFE
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, confusion_matrix, roc_curve

from qiskit import QuantumCircuit, ClassicalRegister, transpile
from qiskit.circuit.library import z_feature_map, XGate, YGate
from qiskit.transpiler import PassManager, InstructionDurations
from qiskit.transpiler.passes import ASAPScheduleAnalysis, PadDynamicalDecoupling

# IMPORTANT: use SamplerV2 explicitly and pass backend POSITIONALLY
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler

# Optional M3 mitigation
try:
    import mthree as m3
    _M3_AVAILABLE = True
except Exception:
    _M3_AVAILABLE = False

# =============================== USER CONFIG ================================
BACKEND_NAME    = "ibm_brisbane"
USE_AUTO_LAYOUT = True
INITIAL_LAYOUT  = [3, 4, 5, 6, 7, 8, 16]

N               = 96
N_SPLITS        = 3
RANDOM_SEED     = 42
NQ              = 7

# Shots / adaptive
BASE_SHOTS      = 256
USE_ADAPTIVE    = True
BAND            = (0.40, 0.60)
SIGMA_TARGET    = 0.0125
MAX_SHOTS       = 1025

# Throughput
RUN_BATCH       = 64
KERNEL_CHUNK    = 512

# SVM grid (trimmed)
C_GRID          = np.logspace(-1.5, 2, 8)   # 0.0316 .. 100

# DD / M3
USE_DD          = True
USE_M3          = True

# Resume / Output
RESUME          = True
OUTDIR          = r"D:\PHYSIQUE MATHEMATIQUE\doctora\4 EME ANNE\articles\breast canser\implementations\qsvm_on_wcds\real QPU\SOURUN"
os.makedirs(OUTDIR, exist_ok=True)

ENTRY_LOG_CSV   = os.path.join(OUTDIR, "per_entry_log.csv")
CHECKPOINT_JSON = os.path.join(OUTDIR, "checkpoint.json")
BACKEND_TXT     = os.path.join(OUTDIR, "backend_properties.txt")
REPORTS_DIR     = os.path.join(OUTDIR, "reports")
os.makedirs(REPORTS_DIR, exist_ok=True)

np.random.seed(RANDOM_SEED)

# =============================== UTILITIES =================================
ENTRY_HEADER = ["timestamp","fold","block","i","j","p0_raw","p0_mitigated",
                "shots_base","shots_added","shots_total","elapsed_s","batch_id","job_id"]

def ensure_entry_log_exists():
    """Create empty CSV with header if it doesn't exist (avoids pandas header error)."""
    if not os.path.exists(ENTRY_LOG_CSV):
        pd.DataFrame(columns=ENTRY_HEADER).to_csv(ENTRY_LOG_CSV, index=False)

def append_csv_rows(rows):
    """Append rows to per_entry_log.csv; assumes file exists with header."""
    if not rows:
        return
    df = pd.DataFrame(rows)[ENTRY_HEADER]
    df.to_csv(ENTRY_LOG_CSV, index=False, mode="a", header=False)

def atomic_write_json(path, obj):
    tmp = f"{path}.tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)
    os.replace(tmp, path)

def now_str():
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

def youden_threshold(scores, y_true):
    fpr, tpr, thr = roc_curve(y_true, scores)
    j = tpr - fpr
    i = int(np.argmax(j))
    return float(thr[i])

# ========================== CONNECT BACKEND / SNAPSHOT =====================
service = QiskitRuntimeService()  # uses your saved IBM Quantum account/instance
backend = service.backend(BACKEND_NAME)
st = backend.status()
print(f"[CONNECTED] backend={backend.name} | operational={st.operational} | pending_jobs={st.pending_jobs} | status='{st.status_msg}'")

# Save properties snapshot
try:
    props = backend.properties()
    with open(BACKEND_TXT, "w", encoding="utf-8") as f:
        f.write(str(props))
except Exception as e_props:
    with open(BACKEND_TXT, "w", encoding="utf-8") as f:
        f.write(f"[no properties available] {e_props}\nTarget summary:\n{getattr(backend, 'target', 'N/A')}")

# ============================== DATA (RFE-7) ===============================
X_df, y_sr = load_breast_cancer(return_X_y=True, as_frame=True)
X_full = X_df.to_numpy()
y_full = (y_sr.values == 1).astype(int)

sss = StratifiedShuffleSplit(n_splits=1, train_size=N, random_state=RANDOM_SEED)
(train_idx, _), = sss.split(X_full, y_full)

scaler_all = StandardScaler().fit(X_full)
X_full_scaled = scaler_all.transform(X_full)

logreg = LogisticRegression(max_iter=500, solver="lbfgs", random_state=RANDOM_SEED)
rfe = RFE(logreg, n_features_to_select=NQ).fit(X_full_scaled, y_full)
rfe_mask = rfe.support_
rfe_names = list(X_df.columns[rfe_mask])
X = X_full_scaled[:, rfe_mask][train_idx]
y = y_full[train_idx]

print(f"[INFO] Using N={N} samples. Class balance: {Counter(int(v) for v in y)}")
print(f"[INFO] RFE-7 features: {rfe_names}")

# ========================== LAYOUT (best 7-qubit chain) ====================
def pick_best_chain(backend, k=7):
    try:
        try:
            cmap = backend.coupling_map
        except Exception:
            cmap = backend.target.build_coupling_map()
        edges = set(tuple(sorted(e)) for e in cmap)
        ro = {}
        e2q = {}
        targ = backend.target
        # readout errors
        for q in range(targ.num_qubits):
            try:
                ro[q] = float(targ.get_instruction("measure").get_properties((q,)).error)
            except Exception:
                ro[q] = 0.02
        # 2q errors (prefer ECR else CX)
        twoq_name = "ecr" if "ecr" in targ.operation_names else ("cx" if "cx" in targ.operation_names else None)
        for (a,b) in edges:
            try:
                e2q[(a,b)] = float(targ.get_instruction(twoq_name).get_properties((a,b)).error) if twoq_name else 0.02
            except Exception:
                e2q[(a,b)] = 0.02
        # brute force DFS over simple paths
        adj = {}
        for a,b in edges:
            adj.setdefault(a, set()).add(b)
            adj.setdefault(b, set()).add(a)
        best = None
        def dfs(path):
            nonlocal best
            if len(path)==k:
                score = sum(ro[q] for q in path) + sum(e2q[tuple(sorted((path[i], path[i+1])))] for i in range(k-1))
                if best is None or score < best[0]:
                    best = (score, path.copy())
                return
            for nb in adj[path[-1]]:
                if nb in path: continue
                path.append(nb); dfs(path); path.pop()
        for s in adj:
            dfs([s])
        return best[1] if best else None
    except Exception:
        return None

if USE_AUTO_LAYOUT:
    best_chain = pick_best_chain(backend, k=NQ)
else:
    best_chain = None

if best_chain is None:
    layout = INITIAL_LAYOUT
    print(f"[LAYOUT] Auto-pick failed → using INITIAL_LAYOUT: {layout}")
else:
    layout = best_chain
    print(f"[LAYOUT] Selected best 7-qubit chain: {layout}")

# ============================ TEMPLATE CIRCUIT =============================
fm_z = z_feature_map(feature_dimension=NQ, reps=1, entanglement="linear", parameter_prefix='z')
fm_x = z_feature_map(feature_dimension=NQ, reps=1, entanglement="linear", parameter_prefix='x')

qc = QuantumCircuit(NQ, name="fid")
qc.compose(fm_z, qubits=range(NQ), inplace=True)
qc.compose(fm_x.inverse(), qubits=range(NQ), inplace=True)
cr = ClassicalRegister(NQ, "m")
qc.add_register(cr)
qc.measure(range(NQ), cr)

t0_build = time.perf_counter()
qc_t = transpile(
    qc,
    backend=backend,                # real backend for mapping
    initial_layout=layout,
    optimization_level=1,
)

# DD after ASAP scheduling
supports_y = False
if USE_DD:
    try:
        durations = InstructionDurations.from_backend(backend)
    except Exception:
        durations = None
    if durations is None:
        print("[DD] WARNING: Could not build InstructionDurations; DD disabled.")
    else:
        try:
            _ = durations.get("y", (0,), unit="dt")
            supports_y = True
        except Exception:
            supports_y = False
        dd_seq = [XGate(), YGate(), XGate(), YGate()] if supports_y else [XGate(), XGate()]
        print(f"[DD] Using {'XY4 (X,Y,X,Y)' if supports_y else 'CPMG (X,X)'}.")

        pm = PassManager([
            ASAPScheduleAnalysis(durations=durations),
            PadDynamicalDecoupling(durations=durations, dd_sequence=dd_seq)
        ])
        qc_t = pm.run(qc_t)

t1_build = time.perf_counter()

# Circuit report
def _backend_dt(bk):
    try:
        if getattr(bk, "dt", None) is not None:
            return bk.dt
    except Exception:
        pass
    try:
        return bk.target.dt
    except Exception:
        return None

dt = _backend_dt(backend)
duration_dt = getattr(qc_t, "duration", None)  # deprecated but available on scheduled circuits
duration_us = (duration_dt * dt * 1e6) if (duration_dt is not None and dt is not None) else None
ops = qc_t.count_ops()
report = {
    "depth": int(qc_t.depth()),
    "ops": {k: int(v) for k, v in ops.items()},
    "duration_dt": int(duration_dt) if duration_dt is not None else None,
    "dt_seconds": float(dt) if dt is not None else None,
    "duration_us": float(duration_us) if duration_us is not None else None,
    "layout": list(layout),
    "dd_sequence": "XY4" if supports_y else "CPMG/XX",
    "scheduling": "ASAP",
    "optimization_level": 1,
}
with open(os.path.join(REPORTS_DIR, "circuit_report.json"), "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)
print(f"[INFO] Prepared circuit (transpile + {'DD' if USE_DD else 'no DD'}). Time: {t1_build - t0_build:.2f}s")

# Parameter order (must match transpiled circuit)
param_order = list(qc_t.parameters)
def extract_params_by_prefix(circ, prefix: str):
    plist = [p for p in circ.parameters if p.name.startswith(f"{prefix}[")]
    plist.sort(key=lambda p: int(p.name[p.name.find('[')+1:-1]))
    return plist
params_x = extract_params_by_prefix(qc_t, "x")
params_z = extract_params_by_prefix(qc_t, "z")

def param_values_for_pair(Arow, Brow):
    mp = {}
    for k in range(NQ):
        mp[params_z[k]] = float(Brow[k])
        mp[params_x[k]] = float(Arow[k])
    return [mp[p] for p in param_order]

# Which register name to read from SamplerV2 results
REG_NAME = qc_t.cregs[0].name if qc_t.cregs else "meas"

# =========================== M3 READOUT MITIGATION =========================
M3_READY = False
MEAS_QUBITS = layout
if USE_M3 and _M3_AVAILABLE:
    try:
        mit = m3.M3Mitigation(backend)
        mit.cals_from_system(MEAS_QUBITS)
        M3_READY = True
        print(f"[M3] Calibrated on qubits: {MEAS_QUBITS}")
    except Exception as e:
        print(f"[M3] Disabled (setup failed): {e}")
        M3_READY = False
else:
    if USE_M3 and not _M3_AVAILABLE:
        print("[M3] Package 'mthree' not installed; mitigation disabled.")

def counts_to_p0(counts_dict):
    shots = int(sum(counts_dict.values()))
    p0 = counts_dict.get("0"*NQ, 0) / max(1, shots)
    return p0, shots

def mitigate_p0(counts_dict):
    """Return (p_raw, p_mitigated, shots). Falls back to raw if M3 not ready."""
    p_raw, s = counts_to_p0(counts_dict)
    if not M3_READY:
        return p_raw, p_raw, s
    try:
        q = mit.apply_correction(counts_dict, MEAS_QUBITS)
        if isinstance(q, dict):
            p0_mit = float(q.get("0"*NQ, p_raw))
        else:
            try:
                p0_mit = float(q[0].get("0"*NQ, p_raw))
            except Exception:
                p0_mit = p_raw
        return p_raw, p0_mit, s
    except Exception as e:
        print(f"[M3] apply_correction failed once: {e} → using raw counts.")
        return p_raw, p_raw, s

# ============================ RESUME STATE =================================
ensure_entry_log_exists()
STATS = {"adapted_pairs": 0, "total_pairs": 0, "max_shots_seen": BASE_SHOTS}

if RESUME and os.path.exists(CHECKPOINT_JSON):
    with open(CHECKPOINT_JSON, "r", encoding="utf-8") as f:
        CKPT = json.load(f)
    print(f"[RESUME] Resuming from {OUTDIR}")
else:
    CKPT = {"fold": 1, "done": []}
    atomic_write_json(CHECKPOINT_JSON, CKPT)

def mark_done(fold, block, i, j=None):
    key = f"fold{fold}-{block}-{i}" if j is None else f"fold{fold}-{block}-{i}-{j}"
    CKPT["done"].append(key)

def is_done(fold, block, i, j=None):
    key = f"fold{fold}-{block}-{i}" if j is None else f"fold{fold}-{block}-{i}-{j}"
    return key in CKPT["done"]

def save_ckpt(fold=None):
    if fold is not None:
        CKPT["fold"] = int(fold)
    atomic_write_json(CHECKPOINT_JSON, CKPT)

# ============================ SAMPLER V2 (NO SESSION) ======================
# NOTE: per IBM docs you can pass the backend POSITIONALLY to SamplerV2.
# We'll submit PUB tuples: (circuit, parameter_values, shots) so we can vary shots.
sampler = Sampler(backend)

def run_pub_batch(paramvals, shots_list):
    """
    Run a batch with possibly different shots per entry.
    paramvals: list of parameter lists (aligned to param_order)
    shots_list: list of int, same length
    Returns (counts_list, job_id, elapsed)
    """
    pubs = [(qc_t, paramvals[k], int(shots_list[k])) for k in range(len(paramvals))]
    t0 = time.time()
    job = sampler.run(pubs)
    try:
        job_id = job.job_id()
    except Exception:
        job_id = ""
    result = job.result()
    elapsed = time.time() - t0

    # V2 result API: result[i].data.<reg>.get_counts()
    counts_list = []
    for i in range(len(paramvals)):
        pub_res = result[i]
        data = getattr(pub_res, "data", None)
        # reg may be "m" (our register) or default "meas"
        reg_obj = getattr(data, REG_NAME, None) if data is not None else None
        if reg_obj is None:
            reg_obj = getattr(data, "meas", None)
        counts = reg_obj.get_counts() if reg_obj is not None else {}
        counts_list.append(counts)
    return counts_list, job_id, elapsed

# ============================ KERNEL HELPERS ================================
def needed_shots_for_sigma(p_hat, sigma):
    p = max(1e-6, min(1-1e-6, float(p_hat)))
    return int(math.ceil(p*(1-p)/(sigma*sigma)))

def process_batch_paramvals(paramvals, indices, fold, block, base_shots, batch_id):
    """
    paramvals: list of parameter lists (aligned to param_order), length=M
    indices  : list of (i,j) pairs matching paramvals
    Logs per-entry rows; returns dict[(i,j)]=(p_mitigated, shots_total)
    """
    label = f"fold{fold}-{block}-batch{batch_id}"

    # base pass (uniform shots)
    counts_list, job_id, _ = run_pub_batch(paramvals, [base_shots]*len(paramvals))

    p_mit = np.zeros(len(paramvals), float)
    p_raw = np.zeros(len(paramvals), float)
    shots_tot = np.zeros(len(paramvals), int)
    for k, counts in enumerate(counts_list):
        pr, pm, s = mitigate_p0(counts)
        p_mit[k] = pm; p_raw[k] = pr; shots_tot[k] = s

    # adaptive top-up
    added = np.zeros(len(paramvals), int)
    lo, hi = BAND
    need = []
    if USE_ADAPTIVE:
        for k, pm in enumerate(p_mit):
            if lo <= pm <= hi:
                req = needed_shots_for_sigma(pm, SIGMA_TARGET)
                add = max(0, min(MAX_SHOTS - shots_tot[k], req - shots_tot[k]))
                if add > 0: need.append((k, add))

    if need:
        groups = defaultdict(list)
        for k, add in need:
            groups[add].append(k)
        for add_shots, idxs in groups.items():
            sub_vals = [paramvals[u] for u in idxs]
            sub_counts, job2_id, _ = run_pub_batch(sub_vals, [add_shots]*len(sub_vals))
            for t, c2 in enumerate(sub_counts):
                u = idxs[t]
                pr2, pm2, s2 = mitigate_p0(c2)
                s1 = shots_tot[u]; pm1 = p_mit[u]; pr1 = p_raw[u]
                s_sum = s1 + s2
                pm_comb = (pm1*s1 + pm2*s2) / max(1, s_sum)
                pr_comb = (pr1*s1 + pr2*s2) / max(1, s_sum)
                p_mit[u] = pm_comb; p_raw[u] = pr_comb; shots_tot[u] = s_sum
                added[u] += s2

    # log rows
    rows = []
    for k, (i, j) in enumerate(indices):
        rows.append({
            "timestamp": now_str(), "fold": fold, "block": block,
            "i": int(i), "j": int(j),
            "p0_raw": float(p_raw[k]), "p0_mitigated": float(p_mit[k]),
            "shots_base": int(min(shots_tot[k], base_shots)),
            "shots_added": int(added[k]),
            "shots_total": int(shots_tot[k]),
            "elapsed_s": float(0.0),             # per-entry elapsed not meaningful with V2; keep 0
            "batch_id": int(batch_id),
            "job_id": str(job_id)
        })
    append_csv_rows(rows)

    # stats
    STATS["total_pairs"] += len(paramvals)
    STATS["adapted_pairs"] += int(np.sum(added > 0))
    STATS["max_shots_seen"] = max(STATS["max_shots_seen"], int(shots_tot.max() if len(shots_tot) else BASE_SHOTS))

    out = {(int(indices[k][0]), int(indices[k][1])): (float(p_mit[k]), int(shots_tot[k])) for k in range(len(indices))}
    return out

def kernel_train_square(A, fold):
    n = A.shape[0]
    K = np.zeros((n, n), float)
    pairs, pvals = [], []
    for i in range(n):
        for j in range(i, n):
            if is_done(fold, "train", i, j): continue
            pairs.append((i, j))
            pvals.append(param_values_for_pair(A[i], A[j]))
    batch_id = 0
    for start in range(0, len(pairs), KERNEL_CHUNK):
        sub_pairs = pairs[start:start+KERNEL_CHUNK]
        sub_vals  = pvals[start:start+KERNEL_CHUNK]
        out = {}
        for b in range(0, len(sub_pairs), RUN_BATCH):
            idxs = sub_pairs[b:b+RUN_BATCH]
            vals = sub_vals[b:b+RUN_BATCH]
            batch_id += 1
            res = process_batch_paramvals(vals, idxs, fold, "train", BASE_SHOTS, batch_id)
            out.update(res)
        for (i, j), (p, s) in out.items():
            K[i, j] = p; K[j, i] = p
            mark_done(fold, "train", i, j)
        save_ckpt(fold)
    # include resumed entries
    df = pd.read_csv(ENTRY_LOG_CSV)
    df = df[(df["fold"]==fold) & (df["block"]=="train")]
    for _, r in df.iterrows():
        i, j = int(r["i"]), int(r["j"])
        K[i, j] = float(r["p0_mitigated"]); K[j, i] = K[i, j]
    return K

def kernel_test_diag(A, fold):
    n = A.shape[0]
    d = np.zeros(n, float)
    pairs, pvals = [], []
    for i in range(n):
        if is_done(fold, "diag", i): continue
        pairs.append((i, i))
        pvals.append(param_values_for_pair(A[i], A[i]))
    batch_id = 0
    for start in range(0, len(pairs), KERNEL_CHUNK):
        sub_pairs = pairs[start:start+KERNEL_CHUNK]
        sub_vals  = pvals[start:start+KERNEL_CHUNK]
        out = {}
        for b in range(0, len(sub_pairs), RUN_BATCH):
            idxs = sub_pairs[b:b+RUN_BATCH]
            vals = sub_vals[b:b+RUN_BATCH]
            batch_id += 1
            res = process_batch_paramvals(vals, idxs, fold, "diag", BASE_SHOTS, batch_id)
            out.update(res)
        for (i, _), (p, s) in out.items():
            d[i] = p
            mark_done(fold, "diag", i)
        save_ckpt(fold)
    df = pd.read_csv(ENTRY_LOG_CSV)
    df = df[(df["fold"]==fold) & (df["block"]=="diag")]
    for _, r in df.iterrows():
        i = int(r["i"])
        d[i] = float(r["p0_mitigated"])
    return d

def kernel_test_cross(A, B, fold):
    nA, nB = A.shape[0], B.shape[0]
    K = np.zeros((nA, nB), float)
    pairs, pvals = [], []
    for i in range(nA):
        for j in range(nB):
            if is_done(fold, "test", i, j): continue
            pairs.append((i, j))
            pvals.append(param_values_for_pair(A[i], B[j]))
    batch_id = 0
    for start in range(0, len(pairs), KERNEL_CHUNK):
        sub_pairs = pairs[start:start+KERNEL_CHUNK]
        sub_vals  = pvals[start:start+KERNEL_CHUNK]
        out = {}
        for b in range(0, len(sub_pairs), RUN_BATCH):
            idxs = sub_pairs[b:b+RUN_BATCH]
            vals = sub_vals[b:b+RUN_BATCH]
            batch_id += 1
            res = process_batch_paramvals(vals, idxs, fold, "test", BASE_SHOTS, batch_id)
            out.update(res)
        for (i, j), (p, s) in out.items():
            K[i, j] = p
            mark_done(fold, "test", i, j)
        save_ckpt(fold)
    df = pd.read_csv(ENTRY_LOG_CSV)
    df = df[(df["fold"]==fold) & (df["block"]=="test")]
    for _, r in df.iterrows():
        i, j = int(r["i"]), int(r["j"])
        K[i, j] = float(r["p0_mitigated"])
    return K

def normalize_square_from_diag(K_raw, d):
    d = np.clip(d, 1e-12, None)
    Kn = K_raw / np.sqrt(np.outer(d, d))
    Kn = 0.5 * (Kn + Kn.T)
    np.fill_diagonal(Kn, 1.0)
    return Kn

def normalize_cross_from_diag(K_raw, d_test, d_train):
    d_test  = np.clip(d_test,  1e-12, None)
    d_train = np.clip(d_train, 1e-12, None)
    return K_raw / np.sqrt(np.outer(d_test, d_train))

def center_train_kernel(K):
    r = K.mean(axis=1, keepdims=True); c = K.mean(axis=0, keepdims=True); g = K.mean()
    return K - r - c + g

def center_test_kernel(K_te, K_tr):
    c_tr = K_tr.mean(axis=0, keepdims=True); g_tr = K_tr.mean(); r_te = K_te.mean(axis=1, keepdims=True)
    return K_te - c_tr - r_te + g_tr

def pick_best_C(K_tr, y_tr, Cs=C_GRID, seed=RANDOM_SEED):
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=seed)
    best_auc, best_C = -np.inf, Cs[0]
    n = len(y_tr)
    for C in Cs:
        aucs = []
        for tr2, va2 in skf.split(np.arange(n), y_tr):
            K_tt = K_tr[np.ix_(tr2, tr2)]
            K_tv = K_tr[np.ix_(va2, tr2)]
            clf2 = SVC(kernel="precomputed", C=C, class_weight="balanced")
            clf2.fit(K_tt, y_tr[tr2])
            scores2 = clf2.decision_function(K_tv)
            aucs.append(roc_auc_score(y_tr[va2], scores2))
        m = float(np.mean(aucs))
        if m > best_auc: best_auc, best_C = m, C
    return best_C, best_auc

# =============================== CV LOOP ===================================
print(f"[INFO] Real QPU run | Adaptive={'ON' if USE_ADAPTIVE else 'OFF'} | Base shots={BASE_SHOTS} | RUN_BATCH={RUN_BATCH} | KERNEL_CHUNK={KERNEL_CHUNK}")
print(f"[INFO] C-grid (trimmed): {C_GRID}")

cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_SEED)
fold_aucs, fold_accs, fold_f1s = [], [], []
fold_sens, fold_spec, fold_prec, fold_rec = [], [], [], []
thr_list = []
agg_tn = agg_fp = agg_fn = agg_tp = 0
fold_ksecs, fold_tsecs = [], []

for fold_id, (tr_idx, te_idx) in enumerate(cv.split(X, y), 1):
    if fold_id < CKPT.get("fold", 1):
        continue

    X_tr, X_te = X[tr_idx], X[te_idx]
    y_tr, y_te = y[tr_idx], y[te_idx]

    t0 = time.perf_counter()
    # TRAIN: square (upper incl diag)
    K_tr_raw = kernel_train_square(X_tr, fold_id)
    d_tr     = np.diag(K_tr_raw).copy()
    K_tr     = normalize_square_from_diag(K_tr_raw, d_tr)
    K_tr     = center_train_kernel(K_tr)
    # TEST: diag + cross
    d_te     = kernel_test_diag(X_te, fold_id)
    K_te_raw = kernel_test_cross(X_te, X_tr, fold_id)
    K_te     = normalize_cross_from_diag(K_te_raw, d_te, d_tr)
    K_te     = center_test_kernel(K_te, K_tr)
    t1k = time.perf_counter()

    # Save kernels
    pd.DataFrame(K_tr).to_csv(os.path.join(OUTDIR, f"RFE7_K_train_f{fold_id}.csv"), index=False)
    pd.DataFrame(K_te).to_csv(os.path.join(OUTDIR, f"RFE7_K_test_f{fold_id}.csv"), index=False)

    # Pick C
    C_best, _ = pick_best_C(K_tr, y_tr)

    # Train & calibrate threshold
    clf = SVC(kernel="precomputed", C=C_best, class_weight="balanced")
    clf.fit(K_tr, y_tr)
    thr_star = youden_threshold(clf.decision_function(K_tr), y_tr)
    thr_list.append(thr_star)

    # Score
    scores = clf.decision_function(K_te)
    y_pred = (scores >= thr_star).astype(int)

    auc = roc_auc_score(y_te, scores)
    acc = accuracy_score(y_te, y_pred)
    f1  = f1_score(y_te, y_pred)
    tn, fp, fn, tp = confusion_matrix(y_te, y_pred, labels=[0,1]).ravel()
    sens = tp / max(1, tp+fn)
    spec = tn / max(1, tn+fp)
    prec = tp / max(1, tp+fp)
    rec  = sens

    t1 = time.perf_counter()

    fold_aucs.append(auc); fold_accs.append(acc); fold_f1s.append(f1)
    fold_sens.append(sens); fold_spec.append(spec); fold_prec.append(prec); fold_rec.append(rec)
    agg_tn += tn; agg_fp += fp; agg_fn += fn; agg_tp += tp
    fold_ksecs.append(t1k - t0); fold_tsecs.append(t1 - t0)

    # Save fold summary immediately
    fold_summary = {
        "fold": fold_id, "C_best": float(C_best), "thr": float(thr_star),
        "AUC": float(auc), "Acc": float(acc), "F1": float(f1),
        "TP": int(tp), "FP": int(fp), "TN": int(tn), "FN": int(fn),
        "Sens": float(sens), "Spec": float(spec), "Prec": float(prec), "Rec": float(rec),
        "kernel_seconds": float(t1k - t0), "total_seconds": float(t1 - t0)
    }
    with open(os.path.join(OUTDIR, f"fold{fold_id}_summary.json"), "w", encoding="utf-8") as f:
        json.dump(fold_summary, f, indent=2)

    CKPT["fold"] = fold_id + 1
    save_ckpt()

    print(f"  fold {fold_id}: AUC={auc:.4f}  Acc={acc:.4f}  F1={f1:.4f}  "
          f"TP={tp} FP={fp} TN={tn} FN={fn}  "
          f"Sens={sens:.4f} Spec={spec:.4f} Prec={prec:.4f} Rec={rec:.4f}  "
          f"| C*={C_best:g} | thr*={thr_star:.6f} | kernel={fold_ksecs[-1]:.2f}s | total={fold_tsecs[-1]:.2f}s")

# ============================== RUN SUMMARY ================================
mean_auc = float(np.mean(fold_aucs)); std_auc = float(np.std(fold_aucs))
mean_acc = float(np.mean(fold_accs)); std_acc = float(np.std(fold_accs))
mean_f1  = float(np.mean(fold_f1s));  std_f1  = float(np.std(fold_f1s))
mean_sens = float(np.mean(fold_sens)); std_sens = float(np.std(fold_sens))
mean_spec = float(np.mean(fold_spec)); std_spec = float(np.std(fold_spec))
mean_prec = float(np.mean(fold_prec)); std_prec = float(np.std(fold_prec))
mean_rec  = float(np.mean(fold_rec));  std_rec  = float(np.std(fold_rec))
mean_thr  = float(np.mean(thr_list));  std_thr  = float(np.std(thr_list))
mean_k   = float(np.mean(fold_ksecs)); mean_t = float(np.mean(fold_tsecs))

summary = {
    "Mean AUC": mean_auc, "Std AUC": std_auc,
    "Mean Acc": mean_acc, "Std Acc": std_acc,
    "Mean F1":  mean_f1,  "Std F1":  std_f1,
    "Mean Sens": mean_sens, "Std Sens": std_sens,
    "Mean Spec": mean_spec, "Std Spec": std_spec,
    "Mean Prec": mean_prec, "Std Prec": std_prec,
    "Mean Rec":  mean_rec,  "Std Rec":  std_rec,
    "Mean Thr": mean_thr,   "Std Thr":  std_thr,
    "Mean kernel sec/fold": mean_k,
    "Mean total sec/fold":  mean_t,
    "Adaptive": {
        "pairs_total": int(STATS["total_pairs"]),
        "pairs_adapted": int(STATS["adapted_pairs"]),
        "adapted_pct": float(100.0 * STATS["adapted_pairs"] / max(1, STATS["total_pairs"])),
        "max_shots_seen": int(STATS["max_shots_seen"]),
        "base_shots": BASE_SHOTS,
        "band": list(BAND),
        "sigma_target": SIGMA_TARGET,
        "max_shots": MAX_SHOTS
    },
    "C_grid": list(map(float, C_GRID)),
    "layout": list(layout),
    "backend": BACKEND_NAME,
    "dd": USE_DD,
    "m3": (USE_M3 and M3_READY),
    "scheduling": "ASAP",
    "optimization_level": 1,
}
with open(os.path.join(OUTDIR, "run_summary.json"), "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("\n=== Summary: RFE-7 (centered, balanced, trimmed C, calibrated thr, DD, M3-if-available) ===")
print(f"Mean AUC={mean_auc:.4f} (+/- {std_auc:.4f})  "
      f"Mean Acc={mean_acc:.4f} (+/- {std_acc:.4f})  "
      f"Mean F1={mean_f1:.4f} (+/- {std_f1:.4f})")
print(f"Mean Sens={mean_sens:.4f} (+/- {std_sens:.4f})  "
      f"Mean Spec={mean_spec:.4f} (+/- {std_spec:.4f})  "
      f"Mean Prec={mean_prec:.4f} (+/- {std_prec:.4f})  "
      f"Mean Rec={mean_rec:.4f} (+/- {std_rec:.4f})")
print(f"Threshold (Youden) mean={mean_thr:.6f} (+/- {std_thr:.6f})")
print(f"Timing: mean kernel/fold={mean_k:.2f}s  |  mean total/fold={mean_t:.2f}s")

print(f"\n[ADAPTIVE] pairs={STATS['total_pairs']}  adapted={STATS['adapted_pairs']}  "
      f"({100.0*STATS['adapted_pairs']/max(1,STATS['total_pairs']):.1f}%)  "
      f"max_shots_seen={STATS['max_shots_seen']}")

print("\n=== QPU RUN COMPLETE ===")
print(f"Saved outputs in: {os.path.abspath(OUTDIR)}")


In [ ]:
# === QSVM fidelity kernel on REAL QPU (Sampler V2, no Session) =============
# - RFE-7 features, N=96, 3-fold CV
# - SamplerV2 bound directly to your instance backend (no Session)
# - ASAP scheduling + DD (XY4 if Y timed; else CPMG/XX)
# - Adaptive shots (base=256; top up to 1025 inside band)
# - Optional M3 readout mitigation (if mthree installed)
# - Kernel centering + class_weight="balanced" + trimmed C grid
# - Full persistence (per-entry CSV, kernels, summaries) + robust RESUME
#
# Tested for:
#   qiskit ~= 2.2.x
#   qiskit-ibm-runtime ~= 0.42.x  (V2 primitives)
#
# V2 usage patterns:
# - Construct SamplerV2 with a backend and call run([...], shots=int)
# - Access counts via result[i].data.<register_name>.get_counts()
#   (IBM learning path / tutorials show this V2 pattern)

import os, json, time, math, numpy as np, pandas as pd
from collections import Counter, defaultdict
from datetime import datetime

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import RFE
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, confusion_matrix, roc_curve

from qiskit import QuantumCircuit, ClassicalRegister, transpile
from qiskit.circuit.library import z_feature_map, XGate, YGate
from qiskit.transpiler import PassManager, InstructionDurations
from qiskit.transpiler.passes import ASAPScheduleAnalysis, PadDynamicalDecoupling

# IMPORTANT: SamplerV2, pass your backend to the constructor
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler

# Optional M3 mitigation
try:
    import mthree as m3
    _M3_AVAILABLE = True
except Exception:
    _M3_AVAILABLE = False

# =============================== USER CONFIG ================================
BACKEND_NAME    = "ibm_brisbane"
USE_AUTO_LAYOUT = True
INITIAL_LAYOUT  = [3, 4, 5, 6, 7, 8, 16]

N               = 96
N_SPLITS        = 3
RANDOM_SEED     = 42
NQ              = 7

# Shots / adaptive
BASE_SHOTS      = 256
USE_ADAPTIVE    = True
BAND            = (0.40, 0.60)
SIGMA_TARGET    = 0.0125
MAX_SHOTS       = 1025

# Throughput
RUN_BATCH       = 64
KERNEL_CHUNK    = 512

# SVM grid (trimmed)
C_GRID          = np.logspace(-1.5, 2, 8)   # 0.0316 .. 100

# DD / M3
USE_DD          = True
USE_M3          = True

# Resume / Output
RESUME          = True
OUTDIR          = r"D:\PHYSIQUE MATHEMATIQUE\doctora\4 EME ANNE\articles\breast canser\implementations\qsvm_on_wcds\real QPU\SOURUN"
os.makedirs(OUTDIR, exist_ok=True)

ENTRY_LOG_CSV   = os.path.join(OUTDIR, "per_entry_log.csv")
CHECKPOINT_JSON = os.path.join(OUTDIR, "checkpoint.json")
BACKEND_TXT     = os.path.join(OUTDIR, "backend_properties.txt")
REPORTS_DIR     = os.path.join(OUTDIR, "reports")
os.makedirs(REPORTS_DIR, exist_ok=True)

np.random.seed(RANDOM_SEED)

# =============================== UTILITIES =================================
ENTRY_HEADER = [
    "timestamp","fold","block","i","j","p0_raw","p0_mitigated",
    "shots_base","shots_added","shots_total","elapsed_s","batch_id","job_id"
]

def now_str():
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

def atomic_write_json(path, obj):
    tmp = f"{path}.tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)
    os.replace(tmp, path)

def ensure_and_repair_entry_log():
    """
    Ensure per_entry_log.csv exists with the correct header.
    If it exists but has no header, add the header and preserve rows.
    If the columns are mismatched, coerce/reorder to ENTRY_HEADER.
    """
    if not os.path.exists(ENTRY_LOG_CSV):
        # create empty file with header
        pd.DataFrame(columns=ENTRY_HEADER).to_csv(ENTRY_LOG_CSV, index=False)
        return

    # File exists—try reading a small sample to detect header presence
    try:
        df_head = pd.read_csv(ENTRY_LOG_CSV, nrows=1)
        cols = list(df_head.columns)
    except Exception:
        cols = []

    if cols and set(cols) >= {"fold","block","i","j"} and len(cols) == len(ENTRY_HEADER):
        # header looks fine; ensure correct order
        if cols != ENTRY_HEADER:
            try:
                df_all = pd.read_csv(ENTRY_LOG_CSV)
                df_all = df_all.reindex(columns=ENTRY_HEADER)
                df_all.to_csv(ENTRY_LOG_CSV, index=False)
            except Exception:
                pass
        return

    # No header or wrong header: read as data-only and set names
    df_raw = pd.read_csv(ENTRY_LOG_CSV, header=None)
    # If there are extra columns, trim; if fewer, pad with NaNs
    if df_raw.shape[1] > len(ENTRY_HEADER):
        df_raw = df_raw.iloc[:, :len(ENTRY_HEADER)]
    elif df_raw.shape[1] < len(ENTRY_HEADER):
        for _ in range(len(ENTRY_HEADER) - df_raw.shape[1]):
            df_raw[df_raw.shape[1]] = np.nan
    df_raw.columns = ENTRY_HEADER
    df_raw.to_csv(ENTRY_LOG_CSV, index=False)

def append_csv_rows(rows):
    """Append rows to per_entry_log.csv safely (no header rewrite here)."""
    if not rows:
        return
    # Enforce column order & types
    df = pd.DataFrame(rows)
    # add missing cols as NaN; extra cols dropped
    for c in ENTRY_HEADER:
        if c not in df.columns:
            df[c] = np.nan
    df = df[ENTRY_HEADER]
    df.to_csv(ENTRY_LOG_CSV, index=False, mode="a", header=False)

def youden_threshold(scores, y_true):
    fpr, tpr, thr = roc_curve(y_true, scores)
    j = tpr - fpr
    i = int(np.argmax(j))
    return float(thr[i])

# ========================== CONNECT BACKEND / SNAPSHOT =====================
service = QiskitRuntimeService()         # uses your saved IBM Quantum account/instance
backend = service.backend(BACKEND_NAME)
st = backend.status()
print(f"[CONNECTED] backend={backend.name} | operational={st.operational} | pending_jobs={st.pending_jobs} | status='{st.status_msg}'")

# Save properties snapshot
try:
    props = backend.properties()
    with open(BACKEND_TXT, "w", encoding="utf-8") as f:
        f.write(str(props))
except Exception as e_props:
    with open(BACKEND_TXT, "w", encoding="utf-8") as f:
        f.write(f"[no properties available] {e_props}\nTarget summary:\n{getattr(backend, 'target', 'N/A')}")

# ============================== DATA (RFE-7) ===============================
X_df, y_sr = load_breast_cancer(return_X_y=True, as_frame=True)
X_full = X_df.to_numpy()
y_full = (y_sr.values == 1).astype(int)

sss = StratifiedShuffleSplit(n_splits=1, train_size=N, random_state=RANDOM_SEED)
(train_idx, _), = sss.split(X_full, y_full)

scaler_all = StandardScaler().fit(X_full)
X_full_scaled = scaler_all.transform(X_full)

logreg = LogisticRegression(max_iter=500, solver="lbfgs", random_state=RANDOM_SEED)
rfe = RFE(logreg, n_features_to_select=NQ).fit(X_full_scaled, y_full)
rfe_mask = rfe.support_
rfe_names = list(X_df.columns[rfe_mask])
X = X_full_scaled[:, rfe_mask][train_idx]
y = y_full[train_idx]

print(f"[INFO] Using N={N} samples. Class balance: {Counter(int(v) for v in y)}")
print(f"[INFO] RFE-7 features: {rfe_names}")

# ========================== LAYOUT (best 7-qubit chain) ====================
def pick_best_chain(backend, k=7):
    try:
        try:
            cmap = backend.coupling_map
        except Exception:
            cmap = backend.target.build_coupling_map()
        edges = set(tuple(sorted(e)) for e in cmap)
        ro = {}
        e2q = {}
        targ = backend.target
        # readout errors
        for q in range(targ.num_qubits):
            try:
                ro[q] = float(targ.get_instruction("measure").get_properties((q,)).error)
            except Exception:
                ro[q] = 0.02
        # 2q errors (prefer ECR else CX)
        twoq_name = "ecr" if "ecr" in targ.operation_names else ("cx" if "cx" in targ.operation_names else None)
        for (a,b) in edges:
            try:
                e2q[(a,b)] = float(targ.get_instruction(twoq_name).get_properties((a,b)).error) if twoq_name else 0.02
            except Exception:
                e2q[(a,b)] = 0.02
        # brute force DFS paths
        adj = {}
        for a,b in edges:
            adj.setdefault(a, set()).add(b)
            adj.setdefault(b, set()).add(a)
        best = None
        def dfs(path):
            nonlocal best
            if len(path)==k:
                score = sum(ro[q] for q in path) + sum(e2q[tuple(sorted((path[i], path[i+1])))] for i in range(k-1))
                if best is None or score < best[0]:
                    best = (score, path.copy())
                return
            for nb in adj[path[-1]]:
                if nb in path: continue
                path.append(nb); dfs(path); path.pop()
        for s in adj:
            dfs([s])
        return best[1] if best else None
    except Exception:
        return None

if USE_AUTO_LAYOUT:
    best_chain = pick_best_chain(backend, k=NQ)
else:
    best_chain = None

if best_chain is None:
    layout = INITIAL_LAYOUT
    print(f"[LAYOUT] Auto-pick failed → using INITIAL_LAYOUT: {layout}")
else:
    layout = best_chain
    print(f"[LAYOUT] Selected best 7-qubit chain: {layout}")

# ============================ TEMPLATE CIRCUIT =============================
fm_z = z_feature_map(feature_dimension=NQ, reps=1, entanglement="linear", parameter_prefix='z')
fm_x = z_feature_map(feature_dimension=NQ, reps=1, entanglement="linear", parameter_prefix='x')

qc = QuantumCircuit(NQ, name="fid")
qc.compose(fm_z, qubits=range(NQ), inplace=True)
qc.compose(fm_x.inverse(), qubits=range(NQ), inplace=True)
cr = ClassicalRegister(NQ, "m")
qc.add_register(cr)
qc.measure(range(NQ), cr)

t0_build = time.perf_counter()
qc_t = transpile(
    qc,
    backend=backend,                # real backend mapping
    initial_layout=layout,
    optimization_level=1,
)

# DD after ASAP scheduling
supports_y = False
if USE_DD:
    try:
        durations = InstructionDurations.from_backend(backend)
    except Exception:
        durations = None
    if durations is None:
        print("[DD] WARNING: Could not build InstructionDurations; DD disabled.")
    else:
        try:
            _ = durations.get("y", (0,), unit="dt")
            supports_y = True
        except Exception:
            supports_y = False
        dd_seq = [XGate(), YGate(), XGate(), YGate()] if supports_y else [XGate(), XGate()]
        print(f"[DD] Using {'XY4 (X,Y,X,Y)' if supports_y else 'CPMG (X,X)'}.")

        pm = PassManager([
            ASAPScheduleAnalysis(durations=durations),
            PadDynamicalDecoupling(durations=durations, dd_sequence=dd_seq)
        ])
        qc_t = pm.run(qc_t)

t1_build = time.perf_counter()

# Circuit report
def _backend_dt(bk):
    try:
        if getattr(bk, "dt", None) is not None:
            return bk.dt
    except Exception:
        pass
    try:
        return bk.target.dt
    except Exception:
        return None

dt = _backend_dt(backend)
duration_dt = getattr(qc_t, "duration", None)  # deprecated but still present on scheduled circuits
duration_us = (duration_dt * dt * 1e6) if (duration_dt is not None and dt is not None) else None
ops = qc_t.count_ops()
report = {
    "depth": int(qc_t.depth()),
    "ops": {k: int(v) for k, v in ops.items()},
    "duration_dt": int(duration_dt) if duration_dt is not None else None,
    "dt_seconds": float(dt) if dt is not None else None,
    "duration_us": float(duration_us) if duration_us is not None else None,
    "layout": list(layout),
    "dd_sequence": "XY4" if supports_y else "CPMG/XX",
    "scheduling": "ASAP",
    "optimization_level": 1,
}
with open(os.path.join(REPORTS_DIR, "circuit_report.json"), "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)
print(f"[INFO] Prepared circuit (transpile + {'DD' if USE_DD else 'no DD'}). Time: {t1_build - t0_build:.2f}s")

# Parameter order (must match transpiled circuit)
param_order = list(qc_t.parameters)
def extract_params_by_prefix(circ, prefix: str):
    plist = [p for p in circ.parameters if p.name.startswith(f"{prefix}[")]
    plist.sort(key=lambda p: int(p.name[p.name.find('[')+1:-1]))
    return plist
params_x = extract_params_by_prefix(qc_t, "x")
params_z = extract_params_by_prefix(qc_t, "z")

def param_values_for_pair(Arow, Brow):
    mp = {}
    for k in range(NQ):
        mp[params_z[k]] = float(Brow[k])
        mp[params_x[k]] = float(Arow[k])
    return [mp[p] for p in param_order]

# Which register name to read from SamplerV2 results
REG_NAME = qc_t.cregs[0].name if qc_t.cregs else "meas"   # typically "m"; fallback "meas"

# =========================== M3 READOUT MITIGATION =========================
M3_READY = False
MEAS_QUBITS = layout
if USE_M3 and _M3_AVAILABLE:
    try:
        mit = m3.M3Mitigation(backend)
        mit.cals_from_system(MEAS_QUBITS)
        M3_READY = True
        print(f"[M3] Calibrated on qubits: {MEAS_QUBITS}")
    except Exception as e:
        print(f"[M3] Disabled (setup failed): {e}")
        M3_READY = False
else:
    if USE_M3 and not _M3_AVAILABLE:
        print("[M3] Package 'mthree' not installed; mitigation disabled.")

def counts_to_p0(counts_dict):
    shots = int(sum(counts_dict.values()))
    p0 = counts_dict.get("0"*NQ, 0) / max(1, shots)
    return p0, shots

def mitigate_p0(counts_dict):
    """Return (p_raw, p_mitigated, shots). Falls back to raw if M3 not ready."""
    p_raw, s = counts_to_p0(counts_dict)
    if not M3_READY:
        return p_raw, p_raw, s
    try:
        q = mit.apply_correction(counts_dict, MEAS_QUBITS)
        if isinstance(q, dict):
            p0_mit = float(q.get("0"*NQ, p_raw))
        else:
            try:
                p0_mit = float(q[0].get("0"*NQ, p_raw))
            except Exception:
                p0_mit = p_raw
        return p_raw, p0_mit, s
    except Exception as e:
        print(f"[M3] apply_correction failed once: {e} → using raw counts.")
        return p_raw, p_raw, s

# ============================ RESUME / CHECKPOINT ==========================
ensure_and_repair_entry_log()
STATS = {"adapted_pairs": 0, "total_pairs": 0, "max_shots_seen": BASE_SHOTS}

if RESUME and os.path.exists(CHECKPOINT_JSON):
    with open(CHECKPOINT_JSON, "r", encoding="utf-8") as f:
        CKPT = json.load(f)
    print(f"[RESUME] Resuming from {OUTDIR}")
else:
    CKPT = {"fold": 1, "done": []}
    atomic_write_json(CHECKPOINT_JSON, CKPT)

def mark_done(fold, block, i, j=None):
    key = f"fold{fold}-{block}-{i}" if j is None else f"fold{fold}-{block}-{i}-{j}"
    CKPT["done"].append(key)

def is_done(fold, block, i, j=None):
    key = f"fold{fold}-{block}-{i}" if j is None else f"fold{fold}-{block}-{i}-{j}"
    return key in CKPT["done"]

def save_ckpt(fold=None):
    if fold is not None:
        CKPT["fold"] = int(fold)
    atomic_write_json(CHECKPOINT_JSON, CKPT)

def rebuild_done_from_log():
    """On resume, add any missing done-keys from the (repaired) CSV log."""
    try:
        df = pd.read_csv(ENTRY_LOG_CSV)
        for _, r in df.iterrows():
            fold = int(r["fold"]); block = str(r["block"])
            i = int(r["i"]); j_val = r["j"]
            if block == "diag":
                key = f"fold{fold}-{block}-{i}"
            else:
                j = int(j_val)
                key = f"fold{fold}-{block}-{i}-{j}"
            if key not in CKPT["done"]:
                CKPT["done"].append(key)
        save_ckpt()
    except Exception as e:
        print(f"[RESUME] WARNING: could not rebuild done-set from log: {e}")

# Make sure done-set includes everything already in the CSV
rebuild_done_from_log()

# ============================ SAMPLER V2 (NO SESSION) ======================
sampler = Sampler(backend)  # bind to your instance backend

def _extract_counts_from_pub_result(pub_result):
    """V2: result[i].data.<reg>.get_counts() with robust fallback."""
    data = getattr(pub_result, "data", None)
    reg_obj = getattr(data, REG_NAME, None) if data is not None else None
    if reg_obj is None:
        reg_obj = getattr(data, "meas", None)
    return reg_obj.get_counts() if reg_obj is not None else {}

def run_pub_batch(paramvals, shots):
    """
    Run a batch with uniform shots per publication (V2 supports shots per run).
    paramvals: list of parameter lists (aligned to param_order)
    Returns (counts_list, job_id, elapsed)
    """
    pubs = [(qc_t, pv) for pv in paramvals]
    t0 = time.time()
    job = sampler.run(pubs, shots=shots)
    try:
        job_id = job.job_id()
    except Exception:
        job_id = ""
    result = job.result()
    elapsed = time.time() - t0
    counts_list = []
    # result is indexable: result[i].data.<reg>.get_counts()
    for i in range(len(paramvals)):
        counts_list.append(_extract_counts_from_pub_result(result[i]))
    return counts_list, job_id, elapsed

# ============================ KERNEL HELPERS ===============================
def needed_shots_for_sigma(p_hat, sigma):
    p = max(1e-6, min(1-1e-6, float(p_hat)))
    return int(math.ceil(p*(1-p)/(sigma*sigma)))

def process_batch_paramvals(paramvals, indices, fold, block, base_shots, batch_id):
    """
    paramvals: list of parameter lists (aligned to param_order), length=M
    indices  : list of (i,j) pairs matching paramvals
    Logs per-entry rows; returns dict[(i,j)]=(p_mitigated, shots_total)
    """
    # base pass (uniform shots)
    counts_list, job_id, elapsed_base = run_pub_batch(paramvals, base_shots)

    p_mit = np.zeros(len(paramvals), float)
    p_raw = np.zeros(len(paramvals), float)
    shots_tot = np.zeros(len(paramvals), int)
    elapsed_each = np.full(len(paramvals), float(elapsed_base) / max(1, len(paramvals)))

    for k, counts in enumerate(counts_list):
        pr, pm, s = mitigate_p0(counts)
        p_mit[k] = pm; p_raw[k] = pr; shots_tot[k] = s

    # adaptive top-up
    added = np.zeros(len(paramvals), int)
    lo, hi = BAND
    need = []
    if USE_ADAPTIVE:
        for k, pm in enumerate(p_mit):
            if lo <= pm <= hi:
                req = needed_shots_for_sigma(pm, SIGMA_TARGET)
                add = max(0, min(MAX_SHOTS - shots_tot[k], req - shots_tot[k]))
                if add > 0: need.append((k, add))

    if need:
        groups = defaultdict(list)
        for k, add in need:
            groups[add].append(k)
        for add_shots, idxs in groups.items():
            sub_vals = [paramvals[u] for u in idxs]
            sub_counts, job2_id, elapsed_add = run_pub_batch(sub_vals, add_shots)
            # distribute this job's elapsed equally across the sub-batch
            per_entry_add = float(elapsed_add) / max(1, len(idxs))
            for t, c2 in enumerate(sub_counts):
                u = idxs[t]
                pr2, pm2, s2 = mitigate_p0(c2)
                s1 = shots_tot[u]; pm1 = p_mit[u]; pr1 = p_raw[u]
                s_sum = s1 + s2
                pm_comb = (pm1*s1 + pm2*s2) / max(1, s_sum)
                pr_comb = (pr1*s1 + pr2*s2) / max(1, s_sum)
                p_mit[u] = pm_comb; p_raw[u] = pr_comb; shots_tot[u] = s_sum
                added[u] += s2
                elapsed_each[u] += per_entry_add

    # log rows (safe append)
    rows = []
    for k, (i, j) in enumerate(indices):
        rows.append({
            "timestamp": now_str(), "fold": fold, "block": block,
            "i": int(i), "j": int(j),
            "p0_raw": float(p_raw[k]), "p0_mitigated": float(p_mit[k]),
            "shots_base": int(min(shots_tot[k], base_shots)),
            "shots_added": int(added[k]),
            "shots_total": int(shots_tot[k]),
            "elapsed_s": float(elapsed_each[k]),
            "batch_id": int(batch_id),
            "job_id": str(job_id)
        })
    append_csv_rows(rows)

    # stats
    STATS["total_pairs"] += len(paramvals)
    STATS["adapted_pairs"] += int(np.sum(added > 0))
    STATS["max_shots_seen"] = max(STATS["max_shots_seen"], int(shots_tot.max() if len(shots_tot) else BASE_SHOTS))

    out = {(int(indices[k][0]), int(indices[k][1])): (float(p_mit[k]), int(shots_tot[k])) for k in range(len(indices))}
    return out

def kernel_train_square(A, fold):
    n = A.shape[0]
    K = np.zeros((n, n), float)
    pairs, pvals = [], []
    for i in range(n):
        for j in range(i, n):
            if is_done(fold, "train", i, j): continue
            pairs.append((i, j))
            pvals.append(param_values_for_pair(A[i], A[j]))
    batch_id = 0
    for start in range(0, len(pairs), KERNEL_CHUNK):
        sub_pairs = pairs[start:start+KERNEL_CHUNK]
        sub_vals  = pvals[start:start+KERNEL_CHUNK]
        out = {}
        for b in range(0, len(sub_pairs), RUN_BATCH):
            idxs = sub_pairs[b:b+RUN_BATCH]
            vals = sub_vals[b:b+RUN_BATCH]
            batch_id += 1
            res = process_batch_paramvals(vals, idxs, fold, "train", BASE_SHOTS, batch_id)
            out.update(res)
        for (i, j), (p, s) in out.items():
            K[i, j] = p; K[j, i] = p
            mark_done(fold, "train", i, j)
        save_ckpt(fold)
    # include resumed entries
    df = pd.read_csv(ENTRY_LOG_CSV)
    df = df[(df["fold"]==fold) & (df["block"]=="train")]
    for _, r in df.iterrows():
        i, j = int(r["i"]), int(r["j"])
        K[i, j] = float(r["p0_mitigated"]); K[j, i] = K[i, j]
    return K

def kernel_test_diag(A, fold):
    n = A.shape[0]
    d = np.zeros(n, float)
    pairs, pvals = [], []
    for i in range(n):
        if is_done(fold, "diag", i): continue
        pairs.append((i, i))
        pvals.append(param_values_for_pair(A[i], A[i]))
    batch_id = 0
    for start in range(0, len(pairs), KERNEL_CHUNK):
        sub_pairs = pairs[start:start+KERNEL_CHUNK]
        sub_vals  = pvals[start:start+KERNEL_CHUNK]
        out = {}
        for b in range(0, len(sub_pairs), RUN_BATCH):
            idxs = sub_pairs[b:b+RUN_BATCH]
            vals = sub_vals[b:b+RUN_BATCH]
            batch_id += 1
            res = process_batch_paramvals(vals, idxs, fold, "diag", BASE_SHOTS, batch_id)
            out.update(res)
        for (i, _), (p, s) in out.items():
            d[i] = p
            mark_done(fold, "diag", i)
        save_ckpt(fold)
    df = pd.read_csv(ENTRY_LOG_CSV)
    df = df[(df["fold"]==fold) & (df["block"]=="diag")]
    for _, r in df.iterrows():
        i = int(r["i"])
        d[i] = float(r["p0_mitigated"])
    return d

def kernel_test_cross(A, B, fold):
    nA, nB = A.shape[0], B.shape[0]
    K = np.zeros((nA, nB), float)
    pairs, pvals = [], []
    for i in range(nA):
        for j in range(nB):
            if is_done(fold, "test", i, j): continue
            pairs.append((i, j))
            pvals.append(param_values_for_pair(A[i], B[j]))
    batch_id = 0
    for start in range(0, len(pairs), KERNEL_CHUNK):
        sub_pairs = pairs[start:start+KERNEL_CHUNK]
        sub_vals  = pvals[start:start+KERNEL_CHUNK]
        out = {}
        for b in range(0, len(sub_pairs), RUN_BATCH):
            idxs = sub_pairs[b:b+RUN_BATCH]
            vals = sub_vals[b:b+RUN_BATCH]
            batch_id += 1
            res = process_batch_paramvals(vals, idxs, fold, "test", BASE_SHOTS, batch_id)
            out.update(res)
        for (i, j), (p, s) in out.items():
            K[i, j] = p
            mark_done(fold, "test", i, j)
        save_ckpt(fold)
    df = pd.read_csv(ENTRY_LOG_CSV)
    df = df[(df["fold"]==fold) & (df["block"]=="test")]
    for _, r in df.iterrows():
        i, j = int(r["i"]), int(r["j"])
        K[i, j] = float(r["p0_mitigated"])
    return K

def normalize_square_from_diag(K_raw, d):
    d = np.clip(d, 1e-12, None)
    Kn = K_raw / np.sqrt(np.outer(d, d))
    Kn = 0.5 * (Kn + Kn.T)
    np.fill_diagonal(Kn, 1.0)
    return Kn

def normalize_cross_from_diag(K_raw, d_test, d_train):
    d_test  = np.clip(d_test,  1e-12, None)
    d_train = np.clip(d_train, 1e-12, None)
    return K_raw / np.sqrt(np.outer(d_test, d_train))

def center_train_kernel(K):
    r = K.mean(axis=1, keepdims=True); c = K.mean(axis=0, keepdims=True); g = K.mean()
    return K - r - c + g

def center_test_kernel(K_te, K_tr):
    c_tr = K_tr.mean(axis=0, keepdims=True); g_tr = K_tr.mean(); r_te = K_te.mean(axis=1, keepdims=True)
    return K_te - c_tr - r_te + g_tr

def pick_best_C(K_tr, y_tr, Cs=C_GRID, seed=RANDOM_SEED):
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=seed)
    best_auc, best_C = -np.inf, Cs[0]
    n = len(y_tr)
    for C in Cs:
        aucs = []
        for tr2, va2 in skf.split(np.arange(n), y_tr):
            K_tt = K_tr[np.ix_(tr2, tr2)]
            K_tv = K_tr[np.ix_(va2, tr2)]
            clf2 = SVC(kernel="precomputed", C=C, class_weight="balanced")
            clf2.fit(K_tt, y_tr[tr2])
            scores2 = clf2.decision_function(K_tv)
            aucs.append(roc_auc_score(y_tr[va2], scores2))
        m = float(np.mean(aucs))
        if m > best_auc: best_auc, best_C = m, C
    return best_C, best_auc

# =============================== CV LOOP ===================================
print(f"[INFO] Real QPU run | Adaptive={'ON' if USE_ADAPTIVE else 'OFF'} | Base shots={BASE_SHOTS} | RUN_BATCH={RUN_BATCH} | KERNEL_CHUNK={KERNEL_CHUNK}")
print(f"[INFO] C-grid (trimmed): {C_GRID}")

cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_SEED)
fold_aucs, fold_accs, fold_f1s = [], [], []
fold_sens, fold_spec, fold_prec, fold_rec = [], [], [], []
thr_list = []
agg_tn = agg_fp = agg_fn = agg_tp = 0
fold_ksecs, fold_tsecs = [], []

for fold_id, (tr_idx, te_idx) in enumerate(cv.split(X, y), 1):
    if fold_id < CKPT.get("fold", 1):
        continue

    X_tr, X_te = X[tr_idx], X[te_idx]
    y_tr, y_te = y[tr_idx], y[te_idx]

    t0 = time.perf_counter()
    # TRAIN: square (upper incl diag)
    K_tr_raw = kernel_train_square(X_tr, fold_id)
    d_tr     = np.diag(K_tr_raw).copy()
    K_tr     = normalize_square_from_diag(K_tr_raw, d_tr)
    K_tr     = center_train_kernel(K_tr)
    # TEST: diag + cross
    d_te     = kernel_test_diag(X_te, fold_id)
    K_te_raw = kernel_test_cross(X_te, X_tr, fold_id)
    K_te     = normalize_cross_from_diag(K_te_raw, d_te, d_tr)
    K_te     = center_test_kernel(K_te, K_tr)
    t1k = time.perf_counter()

    # Save kernels
    pd.DataFrame(K_tr).to_csv(os.path.join(OUTDIR, f"RFE7_K_train_f{fold_id}.csv"), index=False)
    pd.DataFrame(K_te).to_csv(os.path.join(OUTDIR, f"RFE7_K_test_f{fold_id}.csv"), index=False)

    # Pick C
    C_best, _ = pick_best_C(K_tr, y_tr)

    # Train & calibrate threshold
    clf = SVC(kernel="precomputed", C=C_best, class_weight="balanced")
    clf.fit(K_tr, y_tr)
    thr_star = youden_threshold(clf.decision_function(K_tr), y_tr)
    thr_list.append(thr_star)

    # Score
    scores = clf.decision_function(K_te)
    y_pred = (scores >= thr_star).astype(int)

    auc = roc_auc_score(y_te, scores)
    acc = accuracy_score(y_te, y_pred)
    f1  = f1_score(y_te, y_pred)
    tn, fp, fn, tp = confusion_matrix(y_te, y_pred, labels=[0,1]).ravel()
    sens = tp / max(1, tp+fn)
    spec = tn / max(1, tn+fp)
    prec = tp / max(1, tp+fp)
    rec  = sens

    t1 = time.perf_counter()

    fold_aucs.append(auc); fold_accs.append(acc); fold_f1s.append(f1)
    fold_sens.append(sens); fold_spec.append(spec); fold_prec.append(prec); fold_rec.append(rec)
    agg_tn += tn; agg_fp += fp; agg_fn += fn; agg_tp += tp
    fold_ksecs.append(t1k - t0); fold_tsecs.append(t1 - t0)

    # Save fold summary immediately
    fold_summary = {
        "fold": fold_id, "C_best": float(C_best), "thr": float(thr_star),
        "AUC": float(auc), "Acc": float(acc), "F1": float(f1),
        "TP": int(tp), "FP": int(fp), "TN": int(tn), "FN": int(fn),
        "Sens": float(sens), "Spec": float(spec), "Prec": float(prec), "Rec": float(rec),
        "kernel_seconds": float(t1k - t0), "total_seconds": float(t1 - t0)
    }
    with open(os.path.join(OUTDIR, f"fold{fold_id}_summary.json"), "w", encoding="utf-8") as f:
        json.dump(fold_summary, f, indent=2)

    CKPT["fold"] = fold_id + 1
    save_ckpt()

    print(f"  fold {fold_id}: AUC={auc:.4f}  Acc={acc:.4f}  F1={f1:.4f}  "
          f"TP={tp} FP={fp} TN={tn} FN={fn}  "
          f"Sens={sens:.4f} Spec={spec:.4f} Prec={prec:.4f} Rec={rec:.4f}  "
          f"| C*={C_best:g} | thr*={thr_star:.6f} | kernel={fold_ksecs[-1]:.2f}s | total={fold_tsecs[-1]:.2f}s")

# ============================== RUN SUMMARY ================================
mean_auc = float(np.mean(fold_aucs)); std_auc = float(np.std(fold_aucs))
mean_acc = float(np.mean(fold_accs)); std_acc = float(np.std(fold_accs))
mean_f1  = float(np.mean(fold_f1s));  std_f1  = float(np.std(fold_f1s))
mean_sens = float(np.mean(fold_sens)); std_sens = float(np.std(fold_sens))
mean_spec = float(np.mean(fold_spec)); std_spec = float(np.std(fold_spec))
mean_prec = float(np.mean(fold_prec)); std_prec = float(np.std(fold_prec))
mean_rec  = float(np.mean(fold_rec));  std_rec  = float(np.std(fold_rec))
mean_thr  = float(np.mean(thr_list));  std_thr  = float(np.std(thr_list))
mean_k   = float(np.mean(fold_ksecs)); mean_t = float(np.mean(fold_tsecs))

summary = {
    "Mean AUC": mean_auc, "Std AUC": std_auc,
    "Mean Acc": mean_acc, "Std Acc": std_acc,
    "Mean F1":  mean_f1,  "Std F1":  std_f1,
    "Mean Sens": mean_sens, "Std Sens": std_sens,
    "Mean Spec": mean_spec, "Std Spec": std_spec,
    "Mean Prec": mean_prec, "Std Prec": std_prec,
    "Mean Rec":  mean_rec,  "Std Rec":  std_rec,
    "Mean Thr": mean_thr,   "Std Thr":  std_thr,
    "Mean kernel sec/fold": mean_k,
    "Mean total sec/fold":  mean_t,
    "Adaptive": {
        "pairs_total": int(STATS["total_pairs"]),
        "pairs_adapted": int(STATS["adapted_pairs"]),
        "adapted_pct": float(100.0 * STATS["adapted_pairs"] / max(1, STATS["total_pairs"])),
        "max_shots_seen": int(STATS["max_shots_seen"]),
        "base_shots": BASE_SHOTS,
        "band": list(BAND),
        "sigma_target": SIGMA_TARGET,
        "max_shots": MAX_SHOTS
    },
    "C_grid": list(map(float, C_GRID)),
    "layout": list(layout),
    "backend": BACKEND_NAME,
    "dd": USE_DD,
    "m3": (USE_M3 and M3_READY),
    "scheduling": "ASAP",
    "optimization_level": 1,
}
with open(os.path.join(OUTDIR, "run_summary.json"), "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("\n=== Summary: RFE-7 (centered, balanced, trimmed C, calibrated thr, DD, M3-if-available) ===")
print(f"Mean AUC={mean_auc:.4f} (+/- {std_auc:.4f})  "
      f"Mean Acc={mean_acc:.4f} (+/- {std_acc:.4f})  "
      f"Mean F1={mean_f1:.4f} (+/- {std_f1:.4f})")
print(f"Mean Sens={mean_sens:.4f} (+/- {std_sens:.4f})  "
      f"Mean Spec={mean_spec:.4f} (+/- {std_spec:.4f})  "
      f"Mean Prec={mean_prec:.4f} (+/- {std_prec:.4f})  "
      f"Mean Rec={mean_rec:.4f} (+/- {std_rec:.4f})")
print(f"Threshold (Youden) mean={mean_thr:.6f} (+/- {std_thr:.6f})")
print(f"Timing: mean kernel/fold={mean_k:.2f}s  |  mean total/fold={mean_t:.2f}s")

print(f"\n[ADAPTIVE] pairs={STATS['total_pairs']}  adapted={STATS['adapted_pairs']}  "
      f"({100.0*STATS['adapted_pairs']/max(1,STATS['total_pairs']):.1f}%)  "
      f"max_shots_seen={STATS['max_shots_seen']}")

print("\n=== QPU RUN COMPLETE ===")
print(f"Saved outputs in: {os.path.abspath(OUTDIR)}")


In [ ]:
## OPTIONAL: Save IBM Quantum account (uncomment to use)
import os
from qiskit_ibm_runtime import QiskitRuntimeService

QiskitRuntimeService.save_account(
    channel="ibm_quantum_platform",
    token=os.environ["IBM_QUANTUM_TOKEN"],
    instance=os.environ.get("IBM_QUANTUM_INSTANCE", "open"),
    set_as_default=True,
    overwrite=True,             # replace any existing saved account
)

# Use default everywhere:
svc = QiskitRuntimeService()
print("Default instance:", svc.instances()[:1])
